In [1]:
#!pip install --upgrade grpcio

In [2]:
#!conda install -c conda-forge optuna

se realizó particionamiento por nodos:
sample_train_2000_2020.csv
sample_val_2021_2022.csv
sample_test_2023_2024.csv

y se dividieron por FP, LSP y GSP
Los archivos a utilizar son:

sample_train_2000_2020_FP.csv<br>
sample_val_2021_2022_FP.csv<br>
sample_test_2023_2024_FP.csv<br>
<br>
sample_train_2000_2020_LSP.csv<br>
sample_val_2021_2022_LSP.csv<br><br>
sample_test_2023_2024_LSP.csv

sample_train_2000_2020_GSP.csv<br>
sample_val_2021_2022_GSP.csv<br>
sample_test_2023_2024_GSP.csv<br>

MAS SUS CORRESPONDIENTES COMBINACIONES SON 18 ARCHIVOS EN TOTAL
LOS EXPERIMENTOS SON:

20- FP+LSP+GSP
21 GSP+FP
22 GSP+LSP
23 LSP
24 GSP
25 FP
26 LSP+FP




# programa 20 fp*lsp*gsp
df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6730
1.0    3395
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234

📊 P20-Métricas comparativas de GCN, GAT, GIN y SAGE:
      Accuracy  Precision    Recall  F1 Score   AUC-ROC    AUC-PR       MSE
GCN   0.319171   0.319171  1.000000  0.483896  0.735387  0.663546  0.250056
GAT   0.321040   0.319646  0.998935  0.484317  0.569790  0.363654  0.250114
GIN   0.319171   0.319171  1.000000  0.483896  0.689042  0.503494  0.629592
SAGE  0.412644   0.320763  0.751864  0.449682  0.476522  0.359885  0.250030
Métricas del modelo meta (LogisticRegression):
Accuracy: 0.6990
Precision: 0.5425
Recall: 0.3637
F1 Score: 0.4354
AUC-ROC: 0.6100
Confusion Matrix:
[[3430  576]
 [1195  683]]
Métricas del modelo meta (RandomForest):
Accuracy: 0.9905
Precision: 0.9830
Recall: 0.9872
F1 Score: 0.9851
AUC-ROC: 0.9896
Confusion Matrix:
[[3974   32]
 [  24 1854]]
Métricas del modelo meta (GradientBoosting):
Accuracy: 0.9956
Precision: 0.9889
Recall: 0.9973
F1 Score: 0.9931
AUC-ROC: 0.9960
Confusion Matrix:
[[3985   21]
 [   5 1873]]
Métricas del modelo meta (SVC):
Accuracy: 0.7045
Precision: 0.5437
Recall: 0.4606
F1 Score: 0.4987
AUC-ROC: 0.6397
Confusion Matrix:
[[3280  726]
 [1013  865]]
Métricas del modelo meta (XGBoost):
Accuracy: 0.7736
Precision: 0.6641
Recall: 0.5884
F1 Score: 0.6239
AUC-ROC: 0.7244
Confusion Matrix:
[[3447  559]
 [ 773 1105]]




In [3]:
#PROGRAMA 20 ESTE PROGRAMA TOMA EL ARCHIVO Y PARTICIONA POR NODOS EN TRAIN, VAL Y TEST, TOMA LAS 14 METRICAS (FP+LSP+GSP)
# ESTA VERSION AGREGA NODOS FALTANTES, PERO AFECTA EL RENDIMIENTO DEL ENTRENAMIENTO. EL PROBLEMA ES QUE COMO LOS TRES ARCHIVOS SON 
#DE AÑOS DIFERENTES HAY MUCHOS NODOS QUE NO EXISTEN ENTRE ESTOS ARCHIVO.
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import networkx as nx
import torch
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv
import optuna
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import random

SEED = 41
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)

# =============================
# CARGA CSV Y FILTRADO
# =============================
file_paths = {
    "sample_train_2000_2020": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_train_2000_2020.csv",
    "sample_val_2021_2022": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_val_2021_2022.csv",
    "sample_test_2023_2024": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_test_2023_2024.csv",
}
df_train = pd.read_csv(file_paths["sample_train_2000_2020"])
df_val   = pd.read_csv(file_paths["sample_val_2021_2022"])
df_test  = pd.read_csv(file_paths["sample_test_2023_2024"])
                                              

# Filtrar aristas conectadas

#df_train = df_train[df_train['connected'] == 1].reset_index(drop=True)
#df_val = df_val[df_val['connected'] == 1].reset_index(drop=True)
#df_test = df_test[df_test['connected'] == 1].reset_index(drop=True)

# Reemplazar NaN en 'lenght_short_path' por mediana
df_train['lenght_short_path'] = df_train['lenght_short_path'].fillna(df_train['lenght_short_path'].median())
df_val['lenght_short_path'] = df_val['lenght_short_path'].fillna(df_val['lenght_short_path'].median())
df_test['lenght_short_path'] = df_test['lenght_short_path'].fillna(df_test['lenght_short_path'].median())

# Normalizar columnas numéricas
numeric_columns = [
    'sum_of_papers', 'sum_of_neighbors', 'log_secundary_neighbors', 'sum_of_keywords', 
    'keywords_match', 'sum_of_areas', 'lenght_short_path', 'clustering_index_sum', 
    'jaccard_coefficient', 'resource_allocation', 'adamic_adar_index', 'preferential_attachment', 
    'community_cn', 'community_ra'
]
scaler = MinMaxScaler()
df_train[numeric_columns] = scaler.fit_transform(df_train[numeric_columns])
# Transform val/test con el mismo scaler (no fit)
df_test[numeric_columns] = scaler.transform(df_test[numeric_columns])
df_val[numeric_columns] = scaler.transform(df_val[numeric_columns])

df_train[numeric_columns] = df_train[numeric_columns].fillna(0)
df_val[numeric_columns] = df_val[numeric_columns].fillna(0)
df_test[numeric_columns] = df_test[numeric_columns].fillna(0)


def filter_edges_for_existing_nodes(G_train, df_edges, verbose=True):
    """
    Filtra enlaces que tengan nodos no presentes en el grafo de entrenamiento.
    """
    train_nodes = set(G_train.nodes())
    mask = df_edges['source'].isin(train_nodes) & df_edges['target'].isin(train_nodes)
    df_filtered = df_edges[mask].copy()
    ignored_edges = df_edges[~mask].copy()
    
    if verbose:
        print(f"Total enlaces: {len(df_edges)}")
        print(f"Enlaces válidos: {len(df_filtered)}")
        print(f"Enlaces ignorados (nodos faltantes): {len(ignored_edges)}")
    
    return df_filtered, ignored_edges
    
# =============================
# CONSTRUCCIÓN DEL GRAFO
# =============================
def build_graph(df):
    G = nx.Graph()
    for _, row in df.iterrows():
        G.add_edge(row['source'], row['target'], weight=row['connected'])
    return G

def add_node_features(G, df):
    node_features = {}
    for _, row in df.iterrows():
        for node in [row['source'], row['target']]:
            if node not in node_features:
                node_features[node] = []
            node_features[node] = [
                row['sum_of_papers'], 
                row['sum_of_neighbors'], 
                row['log_secundary_neighbors'],
                row['sum_of_keywords'], 
                row['keywords_match'],
                row['sum_of_areas'], 
                row['lenght_short_path'],
                row['clustering_index_sum'],
                row['jaccard_coefficient'],
                row['resource_allocation'],
                row['adamic_adar_index'],
                row['preferential_attachment'],
                row['community_cn'],
                row['community_ra']
            ]
    for node, features in node_features.items():
        if node in G.nodes:
            G.nodes[node]['features'] = features
    return G


G_train = build_graph(df_train[df_train['connected'] == 1])  # solo positivos
G_train = add_node_features(G_train, df_train)

G_val = build_graph(df_val[df_val['connected'] == 1])
G_val = add_node_features(G_val, df_val)

G_test = build_graph(df_test[df_test['connected'] == 1])
G_test = add_node_features(G_test, df_test)

# =============================
# CONVERTIR A PyG
# =============================
#def graph_to_pyg(G):
#    mapping = {node: i for i, node in enumerate(G.nodes)}
#    edge_index = torch.tensor([[mapping[u], mapping[v]] for u, v in G.edges], dtype=torch.long).t().contiguous()
#    num_nodes = len(G.nodes)
#    node_features = torch.tensor([G.nodes[node]['features'] for node in G.nodes], dtype=torch.float)
#    data = Data(x=node_features, edge_index=edge_index, num_nodes=num_nodes)
#    data.node_mapping = mapping
#    return data

extra_nodos = set(df_train.source) | set(df_train.target) \
              | set(df_val.source) | set(df_val.target) \
              | set(df_test.source) | set(df_test.target)

faltantes_train = [n for n in extra_nodos if n not in G_train.nodes()]
faltantes_val   = [n for n in extra_nodos if n not in G_val.nodes()]
faltantes_test  = [n for n in extra_nodos if n not in G_test.nodes()]

#print("Faltantes en G_train:", faltantes_train)
#print("Faltantes en G_val:", faltantes_val)
#print("Faltantes en G_test:", faltantes_test)


#def graph_to_pyg(G, df_edges):
#    mapping = {node: i for i, node in enumerate(G.nodes)}
#    edge_index = torch.tensor(
#        [[mapping[u], mapping[v]] for u, v in G.edges],
#        dtype=torch.long
#    ).t().contiguous()
#
#    num_nodes = len(G.nodes)
#    node_features = torch.tensor(
#        [G.nodes[node]['features'] for node in G.nodes],
#        dtype=torch.float
#    )
#
#    # Usar connected como edge_label
#    edge_label_index = torch.tensor(
#        [[mapping[u], mapping[v]] for u, v in zip(df_edges.source, df_edges.target)],
#        dtype=torch.long
#    ).t().contiguous()
#
#    edge_label = torch.tensor(df_edges.connected.values, dtype=torch.float)  # 👈 OJO, usar connected
#
#    data = Data(
#        x=node_features,
#        edge_index=edge_index,
#        edge_label_index=edge_label_index,
#        edge_label=edge_label,
#        num_nodes=num_nodes
#    )
#    data.node_mapping = mapping
#    return data



#def get_reverse_mapping(data):
#    """Devuelve un diccionario índice → nodo original a partir de data.node_mapping"""
#    return {idx: node for node, idx in data.node_mapping.items()}

def get_reverse_mapping(data):
    """Devuelve un diccionario índice → nodo original a partir de data.node_mapping_dict"""
    return {idx: node for node, idx in data.node_mapping_dict.items()}


# ===========================================
# Ejecutar para tus tres datasets segun lo que desees que haga que agregue nodos faltantes o no
# ===========================================

import torch
from torch_geometric.data import Data

def graph_to_pyg(G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=False):
    """
    Convierte tres grafos NetworkX y sus DataFrames de aristas a objetos PyG Data,
    revisando los tres conjuntos para agregar nodos faltantes si se desea.

    Parámetros:
        G_train, G_val, G_test : grafos NetworkX para cada split
        df_train, df_val, df_test : DataFrames de aristas ['source','target','connected']
        add_missing_nodes : bool, si True agrega nodos que aparecen en cualquier DF y no están en los grafos

    Retorna:
        train_data, val_data, test_data : objetos torch_geometric.data.Data
    """

    # =============================
    # RECOLECTAR TODOS LOS NODOS NECESARIOS
    # =============================
    if add_missing_nodes:
        extra_nodos = set(df_train.source) | set(df_train.target)
        extra_nodos |= set(df_val.source) | set(df_val.target)
        extra_nodos |= set(df_test.source) | set(df_test.target)
        
        for G in [G_train, G_val, G_test]:
            faltantes = [n for n in extra_nodos if n not in G.nodes()]
            if faltantes:
                feature_dim = len(next(iter(G.nodes(data='features')))[1])
                for n in faltantes:
                    G.add_node(n, features=[0.0]*feature_dim)
            # No imprimimos nodos faltantes para no saturar salida
    if not add_missing_nodes:
        df_train = filter_edges(df_train, G_train)
        df_val   = filter_edges(df_val, G_val)
        df_test  = filter_edges(df_test, G_test)

    # =============================
    # FUNCIÓN INTERNA PARA CONVERTIR CADA SPLIT
    # =============================
    def _convert(G, df):
        if df is None:
            return None
        mapping = {node: i for i, node in enumerate(G.nodes())}
        edge_index = torch.tensor(
            [[mapping[u], mapping[v]] for u, v in zip(df.source, df.target)],
            dtype=torch.long
        ).t().contiguous()
        node_features = torch.tensor(
            [G.nodes[node]['features'] for node in G.nodes()],
            dtype=torch.float
        )
        edge_label = torch.tensor(df.connected.values, dtype=torch.float)
        data = Data(
            x=node_features,
            edge_index=edge_index,
            edge_label=edge_label,
            edge_label_index=edge_index.clone(),
            num_nodes=len(G.nodes())
        )
        # Guardar mapping como atributo dict explícito
        data.node_mapping_dict = mapping
        return data

    train_data = _convert(G_train, df_train)
    val_data   = _convert(G_val, df_val)
    test_data  = _convert(G_test, df_test)

    return train_data, val_data, test_data


train_data, val_data, test_data = graph_to_pyg(
    G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=True
)
    
def sort_edge_index(data):
    sorted_indices = torch.argsort(data.edge_index[1])
    data.edge_index = data.edge_index[:, sorted_indices]
    return data

train_data = sort_edge_index(train_data)
val_data = sort_edge_index(val_data)
test_data = sort_edge_index(test_data)

import numpy as np
print("df_train connected counts:\n", df_train['connected'].value_counts())
print("df_val connected counts:\n", df_val['connected'].value_counts())
print("df_test connected counts:\n", df_test['connected'].value_counts())

true_labels = test_data.edge_label.cpu().numpy()
print("true_labels unique/counts:", np.unique(true_labels, return_counts=True))

# =============================
# CALCULAR DENSIDADES
# =============================


def calc_density(G):
    if G.number_of_nodes() == 0:
        return 0
    return nx.density(G)

def density_for_dataset(data, description):
   # nodes = [reverse_mapping[i] for i in range(data.num_nodes)]
    reverse_mapping = get_reverse_mapping(data) 
    G_tmp = nx.Graph()
    for u, v in data.edge_index.t().numpy():
        G_tmp.add_edge(reverse_mapping[u], reverse_mapping[v])
    density = calc_density(G_tmp)
    print(f"Densidad {description}: {density:.4f}")
    return density

density_train = density_for_dataset(train_data, "Train")
density_val = density_for_dataset(val_data, "Validation")
density_test = density_for_dataset(test_data, "Test")

# =============================
# MODELOS GNN (mantener tu código original)
# =============================
class GCNLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x
    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GATLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels=1, heads=1, dropout=0.0):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, dropout=dropout)
        self.dropout = torch.nn.Dropout(dropout)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x
    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GINLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GINConv(torch.nn.Sequential(torch.nn.Linear(in_channels, hidden_channels), torch.nn.ReLU()))
        self.conv2 = GINConv(torch.nn.Sequential(torch.nn.Linear(hidden_channels, 1)))
        self.dropout = torch.nn.Dropout(dropout)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x
    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class SAGELinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x
    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

# =============================
# FUNCIONES DE EVALUACIÓN Y ENTRENAMIENTO (mantener tu código original)
# =============================

def evaluate_model(model, data):
    model.eval()
    with torch.no_grad():
        preds = model(data.x, data.edge_index)
        edge_emb = preds[data.edge_label_index[0]] * preds[data.edge_label_index[1]]
        y_pred = torch.sigmoid(edge_emb.sum(dim=1)).cpu().numpy()

    y_true = data.edge_label.cpu().numpy().flatten()  # ya alineado
    assert len(y_true) == len(y_pred), f"No coinciden: {len(y_true)} vs {len(y_pred)}"

    return {
        "Accuracy": accuracy_score(y_true, y_pred > 0.5),
        "Precision": precision_score(y_true, y_pred > 0.5, zero_division=1),
        "Recall": recall_score(y_true, y_pred > 0.5),
        "F1 Score": f1_score(y_true, y_pred > 0.5),
        "AUC-ROC": roc_auc_score(y_true, y_pred),
        "AUC-PR": average_precision_score(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred)
    }


# Entrenamiento y evaluación

def train_and_evaluate_model(model, train_data, val_data, epochs, criterion, lr, weight_decay):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()

    # Entrenamiento
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        x = model(train_data.x, train_data.edge_index)  # Pasa las features y las aristas
        out = model.decode(x, train_data.edge_label_index).squeeze()  # Decodifica los enlaces
        loss = criterion(out, train_data.edge_label.float())  # Calcula la pérdida
        loss.backward()
        optimizer.step()

    # Evaluación
    model.eval()
    with torch.no_grad():
        x_val = model(val_data.x, val_data.edge_index)
        val_out = model.decode(x_val, val_data.edge_label_index).squeeze()
        val_loss = criterion(val_out, val_data.edge_label.float()).item()  # Calcula la pérdida en validación

    return val_loss


   
# Objetivo para optimización con Optuna
def objective_gcn(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)  
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GCNLinkPredictor(
        in_channels=14,  # Número de características de entrada
        hidden_channels=hidden_channels,
        dropout=dropout_rate,
    )

    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
   # val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)  
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss

def objective_gat(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    heads = trial.suggest_int("heads", 1, 8)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GATLinkPredictor(
        in_channels=14,  # Número de características de entrada
        hidden_channels=hidden_channels,
        heads=heads,
        dropout=dropout_rate
    )
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
    
def objective_gin(trial):
    # Hiperparámetros a optimizar
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GIN
    model = GINLinkPredictor(
        in_channels=14,  # Número de características de entrada
        hidden_channels=hidden_channels,  # Número de canales ocultos
        dropout=dropout_rate  # Tasa de dropout
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss


def objective_sage(trial):
    # Hiperparámetros para optimización
    hidden_channels = trial.suggest_int("hidden_channels", 16, 64)
    num_layers = trial.suggest_int("num_layers", 2, 4)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    aggr_method = trial.suggest_categorical("aggr_method", ["mean", "max", "lstm"])
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GraphSAGE
    
    model = SAGELinkPredictor(
        in_channels=14,  # Ajusta según tu modelo
        hidden_channels=hidden_channels,
        #num_layers=num_layers,
        #aggr=aggr_method,
        dropout=dropout_rate
    )

    # Optimización
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

      # Llama a la función con todos los argumentos
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
           

# Optuna para GCN
#study_gcn = optuna.create_study(direction="minimize")
study_gcn = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gcn.optimize(objective_gcn, n_trials=50)

print("Best GCN hyperparameters:", study_gcn.best_params)
print("Best GCN validation loss:", study_gcn.best_value)

# Optuna para GAT
#study_gat = optuna.create_study(direction="minimize
study_gat = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gat.optimize(objective_gat, n_trials=50)

print("Best GAT hyperparameters:", study_gat.best_params)
print("Best GAT validation loss:", study_gat.best_value)

# Estudio y optimización para GIN
#study_gin = optuna.create_study(direction="minimize")
study_gin = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gin.optimize(objective_gin, n_trials=10)

# Estudio y optimización para SAGE
#study_sage = optuna.create_study(direction="minimize")
study_sage = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_sage.optimize(objective_sage, n_trials=10)

# Imprimir los mejores parámetros para cada modelo
print("Mejores parámetros para GCN:", study_gcn.best_params)
print("Mejores parámetros para GIN:", study_gin.best_params)
print("Mejores parámetros para SAGE:", study_sage.best_params)
print("Mejores parámetros para GAT:", study_gat.best_params)


# Entrenamiento y guardado de los modelos

# Entrenar y guardar el modelo GCN
best_params = study_gcn.best_params
gcn_model = GCNLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gcn = torch.optim.Adam(gcn_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

criterion = torch.nn.BCEWithLogitsLoss()

gcn_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gcn.zero_grad()
    x = gcn_model(train_data.x, train_data.edge_index)
    out = gcn_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gcn.step()

torch.save(gcn_model.state_dict(), "P20-mejor_modelo_GCN.pth")
print("P20-Modelo GCN guardado.")

# Entrenamiento y guardado del modelo GAT
best_params = study_gat.best_params
if 'heads' in best_params:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=best_params["heads"], dropout=best_params["dropout_rate"])
else:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=1, dropout=best_params["dropout_rate"])

optimizer_gat = torch.optim.Adam(gat_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gat_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gat.zero_grad()
    x = gat_model(train_data.x, train_data.edge_index)
    out = gat_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gat.step()

torch.save(gat_model.state_dict(), "P20-mejor_modelo_GAT.pth")
print("P20-Modelo GAT guardado.")

# Entrenamiento y guardado del modelo GIN
best_params = study_gin.best_params
gin_model = GINLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gin = torch.optim.Adam(gin_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gin_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gin.zero_grad()
    x = gin_model(train_data.x, train_data.edge_index)
    out = gin_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gin.step()

torch.save(gin_model.state_dict(), "P20-mejor_modelo_GIN.pth")
print("P20-Modelo GIN guardado.")

# Entrenamiento y guardado del modelo SAGE
#sage_model = SAGELinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout"])
best_params = study_sage.best_params
sage_model = SAGELinkPredictor(
    in_channels=train_data.x.shape[1],  
    hidden_channels=best_params["hidden_channels"],  
    #num_layers=best_params["num_layers"],  
    #aggr=best_params["aggr_method"],  
    dropout=best_params["dropout_rate"]  
)

optimizer_sage = torch.optim.Adam(sage_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])
criterion = torch.nn.BCEWithLogitsLoss()
sage_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_sage.zero_grad()
    x = sage_model(train_data.x, train_data.edge_index)
    out = sage_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_sage.step()

torch.save(sage_model.state_dict(), "P20-mejor_modelo_SAGE.pth")
print("P20-Modelo SAGE guardado.")


# Cargar todos los modelos para evaluación
gcn_model.load_state_dict(torch.load("P20-mejor_modelo_GCN.pth"))
gcn_model.eval()

gat_model.load_state_dict(torch.load("P20-mejor_modelo_GAT.pth"))
gat_model.eval()

gin_model.load_state_dict(torch.load("P20-mejor_modelo_GIN.pth"))
gin_model.eval()

sage_model.load_state_dict(torch.load("P20-mejor_modelo_SAGE.pth"))
sage_model.eval()

# Evaluar todos los modelos
metrics_gcn = evaluate_model(gcn_model, test_data)
metrics_gat = evaluate_model(gat_model, test_data)
metrics_gin = evaluate_model(gin_model, test_data)
metrics_sage = evaluate_model(sage_model, test_data)

# Crear DataFrame para comparar las métricas
df_metrics = pd.DataFrame([metrics_gcn, metrics_gat, metrics_gin, metrics_sage], index=["GCN", "GAT", "GIN", "SAGE"])

# Mostrar las métricas de los cuatro modelos
print("\n📊 P20-Métricas comparativas de GCN, GAT, GIN y SAGE:")
print(df_metrics)
df_metrics.to_csv("P20-metricas_comparativas.csv", index=True)

#predicciones con test_data
#reverse_mapping = {v: k for k, v in test_data.node_mapping_dict.items()}
reverse_mapping = get_reverse_mapping(test_data)
test_pairs = list(zip(test_data.edge_label_index[0].cpu().numpy(), test_data.edge_label_index[1].cpu().numpy()))

# Mostrar las predicciones de los cuatro modelos
if gcn_model is not None:
    with torch.no_grad():
        x_gcn = gcn_model(test_data.x, test_data.edge_index)
        gcn_predictions = gcn_model.decode(x_gcn, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gcn_predictions = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gcn_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gcn_predictions)]
        gcn_edges = sorted(gcn_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GCN:")
        for author1, author2, score in gcn_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gcn_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P20-predicciones_gcn.csv", index=False)

if gat_model is not None:
    with torch.no_grad():
        x_gat = gat_model(test_data.x, test_data.edge_index)
        gat_predictions = gat_model.decode(x_gat, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gat_predictions = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gat_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gat_predictions)]
        gat_edges = sorted(gat_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GAT:")
        for author1, author2, score in gat_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gat_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P20-predicciones_gat.csv", index=False)


if gin_model is not None:
    with torch.no_grad():
        x_gin = gin_model(test_data.x, test_data.edge_index)
        gin_predictions = gin_model.decode(x_gin, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gin_predictions = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gin_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gin_predictions)]
        gin_edges = sorted(gin_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GIN:")
        for author1, author2, score in gin_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gin_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P20-predicciones_gin.csv", index=False)

#print("Primeras características de nodos:", test_data.x[:5])
#print("Primeras conexiones (edge_index):", test_data.edge_index[:, :5])

if sage_model is not None:
    with torch.no_grad():
        x_sage = sage_model(test_data.x, test_data.edge_index)
        sage_predictions = sage_model.decode(x_sage, test_data.edge_label_index).sigmoid().cpu().numpy()
        #sage_predictions = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        sage_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, sage_predictions)]
        sage_edges = sorted(sage_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones SAGE:")
        for author1, author2, score in sage_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(sage_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P20-predicciones_sage.csv", index=False)

#print("¿GCN y GAT producen las mismas predicciones?", np.allclose(gcn_predictions, gat_predictions))
#print("¿GCN y SAGE producen las mismas predicciones?", np.allclose(gin_predictions, sage_predictions))


print("Configuración GCN:", gcn_model)
print("Configuración GAT:", gat_model)
print("Configuración GIN:", gin_model)
print("Configuración SAGE:", sage_model)


## ensamble de aqui hacia arriba ya no cambiar #
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix

#import xgboost as xgb
# Obtener las predicciones de los modelos

true_labels = test_data.edge_label.numpy()
# Verifica la longitud de las etiquetas y las predicciones
#print("Número de etiquetas", len(true_labels))  # Debe ser el mismo tamaño que las predicciones
# Debe ser el mismo tamaño que las etiquetas

predicciones_gcn = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gcn",len(predicciones_gcn))  

predicciones_gat = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gat",len(predicciones_gat))  

predicciones_gin = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gin",len(predicciones_gin))  

predicciones_sage = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de sage",len(predicciones_sage))  

# Ya entrenado y obtenido las predicciones de los modelos base (GCN, GAT, GIN, SAGE)
X_stack = np.column_stack((predicciones_gcn, predicciones_gat, predicciones_gin, predicciones_sage))

####  optimizar con OPtuna el modelo ensamble meta_model para regresion logistica


# ----- Función genérica de entrenamiento y evaluación -----
def entrenar_y_evaluar(modelo, nombre_modelo):
    modelo.fit(X_stack, true_labels)
    predicciones = modelo.predict(X_stack)
    
    accuracy = accuracy_score(true_labels, predicciones)
    precision = precision_score(true_labels, predicciones)
    recall = recall_score(true_labels, predicciones)
    f1 = f1_score(true_labels, predicciones)
    roc_auc = roc_auc_score(true_labels, predicciones)
    conf_matrix = confusion_matrix(true_labels, predicciones)
    
    print(f"20 Métricas del modelo meta ({nombre_modelo}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Predicciones con nombres
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = predicciones[i]
        predicciones_autores.append((author1_name, author2_name, score))
    
    return modelo, predicciones_autores
    
# ----- Modelos Meta -----
# 1. Logistic Regression optimizado con Optuna
def objective_logistic(trial):
    meta_model = LogisticRegression(
        #C=trial.suggest_loguniform('C', 1e-5, 1e5),
        C = trial.suggest_float('C', 1e-5, 1e5, log=True),
        max_iter=trial.suggest_int('max_iter', 100, 1000),
        random_state=SEED
    )
    import numpy as np
    print("Clases en true_labels:", np.unique(true_labels, return_counts=True))
    meta_model.fit(X_stack, true_labels)
    predicciones = meta_model.predict(X_stack)
    return roc_auc_score(true_labels, predicciones)

def objective_random_forest(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 20)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_gb(trial):
    model = GradientBoostingClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        max_depth=trial.suggest_int('max_depth', 3, 10)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_svc(trial):
    model = SVC(
        C=trial.suggest_loguniform('C', 1e-5, 1e5),
        probability=True
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_xgb(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        use_label_encoder=False,
        eval_metric='logloss'
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

# Diccionario de objetivos
objectives = {
    "LogisticRegression": objective_logistic,
    "RandomForest": objective_random_forest,
    "GradientBoosting": objective_gb,
    "SVC": objective_svc,
    "XGBoost": objective_xgb
}

# Resultados
results = {}

for name, obj in objectives.items():
    print(f"Optimizando {name}...")
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(obj, n_trials=50)
    best_params = study.best_params

    if name == "LogisticRegression":
        model = LogisticRegression(**best_params)
    elif name == "RandomForest":
        model = RandomForestClassifier(**best_params)
    elif name == "GradientBoosting":
        model = GradientBoostingClassifier(**best_params)
    elif name == "SVC":
        model = SVC(**best_params, probability=True)
    elif name == "XGBoost":
        model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss')

    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)

    # Evaluación
    accuracy = accuracy_score(true_labels, preds)
    precision = precision_score(true_labels, preds)
    recall = recall_score(true_labels, preds)
    f1 = f1_score(true_labels, preds)
    roc_auc = roc_auc_score(true_labels, preds)
    conf_matrix = confusion_matrix(true_labels, preds)

    print(f"\n20 Métricas del modelo meta ({name}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Guardar predicciones con nombres de autores en diccionario y en archivo
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = float(preds[i])
        predicciones_autores.append((author1_name, author2_name, score))
    results[name] = predicciones_autores
    df = pd.DataFrame(predicciones_autores, columns=["Autor1", "Autor2", "Score"])
        # Formatear la columna Score a 4 decimales
    df["Score"] = df["Score"].map(lambda x: f"{x:.4f}")
    file_name = f"P20-predicciones_meta-{name.replace(' ', '_').lower()}.csv"
    df.to_csv(file_name, index=False)


###  FIN Optimizar con OPtuna modelo ensamble
              


[I 2025-09-12 10:37:54,257] A new study created in memory with name: no-name-212fb2a6-4d48-4a7e-a351-cec0cf1539ab


df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6730
1.0    3395
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234


[I 2025-09-12 10:37:54,550] Trial 0 finished with value: 0.6931173801422119 and parameters: {'hidden_channels': 44, 'dropout_rate': 0.023047910336305877, 'learning_rate': 0.006800480786994529, 'weight_decay': 0.00044426016159107313, 'epochs': 20}. Best is trial 0 with value: 0.6931173801422119.
[I 2025-09-12 10:37:55,143] Trial 1 finished with value: 0.6931477785110474 and parameters: {'hidden_channels': 84, 'dropout_rate': 0.09546533138986957, 'learning_rate': 0.006718305600877424, 'weight_decay': 0.009175304010220257, 'epochs': 48}. Best is trial 0 with value: 0.6931173801422119.
[I 2025-09-12 10:37:55,722] Trial 2 finished with value: 0.6931666731834412 and parameters: {'hidden_channels': 53, 'dropout_rate': 0.14151681758586332, 'learning_rate': 0.0019441944472424217, 'weight_decay': 0.0031779336001614405, 'epochs': 53}. Best is trial 0 with value: 0.6931173801422119.
[I 2025-09-12 10:37:56,176] Trial 3 finished with value: 0.6931619644165039 and parameters: {'hidden_channels': 23, 

Best GCN hyperparameters: {'hidden_channels': 47, 'dropout_rate': 0.41314073299981435, 'learning_rate': 0.008756280184095138, 'weight_decay': 0.001436262330638875, 'epochs': 35}
Best GCN validation loss: 0.6929129362106323


[I 2025-09-12 10:38:17,972] Trial 0 finished with value: 0.693655252456665 and parameters: {'hidden_channels': 44, 'heads': 1, 'dropout_rate': 0.3384081205552792, 'learning_rate': 0.0005303479078830455, 'weight_decay': 0.0011730727946336476, 'epochs': 64}. Best is trial 0 with value: 0.693655252456665.
[I 2025-09-12 10:38:19,128] Trial 1 finished with value: 0.6931559443473816 and parameters: {'hidden_channels': 37, 'heads': 6, 'dropout_rate': 0.45872392443544835, 'learning_rate': 0.0042459228607980995, 'weight_decay': 0.00332927590197695, 'epochs': 35}. Best is trial 1 with value: 0.6931559443473816.
[I 2025-09-12 10:38:20,003] Trial 2 finished with value: 0.6934630870819092 and parameters: {'hidden_channels': 37, 'heads': 3, 'dropout_rate': 0.24058433440788762, 'learning_rate': 0.0007882526313348943, 'weight_decay': 0.007052775890067773, 'epochs': 38}. Best is trial 1 with value: 0.6931559443473816.
[I 2025-09-12 10:38:22,129] Trial 3 finished with value: 0.693164587020874 and parame

...


20 Métricas del modelo meta (GradientBoosting):
Accuracy: 0.9956
Precision: 0.9889
Recall: 0.9973
F1 Score: 0.9931
AUC-ROC: 0.9960
Confusion Matrix:
[[3985   21]
 [   5 1873]]

Optimizando SVC...


C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\1435600602.py:838: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
[I 2025-09-12 10:46:50,644] Trial 0 finished with value: 0.5 and parameters: {'C': 0.0032302507608634305}. Best is trial 0 with value: 0.5.
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\1435600602.py:838: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
[I 2025-09-12 10:46:59,426] Trial 1 finished with value: 0.5 and parameters: {'C': 2.8904017181974373e-05}. Best is trial 0 with value: 0.5.
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\1435600602.py:


20 Métricas del modelo meta (SVC):
Accuracy: 0.7045
Precision: 0.5437
Recall: 0.4606
F1 Score: 0.4987
AUC-ROC: 0.6397
Confusion Matrix:
[[3280  726]
 [1013  865]]

Optimizando XGBoost...


[I 2025-09-12 11:14:09,486] Trial 1 finished with value: 0.6570942308581856 and parameters: {'n_estimators': 60, 'max_depth': 3, 'learning_rate': 0.18512104979515578}. Best is trial 0 with value: 0.6852318699799077.
[I 2025-09-12 11:14:09,696] Trial 2 finished with value: 0.7218799064449118 and parameters: {'n_estimators': 97, 'max_depth': 8, 'learning_rate': 0.27605987617256006}. Best is trial 2 with value: 0.7218799064449118.
[I 2025-09-12 11:14:09,914] Trial 3 finished with value: 0.6957899147019619 and parameters: {'n_estimators': 155, 'max_depth': 5, 'learning_rate': 0.09207975419980072}. Best is trial 2 with value: 0.7218799064449118.
[I 2025-09-12 11:14:10,049] Trial 4 finished with value: 0.6971624565282003 and parameters: {'n_estimators': 96, 'max_depth': 5, 'learning_rate': 0.14953891395657481}. Best is trial 2 with value: 0.7218799064449118.
[I 2025-09-12 11:14:10,216] Trial 5 finished with value: 0.6999829063646277 and parameters: {'n_estimators': 67, 'max_depth': 8, 'learn


20 Métricas del modelo meta (XGBoost):
Accuracy: 0.7736
Precision: 0.6641
Recall: 0.5884
F1 Score: 0.6239
AUC-ROC: 0.7244
Confusion Matrix:
[[3447  559]
 [ 773 1105]]



Esas cifras corresponden a la densidad de cada subgrafo que generaste (train, validation y test). Te explico con detalle:

1️⃣ Qué es la densidad de un grafo

La densidad mide qué tan conectado está un grafo, es decir, qué fracción de todas las posibles aristas existen realmente.

Para un grafo no dirigido con 
𝑁
N nodos:

Densidad
=
2
⋅
𝐸
𝑁
⋅
(
𝑁
−
1
)
Densidad=
N⋅(N−1)
2⋅E
	​


donde:

𝐸
E = número de aristas presentes

𝑁
N = número de nodos

El factor 2 es porque cada arista conecta dos nodos

La densidad siempre está entre 0 y 1:

0 → ningún nodo conectado

1 → todos los nodos conectados entre sí (grafo completo)

2️⃣ Interpretación de tus resultados

Densidad Train: 0.0151 → solo el 1.51 % de todas las posibles conexiones entre nodos están presentes en tu conjunto de entrenamiento.

Densidad Validation: 0.0151 → igual que el train, tu conjunto de validación tiene aproximadamente la misma proporción de conexiones.

Densidad Test: 0.0149 → muy similar, apenas un poco menos conectado.

Esto es normal en redes de tipo social o académico, donde la mayoría de pares de nodos no están conectados. Tu grafo es muy disperso (sparse).

3️⃣ Por qué es importante

Una densidad baja indica que el grafo es escasamente conectado, lo que puede afectar la predicción de enlaces:

Las heurísticas locales (como vecinos en común) pueden ser menos informativas.

Las GNNs pueden necesitar más capas o características para propagar información efectiva.

Que train, val y test tengan densidades similares es bueno, porque significa que los splits están equilibrados y comparables.

# PROGRAMA 21  ESTADISTICAS DE LA UNAM Y METAMODELOS con GSP+FP POR FECHAS POR NODOS
# GSP+FP POR 2000-2020-2021-2022, 2023-2024

# Programa 21 GSp+FP
df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6730
1.0    3395
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234

21 Métricas comparativas de GCN, GAT, GIN y SAGE:
      Accuracy  Precision    Recall  F1 Score   AUC-ROC    AUC-PR       MSE
GCN   0.321040   0.319153  0.994675  0.483249  0.596721  0.415905  0.250014
GAT   0.319171   0.319171  1.000000  0.483896  0.500964  0.333675  0.250002
GIN   0.532801   0.390660  0.828541  0.530967  0.710270  0.640010  0.248709
SAGE  0.319171   0.319171  1.000000  0.483896  0.392875  0.268668  0.250001

21 Métricas del modelo meta (LogisticRegression):
Accuracy: 0.6852
Precision: 0.5739
Recall: 0.0538
F1 Score: 0.0983
AUC-ROC: 0.5175
Confusion Matrix:
[[3931   75]
 [1777  101]]

Optimizando RandomForest...

21 Métricas del modelo meta (RandomForest):
Accuracy: 0.8960
Precision: 0.8540
Recall: 0.8131
F1 Score: 0.8331
AUC-ROC: 0.8740
Confusion Matrix:
[[3745  261]
 [ 351 1527]]

Optimizando GradientBoosting...

21 Métricas del modelo meta (GradientBoosting):
Accuracy: 0.9018
Precision: 0.8476
Recall: 0.8440
F1 Score: 0.8458
AUC-ROC: 0.8864
Confusion Matrix:
[[3721  285]
 [ 293 1585]]




In [4]:
#PROGRAMA 21  ESTADISTICAS DE LA UNAM Y METAMODELOS con GSP+FP POR FECHAS POR NODOS
# GSP+FP POR 2000-2020-2021-2022, 2023-2024
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import torch
import torch.nn.init as init
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv
import optuna
import logging
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error
from sklearn.metrics import average_precision_score
from tabulate import tabulate
from torch_geometric.nn import global_mean_pool
import torch.nn.functional as F

#source	target	connected	community_cn	community_ra	
# sum_of_areas	sum_of_papers	sum_of_keywords	keywords_match (6)


#REDUCIENDO ALEATORIEDAD

import random
import numpy as np
import torch

SEED = 41
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)



# Cargar datasets

# =============================
# CARGA CSV Y FILTRADO
# =============================
#source	target	connected	community_cn	community_ra	
#sum_of_areas	sum_of_papers	sum_of_keywords	keywords_match

file_paths = {
    "sample_train_2000_2020_GSP_FP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_train_2000_2020_GSP_FP.csv",
    "sample_val_2021_2022_GSP_FP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_val_2021_2022_GSP_FP.csv",
    "sample_test_2023_2024_GSP_FP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_test_2023_2024_GSP_FP.csv",
}
df_train = pd.read_csv(file_paths["sample_train_2000_2020_GSP_FP"])
df_val   = pd.read_csv(file_paths["sample_val_2021_2022_GSP_FP"])
df_test  = pd.read_csv(file_paths["sample_test_2023_2024_GSP_FP"])
                                              

# Filtrar aristas conectadas

#df_train = df_train[df_train['connected'] == 1].reset_index(drop=True)
#df_val = df_val[df_val['connected'] == 1].reset_index(drop=True)
#df_test = df_test[df_test['connected'] == 1].reset_index(drop=True)

# Normalizar columnas numéricas
numeric_columns = [
    'sum_of_areas', 'sum_of_papers', 'sum_of_keywords', 
    'keywords_match',  'community_cn', 'community_ra'
]
scaler = MinMaxScaler()
df_train[numeric_columns] = scaler.fit_transform(df_train[numeric_columns])
df_test[numeric_columns] = scaler.transform(df_test[numeric_columns])
df_val[numeric_columns] = scaler.transform(df_val[numeric_columns])

df_train[numeric_columns] = df_train[numeric_columns].fillna(0)
df_val[numeric_columns] = df_val[numeric_columns].fillna(0)
df_test[numeric_columns] = df_test[numeric_columns].fillna(0)

def filter_edges_for_existing_nodes(G_train, df_edges, verbose=True):
    """
    Filtra enlaces que tengan nodos no presentes en el grafo de entrenamiento.
    """
    train_nodes = set(G_train.nodes())
    mask = df_edges['source'].isin(train_nodes) & df_edges['target'].isin(train_nodes)
    df_filtered = df_edges[mask].copy()
    ignored_edges = df_edges[~mask].copy()
    
    if verbose:
        print(f"Total enlaces: {len(df_edges)}")
        print(f"Enlaces válidos: {len(df_filtered)}")
        print(f"Enlaces ignorados (nodos faltantes): {len(ignored_edges)}")
    
    return df_filtered, ignored_edges

# Reemplazar NaN en 'lenght_short_path' por la mediana
#df_train['camino_más_corto'] = df_train['camino_más_corto'].fillna(df_train['camino_más_corto'].median())


# Construir grafo de entrenamiento
def build_graph(df):
    G = nx.Graph()
    for _, row in df.iterrows():
        G.add_edge(row['source'], row['target'], weight=row['connected'])
    return G


# Asignar características a los nodos
def add_node_features(G, df):
    node_features = {}
    for _, row in df.iterrows():
        for node in [row['source'], row['target']]:
            if node not in node_features:
                node_features[node] = []
            node_features[node] = [
                row['sum_of_areas'], 
                row['sum_of_papers'], 
                row['sum_of_keywords'],
                row['keywords_match'], 
                row['community_cn'],
                row['community_ra']           
            ]
    for node, features in node_features.items():
        if node in G.nodes:
            G.nodes[node]['features'] = features
    return G
G_train = build_graph(df_train[df_train['connected'] == 1])  # solo positivos
G_train = add_node_features(G_train, df_train)

G_val = build_graph(df_val[df_val['connected'] == 1])
G_val = add_node_features(G_val, df_val)

G_test = build_graph(df_test[df_test['connected'] == 1])
G_test = add_node_features(G_test, df_test)


extra_nodos = set(df_train.source) | set(df_train.target) \
              | set(df_val.source) | set(df_val.target) \
              | set(df_test.source) | set(df_test.target)

faltantes_train = [n for n in extra_nodos if n not in G_train.nodes()]
faltantes_val   = [n for n in extra_nodos if n not in G_val.nodes()]
faltantes_test  = [n for n in extra_nodos if n not in G_test.nodes()]


def get_reverse_mapping(data):
    """Devuelve un diccionario índice → nodo original a partir de data.node_mapping_dict"""
    return {idx: node for node, idx in data.node_mapping_dict.items()}

# ===========================================
# Ejecutar para tus tres datasets segun lo que desees que haga que agregue nodos faltantes o no
# ===========================================

import torch
from torch_geometric.data import Data

def graph_to_pyg(G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=False):
    """
    Convierte tres grafos NetworkX y sus DataFrames de aristas a objetos PyG Data,
    revisando los tres conjuntos para agregar nodos faltantes si se desea.

    Parámetros:
        G_train, G_val, G_test : grafos NetworkX para cada split
        df_train, df_val, df_test : DataFrames de aristas ['source','target','connected']
        add_missing_nodes : bool, si True agrega nodos que aparecen en cualquier DF y no están en los grafos

    Retorna:
        train_data, val_data, test_data : objetos torch_geometric.data.Data
    """

   # =============================
    # RECOLECTAR TODOS LOS NODOS NECESARIOS
    # =============================
    if add_missing_nodes:
        extra_nodos = set(df_train.source) | set(df_train.target)
        extra_nodos |= set(df_val.source) | set(df_val.target)
        extra_nodos |= set(df_test.source) | set(df_test.target)
        
        for G in [G_train, G_val, G_test]:
            faltantes = [n for n in extra_nodos if n not in G.nodes()]
            if faltantes:
                feature_dim = len(next(iter(G.nodes(data='features')))[1])
                for n in faltantes:
                    G.add_node(n, features=[0.0]*feature_dim)
            # No imprimimos nodos faltantes para no saturar salida
    if not add_missing_nodes:
        df_train = filter_edges(df_train, G_train)
        df_val   = filter_edges(df_val, G_val)
        df_test  = filter_edges(df_test, G_test)

    # =============================
    # FUNCIÓN INTERNA PARA CONVERTIR CADA SPLIT
    # =============================
    def _convert(G, df):
        if df is None:
            return None
        mapping = {node: i for i, node in enumerate(G.nodes())}
        edge_index = torch.tensor(
            [[mapping[u], mapping[v]] for u, v in zip(df.source, df.target)],
            dtype=torch.long
        ).t().contiguous()
        node_features = torch.tensor(
            [G.nodes[node]['features'] for node in G.nodes()],
            dtype=torch.float
        )
        edge_label = torch.tensor(df.connected.values, dtype=torch.float)
        data = Data(
            x=node_features,
            edge_index=edge_index,
            edge_label=edge_label,
            edge_label_index=edge_index.clone(),
            num_nodes=len(G.nodes())
        )
        # Guardar mapping como atributo dict explícito
        data.node_mapping_dict = mapping
        return data

    train_data = _convert(G_train, df_train)
    val_data   = _convert(G_val, df_val)
    test_data  = _convert(G_test, df_test)

    return train_data, val_data, test_data


train_data, val_data, test_data = graph_to_pyg(
    G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=True
)



def sort_edge_index(data):
    sorted_indices = torch.argsort(data.edge_index[1])
    data.edge_index = data.edge_index[:, sorted_indices]
    return data

train_data = sort_edge_index(train_data)
val_data = sort_edge_index(val_data)
test_data = sort_edge_index(test_data)

import numpy as np
print("df_train connected counts:\n", df_train['connected'].value_counts())
print("df_val connected counts:\n", df_val['connected'].value_counts())
print("df_test connected counts:\n", df_test['connected'].value_counts())

true_labels = test_data.edge_label.cpu().numpy()
print("true_labels unique/counts:", np.unique(true_labels, return_counts=True))

# =============================
# CALCULAR DENSIDADES
# =============================
def calc_density(G):
    if G.number_of_nodes() == 0:
        return 0
    return nx.density(G)

def density_for_dataset(data, description):
   # nodes = [reverse_mapping[i] for i in range(data.num_nodes)]
    reverse_mapping = get_reverse_mapping(data) 
    G_tmp = nx.Graph()
    for u, v in data.edge_index.t().numpy():
        G_tmp.add_edge(reverse_mapping[u], reverse_mapping[v])
    density = calc_density(G_tmp)
    print(f"Densidad {description}: {density:.4f}")
    return density

density_train = density_for_dataset(train_data, "Train")
density_val = density_for_dataset(val_data, "Validation")
density_test = density_for_dataset(test_data, "Test")




# eliminar mensajes de los trials de optuna
optuna.logging.set_verbosity(optuna.logging.ERROR)



# Definir modelos
class GCNLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GATLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels=1, heads=1, dropout=0.0):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, dropout=dropout)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GINLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GINConv(torch.nn.Sequential(torch.nn.Linear(in_channels, hidden_channels), torch.nn.ReLU()))
        self.conv2 = GINConv(torch.nn.Sequential(torch.nn.Linear(hidden_channels, 1)))
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)



class SAGELinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

#evaluacion de modelos

def evaluate_model(model, data):
    model.eval()
    with torch.no_grad():
        preds = model(data.x, data.edge_index)
        edge_emb = preds[data.edge_label_index[0]] * preds[data.edge_label_index[1]]
        y_pred = torch.sigmoid(edge_emb.sum(dim=1)).cpu().numpy()

    y_true = data.edge_label.cpu().numpy().flatten()  # ya alineado
    assert len(y_true) == len(y_pred), f"No coinciden: {len(y_true)} vs {len(y_pred)}"

    return {
        "Accuracy": accuracy_score(y_true, y_pred > 0.5),
        "Precision": precision_score(y_true, y_pred > 0.5, zero_division=1),
        "Recall": recall_score(y_true, y_pred > 0.5),
        "F1 Score": f1_score(y_true, y_pred > 0.5),
        "AUC-ROC": roc_auc_score(y_true, y_pred),
        "AUC-PR": average_precision_score(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred)
    }

# Entrenamiento y evaluación

def train_and_evaluate_model(model, train_data, val_data, epochs, criterion, lr, weight_decay):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()

    # Entrenamiento
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        x = model(train_data.x, train_data.edge_index)  # Pasa las features y las aristas
        out = model.decode(x, train_data.edge_label_index).squeeze()  # Decodifica los enlaces
        loss = criterion(out, train_data.edge_label.float())  # Calcula la pérdida
        loss.backward()
        optimizer.step()

    # Evaluación
    model.eval()
    with torch.no_grad():
        x_val = model(val_data.x, val_data.edge_index)
        val_out = model.decode(x_val, val_data.edge_label_index).squeeze()
        val_loss = criterion(val_out, val_data.edge_label.float()).item()  # Calcula la pérdida en validación

    return val_loss


   
# Objetivo para optimización con Optuna
def objective_gcn(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)  
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GCNLinkPredictor(
        in_channels=6,  # Número de características de entrada
        hidden_channels=hidden_channels,
        dropout=dropout_rate,
    )

    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
   # val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)  
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss

def objective_gat(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    heads = trial.suggest_int("heads", 1, 8)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GATLinkPredictor(
        in_channels=6,  # Número de características de entrada
        hidden_channels=hidden_channels,
        heads=heads,
        dropout=dropout_rate
    )
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
    
def objective_gin(trial):
    # Hiperparámetros a optimizar
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GIN
    model = GINLinkPredictor(
        in_channels=6,  # Número de características de entrada
        hidden_channels=hidden_channels,  # Número de canales ocultos
        dropout=dropout_rate  # Tasa de dropout
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss


def objective_sage(trial):
    # Hiperparámetros para optimización
    hidden_channels = trial.suggest_int("hidden_channels", 16, 64)
    num_layers = trial.suggest_int("num_layers", 2, 4)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    aggr_method = trial.suggest_categorical("aggr_method", ["mean", "max", "lstm"])
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GraphSAGE
    
    model = SAGELinkPredictor(
        in_channels=6,  # Ajusta según tu modelo
        hidden_channels=hidden_channels,
        #num_layers=num_layers,
        #aggr=aggr_method,
        dropout=dropout_rate
    )

    # Optimización
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

      # Llama a la función con todos los argumentos
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
           

# Optuna para GCN
#study_gcn = optuna.create_study(direction="minimize")
study_gcn = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gcn.optimize(objective_gcn, n_trials=50)

print("Best GCN hyperparameters:", study_gcn.best_params)
print("Best GCN validation loss:", study_gcn.best_value)

# Optuna para GAT
#study_gat = optuna.create_study(direction="minimize")
study_gat = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gat.optimize(objective_gat, n_trials=50)

print("Best GAT hyperparameters:", study_gat.best_params)
print("Best GAT validation loss:", study_gat.best_value)

# Estudio y optimización para GIN
#study_gin = optuna.create_study(direction="minimize")
study_gin = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gin.optimize(objective_gin, n_trials=10)

# Estudio y optimización para SAGE
#study_sage = optuna.create_study(direction="minimize")
study_sage = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_sage.optimize(objective_sage, n_trials=10)

# Imprimir los mejores parámetros para cada modelo
print("Mejores parámetros para GCN:", study_gcn.best_params)
print("Mejores parámetros para GIN:", study_gin.best_params)
print("Mejores parámetros para SAGE:", study_sage.best_params)
print("Mejores parámetros para GAT:", study_gat.best_params)


# Entrenamiento y guardado de los modelos

# Entrenar y guardar el modelo GCN
best_params = study_gcn.best_params
gcn_model = GCNLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gcn = torch.optim.Adam(gcn_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

criterion = torch.nn.BCEWithLogitsLoss()

gcn_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gcn.zero_grad()
    x = gcn_model(train_data.x, train_data.edge_index)
    out = gcn_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gcn.step()

torch.save(gcn_model.state_dict(), "P21-mejor_modelo_GCN.pth")
print("Modelo GCN guardado.")

# Entrenamiento y guardado del modelo GAT
best_params = study_gat.best_params
if 'heads' in best_params:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=best_params["heads"], dropout=best_params["dropout_rate"])
else:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=1, dropout=best_params["dropout_rate"])

optimizer_gat = torch.optim.Adam(gat_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gat_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gat.zero_grad()
    x = gat_model(train_data.x, train_data.edge_index)
    out = gat_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gat.step()

torch.save(gat_model.state_dict(), "P21-mejor_modelo_GAT.pth")
print("Modelo GAT guardado.")

# Entrenamiento y guardado del modelo GIN
best_params = study_gin.best_params
gin_model = GINLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gin = torch.optim.Adam(gin_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gin_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gin.zero_grad()
    x = gin_model(train_data.x, train_data.edge_index)
    out = gin_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gin.step()

torch.save(gin_model.state_dict(), "P21-mejor_modelo_GIN.pth")
print("Modelo GIN guardado.")

# Entrenamiento y guardado del modelo SAGE
#sage_model = SAGELinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout"])
best_params = study_sage.best_params
sage_model = SAGELinkPredictor(
    in_channels=train_data.x.shape[1],  
    hidden_channels=best_params["hidden_channels"],  
    #num_layers=best_params["num_layers"],  
    #aggr=best_params["aggr_method"],  
    dropout=best_params["dropout_rate"]  
)

optimizer_sage = torch.optim.Adam(sage_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])
criterion = torch.nn.BCEWithLogitsLoss()
sage_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_sage.zero_grad()
    x = sage_model(train_data.x, train_data.edge_index)
    out = sage_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_sage.step()

torch.save(sage_model.state_dict(), "P21-mejor_modelo_SAGE.pth")
print("Modelo SAGE guardado.")


# Cargar todos los modelos para evaluación
gcn_model.load_state_dict(torch.load("P21-mejor_modelo_GCN.pth"))
gcn_model.eval()

gat_model.load_state_dict(torch.load("P21-mejor_modelo_GAT.pth"))
gat_model.eval()

gin_model.load_state_dict(torch.load("P21-mejor_modelo_GIN.pth"))
gin_model.eval()

sage_model.load_state_dict(torch.load("P21-mejor_modelo_SAGE.pth"))
sage_model.eval()

# Evaluar todos los modelos
metrics_gcn = evaluate_model(gcn_model, test_data)
metrics_gat = evaluate_model(gat_model, test_data)
metrics_gin = evaluate_model(gin_model, test_data)
metrics_sage = evaluate_model(sage_model, test_data)

# Crear DataFrame para comparar las métricas
df_metrics = pd.DataFrame([metrics_gcn, metrics_gat, metrics_gin, metrics_sage], index=["GCN", "GAT", "GIN", "SAGE"])

# Mostrar las métricas de los cuatro modelos
print("\n📊 21 Métricas comparativas de GCN, GAT, GIN y SAGE:")
print(df_metrics)
df_metrics.to_csv("P21-metricas_comparativas.csv", index=True)

#predicciones con test_data

test_pairs = list(zip(test_data.edge_label_index[0].cpu().numpy(), test_data.edge_label_index[1].cpu().numpy()))
reverse_mapping = get_reverse_mapping(test_data)
# Mostrar las predicciones de los cuatro modelos
if gcn_model is not None:
    with torch.no_grad():
        x_gcn = gcn_model(test_data.x, test_data.edge_index)
        gcn_predictions = gcn_model.decode(x_gcn, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gcn_predictions = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gcn_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gcn_predictions)]
        gcn_edges = sorted(gcn_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GCN:")
        for author1, author2, score in gcn_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gcn_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P21-predicciones_gcn.csv", index=False)

if gat_model is not None:
    with torch.no_grad():
        x_gat = gat_model(test_data.x, test_data.edge_index)
        gat_predictions = gat_model.decode(x_gat, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gat_predictions = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gat_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gat_predictions)]
        gat_edges = sorted(gat_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GAT:")
        for author1, author2, score in gat_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gat_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P21-predicciones_gat.csv", index=False)


if gin_model is not None:
    with torch.no_grad():
        x_gin = gin_model(test_data.x, test_data.edge_index)
        gin_predictions = gin_model.decode(x_gin, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gin_predictions = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gin_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gin_predictions)]
        gin_edges = sorted(gin_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GIN:")
        for author1, author2, score in gin_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gin_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P21-predicciones_gin.csv", index=False)

#print("Primeras características de nodos:", test_data.x[:5])
#print("Primeras conexiones (edge_index):", test_data.edge_index[:, :5])

if sage_model is not None:
    with torch.no_grad():
        x_sage = sage_model(test_data.x, test_data.edge_index)
        sage_predictions = sage_model.decode(x_sage, test_data.edge_label_index).sigmoid().cpu().numpy()
        #sage_predictions = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        sage_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, sage_predictions)]
        sage_edges = sorted(sage_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones SAGE:")
        for author1, author2, score in sage_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(sage_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P21-predicciones_sage.csv", index=False)

#print("¿GCN y GAT producen las mismas predicciones?", np.allclose(gcn_predictions, gat_predictions))
#print("¿GCN y SAGE producen las mismas predicciones?", np.allclose(gin_predictions, sage_predictions))


print("Configuración GCN:", gcn_model)
print("Configuración GAT:", gat_model)
print("Configuración GIN:", gin_model)
print("Configuración SAGE:", sage_model)


## ensamble de aqui hacia arriba ya no cambiar #
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix

#import xgboost as xgb
# Obtener las predicciones de los modelos

true_labels = test_data.edge_label.numpy()
# Verifica la longitud de las etiquetas y las predicciones
#print("Número de etiquetas", len(true_labels))  # Debe ser el mismo tamaño que las predicciones
# Debe ser el mismo tamaño que las etiquetas

predicciones_gcn = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gcn",len(predicciones_gcn))  

predicciones_gat = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gat",len(predicciones_gat))  

predicciones_gin = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gin",len(predicciones_gin))  

predicciones_sage = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de sage",len(predicciones_sage))  

# Ya entrenado y obtenido las predicciones de los modelos base (GCN, GAT, GIN, SAGE)
X_stack = np.column_stack((predicciones_gcn, predicciones_gat, predicciones_gin, predicciones_sage))

####  optimizar con OPtuna el modelo ensamble meta_model para regresion logistica


# ----- Función genérica de entrenamiento y evaluación -----
def entrenar_y_evaluar(modelo, nombre_modelo):
    modelo.fit(X_stack, true_labels)
    predicciones = modelo.predict(X_stack)
    
    accuracy = accuracy_score(true_labels, predicciones)
    precision = precision_score(true_labels, predicciones)
    recall = recall_score(true_labels, predicciones)
    f1 = f1_score(true_labels, predicciones)
    roc_auc = roc_auc_score(true_labels, predicciones)
    conf_matrix = confusion_matrix(true_labels, predicciones)
    
    print(f"21 Métricas del modelo meta ({nombre_modelo}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Predicciones con nombres
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = predicciones[i]
        predicciones_autores.append((author1_name, author2_name, score))
    
    return modelo, predicciones_autores
    
# ----- Modelos Meta -----
# 1. Logistic Regression optimizado con Optuna
def objective_logistic(trial):
    meta_model = LogisticRegression(
        #C=trial.suggest_loguniform('C', 1e-5, 1e5),
        C = trial.suggest_float('C', 1e-5, 1e5, log=True),
        max_iter=trial.suggest_int('max_iter', 100, 1000),
        random_state=SEED
    )
    meta_model.fit(X_stack, true_labels)
    predicciones = meta_model.predict(X_stack)
    return roc_auc_score(true_labels, predicciones)

def objective_random_forest(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 20)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_gb(trial):
    model = GradientBoostingClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        max_depth=trial.suggest_int('max_depth', 3, 10)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_svc(trial):
    model = SVC(
        C=trial.suggest_loguniform('C', 1e-5, 1e5),
        probability=True
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_xgb(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        use_label_encoder=False,
        eval_metric='logloss'
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

# Diccionario de objetivos
objectives = {
    "LogisticRegression": objective_logistic,
    "RandomForest": objective_random_forest,
    "GradientBoosting": objective_gb,
    "SVC": objective_svc,
    "XGBoost": objective_xgb
}

# Resultados
results = {}

for name, obj in objectives.items():
    print(f"Optimizando {name}...")
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(obj, n_trials=50)
    best_params = study.best_params

    if name == "LogisticRegression":
        model = LogisticRegression(**best_params)
    elif name == "RandomForest":
        model = RandomForestClassifier(**best_params)
    elif name == "GradientBoosting":
        model = GradientBoostingClassifier(**best_params)
    elif name == "SVC":
        model = SVC(**best_params, probability=True)
    elif name == "XGBoost":
        model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss')

    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)

    # Evaluación
    accuracy = accuracy_score(true_labels, preds)
    precision = precision_score(true_labels, preds)
    recall = recall_score(true_labels, preds)
    f1 = f1_score(true_labels, preds)
    roc_auc = roc_auc_score(true_labels, preds)
    conf_matrix = confusion_matrix(true_labels, preds)

    print(f"\n21 Métricas del modelo meta ({name}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Guardar predicciones con nombres de autores en diccionario y en archivo
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = float(preds[i])
        predicciones_autores.append((author1_name, author2_name, score))
    results[name] = predicciones_autores
    df = pd.DataFrame(predicciones_autores, columns=["Autor1", "Autor2", "Score"])
        # Formatear la columna Score a 4 decimales
    df["Score"] = df["Score"].map(lambda x: f"{x:.4f}")
    file_name = f"P21-predicciones_meta-{name.replace(' ', '_').lower()}.csv"
    df.to_csv(file_name, index=False)


###  FIN Optimizar con OPtuna modelo ensamble




df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6730
1.0    3395
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234
Best GCN hyperparameters: {'hidden_channels': 99, 'dropout_rate': 0.17161203322770524, 'learning_rate': 0.005438127114424023, 'weight_decay': 0.005441853438836154, 'epochs': 11}
Best GCN validation loss: 0.6931418776512146
Best GAT hyperparameters: {'hidden_channels': 37, 'heads': 6, 'dropout_rate': 0.45872392443544835, 'learning_rate': 0.0042459228607980995, 'weight_decay': 0.00332927590197695, 'epochs': 35}
Best GAT validation loss: 0.6931471824645996
Mejores parámetros para GCN: {'hidden_channels': 99, 'dropout_rate': 0.17161203322770524, 'learning_rate': 0.0054381271144240

C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\722109058.py:798: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\722109058.py:798: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\722109058.py:798: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppData\


21 Métricas del modelo meta (SVC):
Accuracy: 0.7046
Precision: 0.6101
Recall: 0.2066
F1 Score: 0.3087
AUC-ROC: 0.5723
Confusion Matrix:
[[3758  248]
 [1490  388]]

Optimizando XGBoost...

21 Métricas del modelo meta (XGBoost):
Accuracy: 0.7505
Precision: 0.6306
Recall: 0.5272
F1 Score: 0.5742
AUC-ROC: 0.6912
Confusion Matrix:
[[3426  580]
 [ 888  990]]



# PROGRAMA 22 SE CORRE CON DATOS UNAM  GSP + LSP UNAM NODOS
df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6730
1.0    3395
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234

Métricas comparativas de GCN, GAT, GIN y SAGE:
      Accuracy  Precision    Recall  F1 Score   AUC-ROC    AUC-PR       MSE
GCN   0.319171   0.319171  1.000000  0.483896  0.512595  0.336800  0.250233
GAT   0.319171   0.319171  1.000000  0.483896  0.330460  0.247457  0.250108
GIN   0.320700   0.319660  1.000000  0.484458  0.726417  0.650207  0.489161
SAGE  0.490823   0.335685  0.608094  0.432576  0.529126  0.332617  0.250008


Métricas del modelo meta (LogisticRegression):
Accuracy: 0.7014
Precision: 0.5475
Recall: 0.3711
F1 Score: 0.4424
AUC-ROC: 0.6137
Confusion Matrix:
[[3430  576]
 [1181  697]]

Optimizando RandomForest...

Métricas del modelo meta (RandomForest):
Accuracy: 0.9663
Precision: 0.9531
Recall: 0.9409
F1 Score: 0.9469
AUC-ROC: 0.9596
Confusion Matrix:
[[3919   87]
 [ 111 1767]]

Métricas del modelo meta (GradientBoosting):
Accuracy: 0.9691
Precision: 0.9506
Recall: 0.9526
F1 Score: 0.9516
AUC-ROC: 0.9647
Confusion Matrix:
[[3913   93]
 [  89 1789]]
 Métricas del modelo meta (SVC):
Accuracy: 0.7056
Precision: 0.5493
Recall: 0.4334
F1 Score: 0.4845
AUC-ROC: 0.6333
Confusion Matrix:
[[3338  668]
 [1064  814]]

Optimizando XGBoost...

Métricas del modelo meta (XGBoost):
Accuracy: 0.7820
Precision: 0.6806
Recall: 0.5969
F1 Score: 0.6360
AUC-ROC: 0.7328
Confusion Matrix:
[[3480  526]
 [ 757 1121]]

In [5]:
#PROGRAMA 22  ESTADISTICAS CON UNAM Y METAMODELOS LOS ARCHIVOS DE ENTRENAMIENTO ESTAN ORDENADOS POR NODOS POR FECHAS 
# GSP + LSP UNAM NODOS
#source	target	connected	lenght_short_path	jaccard_coefficient	adamic_adar_index	
#resource_allocation	preferential_attachment	sum_of_neighbors	log_secundary_neighbors	clustering_index_sum	community_cn community_ra (10)

from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import torch
import torch.nn.init as init
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv
import optuna
import logging
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error
from sklearn.metrics import average_precision_score
from tabulate import tabulate
from torch_geometric.nn import global_mean_pool
import torch.nn.functional as F

#REDUCIENDO ALEATORIEDAD
import random
import numpy as np
import torch

SEED = 41
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)

#source	target	connected	camino_mÃ¡s_corto	centralidad_grado_source	centralidad_grado_target, coeficiente_agrupamiento_source	coeficiente_agrupamiento_target	centralidad_eigenvector_source	centralidad_eigenvector_target	pagerank_source	pagerank_target	k_core_number_source	k_core_number_target	

# Cargar datasets
#file_paths = {
#    "train_features": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/9_estadisticas_autores_2018-2024.csv",
#    #"train_features": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/9_estadisticas_autores_1993-2003-final.csv",
#}


file_paths = {
    "sample_train_2000_2020_LSP_GSP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_train_2000_2020_LSP_GSP.csv",
    "sample_val_2021_2022_LSP_GSP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_val_2021_2022_LSP_GSP.csv",
    "sample_test_2023_2024_LSP_GSP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_test_2023_2024_LSP_GSP.csv",
}
df_train = pd.read_csv(file_paths["sample_train_2000_2020_LSP_GSP"])
df_val   = pd.read_csv(file_paths["sample_val_2021_2022_LSP_GSP"])
df_test  = pd.read_csv(file_paths["sample_test_2023_2024_LSP_GSP"])
                                              

# Filtrar aristas conectadas

#df_train = df_train[df_train['connected'] == 1].reset_index(drop=True)
#df_val = df_val[df_val['connected'] == 1].reset_index(drop=True)
#df_test = df_test[df_test['connected'] == 1].reset_index(drop=True)

# Reemplazar NaN en 'lenght_short_path' por mediana
df_train['lenght_short_path'] = df_train['lenght_short_path'].fillna(df_train['lenght_short_path'].median())
df_val['lenght_short_path'] = df_val['lenght_short_path'].fillna(df_val['lenght_short_path'].median())
df_test['lenght_short_path'] = df_test['lenght_short_path'].fillna(df_test['lenght_short_path'].median())

# Normalizar columnas numéricas

#lenght_short_path	jaccard_coefficient	adamic_adar_index	
#resource_allocation	preferential_attachment	sum_of_neighbors	log_secundary_neighbors	clustering_index_sum	community_cn community_ra (10)


numeric_columns = [
    'lenght_short_path', 'jaccard_coefficient', 'adamic_adar_index', 'resource_allocation', 
    'preferential_attachment', 'sum_of_neighbors', 'log_secundary_neighbors','clustering_index_sum',
    'community_cn', 'community_ra'
]
scaler = MinMaxScaler()
df_train[numeric_columns] = scaler.fit_transform(df_train[numeric_columns])
# Transform val/test con el mismo scaler (no fit)
df_test[numeric_columns] = scaler.transform(df_test[numeric_columns])
df_val[numeric_columns] = scaler.transform(df_val[numeric_columns])

df_train[numeric_columns] = df_train[numeric_columns].fillna(0)
df_val[numeric_columns] = df_val[numeric_columns].fillna(0)
df_test[numeric_columns] = df_test[numeric_columns].fillna(0)

def filter_edges_for_existing_nodes(G_train, df_edges, verbose=True):
    """
    Filtra enlaces que tengan nodos no presentes en el grafo de entrenamiento.
    """
    train_nodes = set(G_train.nodes())
    mask = df_edges['source'].isin(train_nodes) & df_edges['target'].isin(train_nodes)
    df_filtered = df_edges[mask].copy()
    ignored_edges = df_edges[~mask].copy()
    
    if verbose:
        print(f"Total enlaces: {len(df_edges)}")
        print(f"Enlaces válidos: {len(df_filtered)}")
        print(f"Enlaces ignorados (nodos faltantes): {len(ignored_edges)}")
    
    return df_filtered, ignored_edges

# Construir grafo de entrenamiento
def build_graph(df):
    G = nx.Graph()
    for _, row in df.iterrows():
        G.add_edge(row['source'], row['target'], weight=row['connected'])
    return G



# Asignar características a los nodos
def add_node_features(G, df):
    node_features = {}
    for _, row in df.iterrows():
        for node in [row['source'], row['target']]:
            if node not in node_features:
                node_features[node] = []
            node_features[node] = [
                row['lenght_short_path'], 
                row['jaccard_coefficient'],
                row['adamic_adar_index'],
                row['resource_allocation'],
                row['preferential_attachment'],
                row['sum_of_neighbors'],
                row['log_secundary_neighbors'],
                row['clustering_index_sum'],
                row['community_cn'],
                row['community_ra'],
            ]
    for node, features in node_features.items():
        if node in G.nodes:
            G.nodes[node]['features'] = features
    return G
    
G_train = build_graph(df_train[df_train['connected'] == 1])  # solo positivos
G_train = add_node_features(G_train, df_train)

G_val = build_graph(df_val[df_val['connected'] == 1])
G_val = add_node_features(G_val, df_val)

G_test = build_graph(df_test[df_test['connected'] == 1])
G_test = add_node_features(G_test, df_test)

extra_nodos = set(df_train.source) | set(df_train.target) \
              | set(df_val.source) | set(df_val.target) \
              | set(df_test.source) | set(df_test.target)

faltantes_train = [n for n in extra_nodos if n not in G_train.nodes()]
faltantes_val   = [n for n in extra_nodos if n not in G_val.nodes()]
faltantes_test  = [n for n in extra_nodos if n not in G_test.nodes()]


def get_reverse_mapping(data):
    """Devuelve un diccionario índice → nodo original a partir de data.node_mapping_dict"""
    return {idx: node for node, idx in data.node_mapping_dict.items()}



# eliminar mensajes de los trials de optuna
optuna.logging.set_verbosity(optuna.logging.ERROR)


# ===========================================
# Ejecutar para tus tres datasets segun lo que desees que haga que agregue nodos faltantes o no
# ===========================================

import torch
from torch_geometric.data import Data
def graph_to_pyg(G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=False):
    """
    Convierte tres grafos NetworkX y sus DataFrames de aristas a objetos PyG Data,
    revisando los tres conjuntos para agregar nodos faltantes si se desea.

    Parámetros:
        G_train, G_val, G_test : grafos NetworkX para cada split
        df_train, df_val, df_test : DataFrames de aristas ['source','target','connected']
        add_missing_nodes : bool, si True agrega nodos que aparecen en cualquier DF y no están en los grafos

    Retorna:
        train_data, val_data, test_data : objetos torch_geometric.data.Data
    """

     # =============================
    # RECOLECTAR TODOS LOS NODOS NECESARIOS
    # =============================
    if add_missing_nodes:
        extra_nodos = set(df_train.source) | set(df_train.target)
        extra_nodos |= set(df_val.source) | set(df_val.target)
        extra_nodos |= set(df_test.source) | set(df_test.target)
        
        for G in [G_train, G_val, G_test]:
            faltantes = [n for n in extra_nodos if n not in G.nodes()]
            if faltantes:
                feature_dim = len(next(iter(G.nodes(data='features')))[1])
                for n in faltantes:
                    G.add_node(n, features=[0.0]*feature_dim)
            # No imprimimos nodos faltantes para no saturar salida
    if not add_missing_nodes:
        df_train = filter_edges(df_train, G_train)
        df_val   = filter_edges(df_val, G_val)
        df_test  = filter_edges(df_test, G_test)

    # =============================
    # FUNCIÓN INTERNA PARA CONVERTIR CADA SPLIT
    # =============================
    def _convert(G, df):
        if df is None:
            return None
        mapping = {node: i for i, node in enumerate(G.nodes())}
        edge_index = torch.tensor(
            [[mapping[u], mapping[v]] for u, v in zip(df.source, df.target)],
            dtype=torch.long
        ).t().contiguous()
        node_features = torch.tensor(
            [G.nodes[node]['features'] for node in G.nodes()],
            dtype=torch.float
        )
        edge_label = torch.tensor(df.connected.values, dtype=torch.float)
        data = Data(
            x=node_features,
            edge_index=edge_index,
            edge_label=edge_label,
            edge_label_index=edge_index.clone(),
            num_nodes=len(G.nodes())
        )
        # Guardar mapping como atributo dict explícito
        data.node_mapping_dict = mapping
        return data

    train_data = _convert(G_train, df_train)
    val_data   = _convert(G_val, df_val)
    test_data  = _convert(G_test, df_test)

    return train_data, val_data, test_data


train_data, val_data, test_data = graph_to_pyg(
    G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=True
)
    
def sort_edge_index(data):
    sorted_indices = torch.argsort(data.edge_index[1])
    data.edge_index = data.edge_index[:, sorted_indices]
    return data

train_data = sort_edge_index(train_data)
val_data = sort_edge_index(val_data)
test_data = sort_edge_index(test_data)

import numpy as np
print("df_train connected counts:\n", df_train['connected'].value_counts())
print("df_val connected counts:\n", df_val['connected'].value_counts())
print("df_test connected counts:\n", df_test['connected'].value_counts())

true_labels = test_data.edge_label.cpu().numpy()
print("true_labels unique/counts:", np.unique(true_labels, return_counts=True))


# =============================
# CALCULAR DENSIDADES
# =============================


def calc_density(G):
    if G.number_of_nodes() == 0:
        return 0
    return nx.density(G)

def density_for_dataset(data, description):
   # nodes = [reverse_mapping[i] for i in range(data.num_nodes)]
    reverse_mapping = get_reverse_mapping(data) 
    G_tmp = nx.Graph()
    for u, v in data.edge_index.t().numpy():
        G_tmp.add_edge(reverse_mapping[u], reverse_mapping[v])
    density = calc_density(G_tmp)
    print(f"Densidad {description}: {density:.4f}")
    return density

density_train = density_for_dataset(train_data, "Train")
density_val = density_for_dataset(val_data, "Validation")
density_test = density_for_dataset(test_data, "Test")


# Definir modelos
class GCNLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GATLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels=1, heads=1, dropout=0.0):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, dropout=dropout)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GINLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GINConv(torch.nn.Sequential(torch.nn.Linear(in_channels, hidden_channels), torch.nn.ReLU()))
        self.conv2 = GINConv(torch.nn.Sequential(torch.nn.Linear(hidden_channels, 1)))
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)



class SAGELinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

#evaluacion de modelos
def evaluate_model(model, data):
    model.eval()
    with torch.no_grad():
        preds = model(data.x, data.edge_index)
        edge_emb = preds[data.edge_label_index[0]] * preds[data.edge_label_index[1]]
        y_pred = torch.sigmoid(edge_emb.sum(dim=1)).cpu().numpy()

    y_true = data.edge_label.cpu().numpy().flatten()  # ya alineado
    assert len(y_true) == len(y_pred), f"No coinciden: {len(y_true)} vs {len(y_pred)}"

    return {
        "Accuracy": accuracy_score(y_true, y_pred > 0.5),
        "Precision": precision_score(y_true, y_pred > 0.5, zero_division=1),
        "Recall": recall_score(y_true, y_pred > 0.5),
        "F1 Score": f1_score(y_true, y_pred > 0.5),
        "AUC-ROC": roc_auc_score(y_true, y_pred),
        "AUC-PR": average_precision_score(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred)
    }


# Entrenamiento y evaluación

def train_and_evaluate_model(model, train_data, val_data, epochs, criterion, lr, weight_decay):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()

    # Entrenamiento
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        x = model(train_data.x, train_data.edge_index)  # Pasa las features y las aristas
        out = model.decode(x, train_data.edge_label_index).squeeze()  # Decodifica los enlaces
        loss = criterion(out, train_data.edge_label.float())  # Calcula la pérdida
        loss.backward()
        optimizer.step()

    # Evaluación
    model.eval()
    with torch.no_grad():
        x_val = model(val_data.x, val_data.edge_index)
        val_out = model.decode(x_val, val_data.edge_label_index).squeeze()
        val_loss = criterion(val_out, val_data.edge_label.float()).item()  # Calcula la pérdida en validación

    return val_loss


   
# Objetivo para optimización con Optuna
def objective_gcn(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)  
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GCNLinkPredictor(
        in_channels=10,  # Número de características de entrada
        hidden_channels=hidden_channels,
        dropout=dropout_rate,
    )

    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
   # val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)  
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss

def objective_gat(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    heads = trial.suggest_int("heads", 1, 8)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GATLinkPredictor(
        in_channels=10,  # Número de características de entrada
        hidden_channels=hidden_channels,
        heads=heads,
        dropout=dropout_rate
    )
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
    
def objective_gin(trial):
    # Hiperparámetros a optimizar
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GIN
    model = GINLinkPredictor(
        in_channels=10,  # Número de características de entrada, antes 15 estadisticas, ahora 9
        hidden_channels=hidden_channels,  # Número de canales ocultos
        dropout=dropout_rate  # Tasa de dropout
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss


def objective_sage(trial):
    # Hiperparámetros para optimización
    hidden_channels = trial.suggest_int("hidden_channels", 16, 64)
    num_layers = trial.suggest_int("num_layers", 2, 4)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    aggr_method = trial.suggest_categorical("aggr_method", ["mean", "max", "lstm"])
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GraphSAGE
    
    model = SAGELinkPredictor(
        in_channels=10,  # Ajusta según tu modelo al numero de caracteristicas de entrada (estadisticas,no los campos)
        hidden_channels=hidden_channels,
        #num_layers=num_layers,
        #aggr=aggr_method,
        dropout=dropout_rate
    )

    # Optimización
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

      # Llama a la función con todos los argumentos
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
           

# Optuna para GCN
#study_gcn = optuna.create_study(direction="minimize")
study_gcn = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gcn.optimize(objective_gcn, n_trials=50)

print("Best GCN hyperparameters:", study_gcn.best_params)
print("Best GCN validation loss:", study_gcn.best_value)

# Optuna para GAT
#study_gat = optuna.create_study(direction="minimize")
study_gat = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gat.optimize(objective_gat, n_trials=50)

print("Best GAT hyperparameters:", study_gat.best_params)
print("Best GAT validation loss:", study_gat.best_value)

# Estudio y optimización para GIN
#study_gin = optuna.create_study(direction="minimize")
study_gin = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gin.optimize(objective_gin, n_trials=10)

# Estudio y optimización para SAGE
#study_sage = optuna.create_study(direction="minimize")
study_sage = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_sage.optimize(objective_sage, n_trials=10)

# Imprimir los mejores parámetros para cada modelo
print("Mejores parámetros para GCN:", study_gcn.best_params)
print("Mejores parámetros para GIN:", study_gin.best_params)
print("Mejores parámetros para SAGE:", study_sage.best_params)
print("Mejores parámetros para GAT:", study_gat.best_params)


# Entrenamiento y guardado de los modelos

# Entrenar y guardar el modelo GCN
best_params = study_gcn.best_params
gcn_model = GCNLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gcn = torch.optim.Adam(gcn_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

criterion = torch.nn.BCEWithLogitsLoss()

gcn_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gcn.zero_grad()
    x = gcn_model(train_data.x, train_data.edge_index)
    out = gcn_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gcn.step()

torch.save(gcn_model.state_dict(), "P22-mejor_modelo_GCN.pth")
print("Modelo GCN guardado.")

# Entrenamiento y guardado del modelo GAT
best_params = study_gat.best_params
if 'heads' in best_params:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=best_params["heads"], dropout=best_params["dropout_rate"])
else:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=1, dropout=best_params["dropout_rate"])

optimizer_gat = torch.optim.Adam(gat_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gat_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gat.zero_grad()
    x = gat_model(train_data.x, train_data.edge_index)
    out = gat_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gat.step()

torch.save(gat_model.state_dict(), "P22-mejor_modelo_GAT.pth")
print("Modelo GAT guardado.")

# Entrenamiento y guardado del modelo GIN
best_params = study_gin.best_params
gin_model = GINLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gin = torch.optim.Adam(gin_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gin_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gin.zero_grad()
    x = gin_model(train_data.x, train_data.edge_index)
    out = gin_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gin.step()

torch.save(gin_model.state_dict(), "P22-mejor_modelo_GIN.pth")
print("P22-Modelo GIN guardado.")

# Entrenamiento y guardado del modelo SAGE
#sage_model = SAGELinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout"])
best_params = study_sage.best_params
sage_model = SAGELinkPredictor(
    in_channels=train_data.x.shape[1],  
    hidden_channels=best_params["hidden_channels"],  
    #num_layers=best_params["num_layers"],  
    #aggr=best_params["aggr_method"],  
    dropout=best_params["dropout_rate"]  
)

optimizer_sage = torch.optim.Adam(sage_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])
criterion = torch.nn.BCEWithLogitsLoss()
sage_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_sage.zero_grad()
    x = sage_model(train_data.x, train_data.edge_index)
    out = sage_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_sage.step()

torch.save(sage_model.state_dict(), "P22-mejor_modelo_SAGE.pth")
print("P22-Modelo SAGE guardado.")


# Cargar todos los modelos para evaluación
gcn_model.load_state_dict(torch.load("P22-mejor_modelo_GCN.pth"))
gcn_model.eval()

gat_model.load_state_dict(torch.load("P22-mejor_modelo_GAT.pth"))
gat_model.eval()

gin_model.load_state_dict(torch.load("P22-mejor_modelo_GIN.pth"))
gin_model.eval()

sage_model.load_state_dict(torch.load("P22-mejor_modelo_SAGE.pth"))
sage_model.eval()

# Evaluar todos los modelos
metrics_gcn = evaluate_model(gcn_model, test_data)
metrics_gat = evaluate_model(gat_model, test_data)
metrics_gin = evaluate_model(gin_model, test_data)
metrics_sage = evaluate_model(sage_model, test_data)

# Crear DataFrame para comparar las métricas
df_metrics = pd.DataFrame([metrics_gcn, metrics_gat, metrics_gin, metrics_sage], index=["GCN", "GAT", "GIN", "SAGE"])

# Mostrar las métricas de los cuatro modelos
print("\n📊 22 Métricas comparativas de GCN, GAT, GIN y SAGE:")
print(df_metrics)
df_metrics.to_csv("22-metricas_comparativas.csv", index=True)

#predicciones con test_data
#reverse_mapping = {v: k for k, v in test_data.node_mapping.items()}
reverse_mapping = get_reverse_mapping(test_data)
test_pairs = list(zip(test_data.edge_label_index[0].cpu().numpy(), test_data.edge_label_index[1].cpu().numpy()))

# Mostrar las predicciones de los cuatro modelos
if gcn_model is not None:
    with torch.no_grad():
        x_gcn = gcn_model(test_data.x, test_data.edge_index)
        gcn_predictions = gcn_model.decode(x_gcn, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gcn_predictions = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gcn_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gcn_predictions)]
        gcn_edges = sorted(gcn_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GCN:")
        for author1, author2, score in gcn_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gcn_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P22-predicciones_gcn.csv", index=False)

if gat_model is not None:
    with torch.no_grad():
        x_gat = gat_model(test_data.x, test_data.edge_index)
        gat_predictions = gat_model.decode(x_gat, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gat_predictions = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gat_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gat_predictions)]
        gat_edges = sorted(gat_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GAT:")
        for author1, author2, score in gat_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gat_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P22-predicciones_gat.csv", index=False)


if gin_model is not None:
    with torch.no_grad():
        x_gin = gin_model(test_data.x, test_data.edge_index)
        gin_predictions = gin_model.decode(x_gin, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gin_predictions = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gin_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gin_predictions)]
        gin_edges = sorted(gin_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GIN:")
        for author1, author2, score in gin_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gin_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P22-predicciones_gin.csv", index=False)

#print("Primeras características de nodos:", test_data.x[:5])
#print("Primeras conexiones (edge_index):", test_data.edge_index[:, :5])

if sage_model is not None:
    with torch.no_grad():
        x_sage = sage_model(test_data.x, test_data.edge_index)
        sage_predictions = sage_model.decode(x_sage, test_data.edge_label_index).sigmoid().cpu().numpy()
        #sage_predictions = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        sage_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, sage_predictions)]
        sage_edges = sorted(sage_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones SAGE:")
        for author1, author2, score in sage_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(sage_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P22-predicciones_sage.csv", index=False)

#print("¿GCN y GAT producen las mismas predicciones?", np.allclose(gcn_predictions, gat_predictions))
#print("¿GCN y SAGE producen las mismas predicciones?", np.allclose(gin_predictions, sage_predictions))


print("Configuración GCN:", gcn_model)
print("Configuración GAT:", gat_model)
print("Configuración GIN:", gin_model)
print("Configuración SAGE:", sage_model)


## ensamble de aqui hacia arriba ya no cambiar  #
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix

#import xgboost as xgb
# Obtener las predicciones de los modelos

true_labels = test_data.edge_label.numpy()
# Verifica la longitud de las etiquetas y las predicciones
#print("Número de etiquetas", len(true_labels))  # Debe ser el mismo tamaño que las predicciones
# Debe ser el mismo tamaño que las etiquetas

predicciones_gcn = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gcn",len(predicciones_gcn))  

predicciones_gat = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gat",len(predicciones_gat))  

predicciones_gin = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gin",len(predicciones_gin))  

predicciones_sage = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de sage",len(predicciones_sage))  

# Ya entrenado y obtenido las predicciones de los modelos base (GCN, GAT, GIN, SAGE)
X_stack = np.column_stack((predicciones_gcn, predicciones_gat, predicciones_gin, predicciones_sage))

####  optimizar con OPtuna el modelo ensamble meta_model para regresion logistica


# ----- Función genérica de entrenamiento y evaluación -----
def entrenar_y_evaluar(modelo, nombre_modelo):
    modelo.fit(X_stack, true_labels)
    predicciones = modelo.predict(X_stack)
    
    accuracy = accuracy_score(true_labels, predicciones)
    precision = precision_score(true_labels, predicciones)
    recall = recall_score(true_labels, predicciones)
    f1 = f1_score(true_labels, predicciones)
    roc_auc = roc_auc_score(true_labels, predicciones)
    conf_matrix = confusion_matrix(true_labels, predicciones)
    
    print(f"Métricas del modelo meta ({nombre_modelo}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Predicciones con nombres
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = predicciones[i]
        predicciones_autores.append((author1_name, author2_name, score))
    
    return modelo, predicciones_autores
    
# ----- Modelos Meta -----
# 1. Logistic Regression optimizado con Optuna
def objective_logistic(trial):
    meta_model = LogisticRegression(
        #C=trial.suggest_loguniform('C', 1e-5, 1e5),
        C = trial.suggest_float('C', 1e-5, 1e5, log=True),
        max_iter=trial.suggest_int('max_iter', 100, 1000),
        random_state=SEED
    )
    meta_model.fit(X_stack, true_labels)
    predicciones = meta_model.predict(X_stack)
    return roc_auc_score(true_labels, predicciones)

def objective_random_forest(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 20)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_gb(trial):
    model = GradientBoostingClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        max_depth=trial.suggest_int('max_depth', 3, 10)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_svc(trial):
    model = SVC(
        C=trial.suggest_loguniform('C', 1e-5, 1e5),
        probability=True
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_xgb(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        use_label_encoder=False,
        eval_metric='logloss'
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

# Diccionario de objetivos
objectives = {
    "LogisticRegression": objective_logistic,
    "RandomForest": objective_random_forest,
    "GradientBoosting": objective_gb,
    "SVC": objective_svc,
    "XGBoost": objective_xgb
}

# Resultados
results = {}

for name, obj in objectives.items():
    print(f"Optimizando {name}...")
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(obj, n_trials=50)
    best_params = study.best_params

    if name == "LogisticRegression":
        model = LogisticRegression(**best_params)
    elif name == "RandomForest":
        model = RandomForestClassifier(**best_params)
    elif name == "GradientBoosting":
        model = GradientBoostingClassifier(**best_params)
    elif name == "SVC":
        model = SVC(**best_params, probability=True)
    elif name == "XGBoost":
        model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss')

    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)

    # Evaluación
    accuracy = accuracy_score(true_labels, preds)
    precision = precision_score(true_labels, preds)
    recall = recall_score(true_labels, preds)
    f1 = f1_score(true_labels, preds)
    roc_auc = roc_auc_score(true_labels, preds)
    conf_matrix = confusion_matrix(true_labels, preds)

    print(f"\nMétricas del modelo meta ({name}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Guardar predicciones con nombres de autores en diccionario y en archivo
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = float(preds[i])
        predicciones_autores.append((author1_name, author2_name, score))
    results[name] = predicciones_autores
    df = pd.DataFrame(predicciones_autores, columns=["Autor1", "Autor2", "Score"])
        # Formatear la columna Score a 4 decimales
    df["Score"] = df["Score"].map(lambda x: f"{x:.4f}")
    file_name = f"P10-predicciones_meta-{name.replace(' ', '_').lower()}.csv"
    df.to_csv(file_name, index=False)


###  FIN Optimizar con OPtuna modelo ensamble




df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6730
1.0    3395
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234
Best GCN hyperparameters: {'hidden_channels': 40, 'dropout_rate': 0.42029131709525286, 'learning_rate': 0.002707258936283181, 'weight_decay': 0.007075884708681667, 'epochs': 11}
Best GCN validation loss: 0.6928912997245789
Best GAT hyperparameters: {'hidden_channels': 28, 'heads': 1, 'dropout_rate': 0.3643301344065235, 'learning_rate': 0.009739884057791966, 'weight_decay': 0.006945981553573206, 'epochs': 61}
Best GAT validation loss: 0.6931172013282776
Mejores parámetros para GCN: {'hidden_channels': 40, 'dropout_rate': 0.42029131709525286, 'learning_rate': 0.00270725893628318

C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\4011925397.py:808: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\4011925397.py:808: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\4011925397.py:808: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppDa


Métricas del modelo meta (SVC):
Accuracy: 0.7056
Precision: 0.5493
Recall: 0.4334
F1 Score: 0.4845
AUC-ROC: 0.6333
Confusion Matrix:
[[3338  668]
 [1064  814]]

Optimizando XGBoost...

Métricas del modelo meta (XGBoost):
Accuracy: 0.7820
Precision: 0.6806
Recall: 0.5969
F1 Score: 0.6360
AUC-ROC: 0.7328
Confusion Matrix:
[[3480  526]
 [ 757 1121]]



In [6]:
# 


# PROGRAMA 23 estadisticas con UNAM  LSP por nodos y fechas
df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6731
1.0    3386
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234

23 Métricas comparativas de GCN, GAT, GIN y SAGE:
      Accuracy  Precision    Recall  F1 Score   AUC-ROC    AUC-PR       MSE
GCN   0.319171   0.319171  1.000000  0.483896  0.314428  0.235676  0.250012
GAT   0.319171   0.319171  1.000000  0.483896  0.378100  0.252354  0.250001
GIN   0.319171   0.319171  1.000000  0.483896  0.569110  0.353815  0.679151
SAGE  0.491502   0.363279  0.788072  0.497312  0.671170  0.606204  0.249856
Métricas del modelo meta (LogisticRegression):
Accuracy: 0.7014
Precision: 0.5475
Recall: 0.3711
F1 Score: 0.4424
AUC-ROC: 0.6137
Confusion Matrix:
[[3430  576]
 [1181  697]]

Optimizando RandomForest...

Métricas del modelo meta (RandomForest):
Accuracy: 0.9663
Precision: 0.9531
Recall: 0.9409
F1 Score: 0.9469
AUC-ROC: 0.9596
Confusion Matrix:
[[3919   87]
 [ 111 1767]]
Métricas del modelo meta (LogisticRegression):
Accuracy: 0.7014
Precision: 0.5475
Recall: 0.3711
F1 Score: 0.4424
AUC-ROC: 0.6137
Confusion Matrix:
[[3430  576]
 [1181  697]]

Optimizando RandomForest...

Métricas del modelo meta (RandomForest):
Accuracy: 0.9663
Precision: 0.9531
Recall: 0.9409
F1 Score: 0.9469
AUC-ROC: 0.9596
Confusion Matrix:
[[3919   87]
 [ 111 1767]]

Optimizando GradientBoosting...

Métricas del modelo meta (GradientBoosting):
Accuracy: 0.9691
Precision: 0.9506
Recall: 0.9526
F1 Score: 0.9516
AUC-ROC: 0.9647
Confusion Matrix:
[[3913   93]
 [  89 1789]]

Optimizando SVC...

In [7]:
#PROGRAMA 23 estadisticas con UNAM  LSP por nodos y fechas
#source	target	connected	lenght_short_path	jaccard_coefficient	adamic_adar_index	resource_allocation	preferential_attachment	sum_of_neighbors	log_secundary_neighbors	clustering_index_sum


from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import torch
import torch.nn.init as init
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv
import optuna
import logging
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error
from sklearn.metrics import average_precision_score
from tabulate import tabulate
from torch_geometric.nn import global_mean_pool
import torch.nn.functional as F
#REDUCIENDO ALEATORIEDAD
import random
import numpy as np
import torch

SEED = 41
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)

#lenght_short_path	jaccard_coefficient	adamic_adar_index	
#resource_allocation	preferential_attachment	sum_of_neighbors	log_secundary_neighbors	clustering_index_summ(8)

# Cargar datasets

file_paths = {
    "sample_train_2000_2020_LSP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_train_2000_2020_LSP.csv",
    "sample_val_2021_2022_LSP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_val_2021_2022_LSP.csv",
    "sample_test_2023_2024_LSP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_test_2023_2024_LSP.csv",
}
df_train = pd.read_csv(file_paths["sample_train_2000_2020_LSP"])
df_val   = pd.read_csv(file_paths["sample_val_2021_2022_LSP"])
df_test  = pd.read_csv(file_paths["sample_test_2023_2024_LSP"])
                                              

# Filtrar aristas conectadas

#df_train = df_train[df_train['connected'] == 1].reset_index(drop=True)
#df_val = df_val[df_val['connected'] == 1].reset_index(drop=True)
#df_test = df_test[df_test['connected'] == 1].reset_index(drop=True)

# Reemplazar NaN en 'lenght_short_path' por mediana
df_train['lenght_short_path'] = df_train['lenght_short_path'].fillna(df_train['lenght_short_path'].median())
df_val['lenght_short_path'] = df_val['lenght_short_path'].fillna(df_val['lenght_short_path'].median())
df_test['lenght_short_path'] = df_test['lenght_short_path'].fillna(df_test['lenght_short_path'].median())

# Normalizar las columnas numéricas, se hace aqui o en el splitRandom, estan las dos opciones, esta da mejores resultados

#lenght_short_path	jaccard_coefficient	adamic_adar_index	resource_allocation	preferential_attachment	sum_of_neighbors	log_secundary_neighbors	clustering_index_sum (8)

numeric_columns = [  'lenght_short_path', 'jaccard_coefficient',	'adamic_adar_index',	
                     'resource_allocation',	'preferential_attachment',	'sum_of_neighbors',	
                     'log_secundary_neighbors',	'clustering_index_sum']

scaler = MinMaxScaler()
df_train[numeric_columns] = scaler.fit_transform(df_train[numeric_columns])
# Transform val/test con el mismo scaler (no fit)
df_test[numeric_columns] = scaler.transform(df_test[numeric_columns])
df_val[numeric_columns] = scaler.transform(df_val[numeric_columns])

df_train[numeric_columns] = df_train[numeric_columns].fillna(0)
df_val[numeric_columns] = df_val[numeric_columns].fillna(0)
df_test[numeric_columns] = df_test[numeric_columns].fillna(0)

def filter_edges_for_existing_nodes(G_train, df_edges, verbose=True):
    """
    Filtra enlaces que tengan nodos no presentes en el grafo de entrenamiento.
    """
    train_nodes = set(G_train.nodes())
    mask = df_edges['source'].isin(train_nodes) & df_edges['target'].isin(train_nodes)
    df_filtered = df_edges[mask].copy()
    ignored_edges = df_edges[~mask].copy()
    
    if verbose:
        print(f"Total enlaces: {len(df_edges)}")
        print(f"Enlaces válidos: {len(df_filtered)}")
        print(f"Enlaces ignorados (nodos faltantes): {len(ignored_edges)}")
    
    return df_filtered, ignored_edges
    
# Construir grafo de entrenamiento
def build_graph(df):
    G = nx.Graph()
    for _, row in df.iterrows():
        G.add_edge(row['source'], row['target'], weight=row['connected'])
    return G

# Asignar características a los nodos
def add_node_features(G, df):
    node_features = {}
    for _, row in df.iterrows():
        for node in [row['source'], row['target']]:
            if node not in node_features:
                node_features[node] = []
            node_features[node] = [
                row['lenght_short_path'], 
                row['jaccard_coefficient'], 
                row['adamic_adar_index'],
                row['resource_allocation'],
                row['preferential_attachment'],
                row['sum_of_neighbors'],
                row['log_secundary_neighbors'],
                row['clustering_index_sum']
            ]
    for node, features in node_features.items():
        if node in G.nodes:
            G.nodes[node]['features'] = features
    return G

G_train = build_graph(df_train[df_train['connected'] == 1])  # solo positivos
G_train = add_node_features(G_train, df_train)

G_val = build_graph(df_val[df_val['connected'] == 1])
G_val = add_node_features(G_val, df_val)

G_test = build_graph(df_test[df_test['connected'] == 1])
G_test = add_node_features(G_test, df_test)

extra_nodos = set(df_train.source) | set(df_train.target) \
              | set(df_val.source) | set(df_val.target) \
              | set(df_test.source) | set(df_test.target)

faltantes_train = [n for n in extra_nodos if n not in G_train.nodes()]
faltantes_val   = [n for n in extra_nodos if n not in G_val.nodes()]
faltantes_test  = [n for n in extra_nodos if n not in G_test.nodes()]

def get_reverse_mapping(data):
    """Devuelve un diccionario índice → nodo original a partir de data.node_mapping_dict"""
    return {idx: node for node, idx in data.node_mapping_dict.items()}

# Convertir a PyTorch Geometric
# ===========================================
# Ejecutar para tus tres datasets segun lo que desees que haga que agregue nodos faltantes o no
# ===========================================

import torch
from torch_geometric.data import Data

def graph_to_pyg(G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=False):
    """
    Convierte tres grafos NetworkX y sus DataFrames de aristas a objetos PyG Data,
    revisando los tres conjuntos para agregar nodos faltantes si se desea.

    Parámetros:
        G_train, G_val, G_test : grafos NetworkX para cada split
        df_train, df_val, df_test : DataFrames de aristas ['source','target','connected']
        add_missing_nodes : bool, si True agrega nodos que aparecen en cualquier DF y no están en los grafos

    Retorna:
        train_data, val_data, test_data : objetos torch_geometric.data.Data
    """

    # =============================
    # RECOLECTAR TODOS LOS NODOS NECESARIOS
    # =============================
    if add_missing_nodes:
        extra_nodos = set(df_train.source) | set(df_train.target)
        extra_nodos |= set(df_val.source) | set(df_val.target)
        extra_nodos |= set(df_test.source) | set(df_test.target)
        
        for G in [G_train, G_val, G_test]:
            faltantes = [n for n in extra_nodos if n not in G.nodes()]
            if faltantes:
                feature_dim = len(next(iter(G.nodes(data='features')))[1])
                for n in faltantes:
                    G.add_node(n, features=[0.0]*feature_dim)
            # No imprimimos nodos faltantes para no saturar salida
    if not add_missing_nodes:
        df_train = filter_edges(df_train, G_train)
        df_val   = filter_edges(df_val, G_val)
        df_test  = filter_edges(df_test, G_test)
        
    # =============================
    # FUNCIÓN INTERNA PARA CONVERTIR CADA SPLIT
    # =============================
    def _convert(G, df):
        if df is None:
            return None
        mapping = {node: i for i, node in enumerate(G.nodes())}
        edge_index = torch.tensor(
            [[mapping[u], mapping[v]] for u, v in zip(df.source, df.target)],
            dtype=torch.long
        ).t().contiguous()
        node_features = torch.tensor(
            [G.nodes[node]['features'] for node in G.nodes()],
            dtype=torch.float
        )
        edge_label = torch.tensor(df.connected.values, dtype=torch.float)
        data = Data(
            x=node_features,
            edge_index=edge_index,
            edge_label=edge_label,
            edge_label_index=edge_index.clone(),
            num_nodes=len(G.nodes())
        )
        # Guardar mapping como atributo dict explícito
        data.node_mapping_dict = mapping
        return data

    train_data = _convert(G_train, df_train)
    val_data   = _convert(G_val, df_val)
    test_data  = _convert(G_test, df_test)

    return train_data, val_data, test_data


train_data, val_data, test_data = graph_to_pyg(
    G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=True
)
    

def sort_edge_index(data):
    sorted_indices = torch.argsort(data.edge_index[1])  # Ordenar por destino
    data.edge_index = data.edge_index[:, sorted_indices]  # Aplicar orden
    return data

    # Ordenar los conjuntos de datos por el modelo SAGE
train_data = sort_edge_index(train_data)
val_data = sort_edge_index(val_data)
test_data = sort_edge_index(test_data)


import numpy as np
print("df_train connected counts:\n", df_train['connected'].value_counts())
print("df_val connected counts:\n", df_val['connected'].value_counts())
print("df_test connected counts:\n", df_test['connected'].value_counts())

true_labels = test_data.edge_label.cpu().numpy()
print("true_labels unique/counts:", np.unique(true_labels, return_counts=True))


# eliminar mensajes de los trials de optuna
optuna.logging.set_verbosity(optuna.logging.ERROR)

# =============================
# CALCULAR DENSIDADES
# =============================


def calc_density(G):
    if G.number_of_nodes() == 0:
        return 0
    return nx.density(G)

def density_for_dataset(data, description):
   # nodes = [reverse_mapping[i] for i in range(data.num_nodes)]
    reverse_mapping = get_reverse_mapping(data) 
    G_tmp = nx.Graph()
    for u, v in data.edge_index.t().numpy():
        G_tmp.add_edge(reverse_mapping[u], reverse_mapping[v])
    density = calc_density(G_tmp)
    print(f"Densidad {description}: {density:.4f}")
    return density

density_train = density_for_dataset(train_data, "Train")
density_val = density_for_dataset(val_data, "Validation")
density_test = density_for_dataset(test_data, "Test")



# Definir modelos
class GCNLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GATLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels=1, heads=1, dropout=0.0):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, dropout=dropout)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GINLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GINConv(torch.nn.Sequential(torch.nn.Linear(in_channels, hidden_channels), torch.nn.ReLU()))
        self.conv2 = GINConv(torch.nn.Sequential(torch.nn.Linear(hidden_channels, 1)))
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)



class SAGELinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

#evaluacion de modelos
def evaluate_model(model, data):
    model.eval()
    with torch.no_grad():
        preds = model(data.x, data.edge_index)
        edge_emb = preds[data.edge_label_index[0]] * preds[data.edge_label_index[1]]
        y_pred = torch.sigmoid(edge_emb.sum(dim=1)).cpu().numpy()

    y_true = data.edge_label.cpu().numpy().flatten()  # ya alineado
    assert len(y_true) == len(y_pred), f"No coinciden: {len(y_true)} vs {len(y_pred)}"

    return {
        "Accuracy": accuracy_score(y_true, y_pred > 0.5),
        "Precision": precision_score(y_true, y_pred > 0.5, zero_division=1),
        "Recall": recall_score(y_true, y_pred > 0.5),
        "F1 Score": f1_score(y_true, y_pred > 0.5),
        "AUC-ROC": roc_auc_score(y_true, y_pred),
        "AUC-PR": average_precision_score(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred)
    }


# Entrenamiento y evaluación

def train_and_evaluate_model(model, train_data, val_data, epochs, criterion, lr, weight_decay):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()

    # Entrenamiento
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        x = model(train_data.x, train_data.edge_index)  # Pasa las features y las aristas
        out = model.decode(x, train_data.edge_label_index).squeeze()  # Decodifica los enlaces
        loss = criterion(out, train_data.edge_label.float())  # Calcula la pérdida
        loss.backward()
        optimizer.step()

    # Evaluación
    model.eval()
    with torch.no_grad():
        x_val = model(val_data.x, val_data.edge_index)
        val_out = model.decode(x_val, val_data.edge_label_index).squeeze()
        val_loss = criterion(val_out, val_data.edge_label.float()).item()  # Calcula la pérdida en validación

    return val_loss

   
# Objetivo para optimización con Optuna
def objective_gcn(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)  
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GCNLinkPredictor(
        in_channels=8,  # Número de características de entrada
        hidden_channels=hidden_channels,
        dropout=dropout_rate,
    )

    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
   # val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)  
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss

def objective_gat(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    heads = trial.suggest_int("heads", 1, 8)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GATLinkPredictor(
        in_channels=8,  # Número de características de entrada
        hidden_channels=hidden_channels,
        heads=heads,
        dropout=dropout_rate
    )
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
    
def objective_gin(trial):
    # Hiperparámetros a optimizar
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GIN
    model = GINLinkPredictor(
        in_channels=8,  # Número de características de entrada, antes 15 estadisticas, ahora 9
        hidden_channels=hidden_channels,  # Número de canales ocultos
        dropout=dropout_rate  # Tasa de dropout
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss


def objective_sage(trial):
    # Hiperparámetros para optimización
    hidden_channels = trial.suggest_int("hidden_channels", 16, 64)
    num_layers = trial.suggest_int("num_layers", 2, 4)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    aggr_method = trial.suggest_categorical("aggr_method", ["mean", "max", "lstm"])
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GraphSAGE
    
    model = SAGELinkPredictor(
        in_channels=8,  # Ajusta según tu modelo al numero de caracteristicas de entrada (estadisticas,no los campos)
        hidden_channels=hidden_channels,
        #num_layers=num_layers,
        #aggr=aggr_method,
        dropout=dropout_rate
    )

    # Optimización
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

      # Llama a la función con todos los argumentos
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
           

# Optuna para GCN
#study_gcn = optuna.create_study(direction="minimize")
study_gcn = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study_gcn.optimize(objective_gcn, n_trials=50)

print("Best GCN hyperparameters:", study_gcn.best_params)
print("Best GCN validation loss:", study_gcn.best_value)

# Optuna para GAT
#study_gat = optuna.create_study(direction="minimize")
study_gat = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study_gat.optimize(objective_gat, n_trials=50)

print("Best GAT hyperparameters:", study_gat.best_params)
print("Best GAT validation loss:", study_gat.best_value)

# Estudio y optimización para GIN
#study_gin = optuna.create_study(direction="minimize")
study_gin = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study_gin.optimize(objective_gin, n_trials=10)

# Estudio y optimización para SAGE
#study_sage = optuna.create_study(direction="minimize")
study_sage = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study_sage.optimize(objective_sage, n_trials=10)

# Imprimir los mejores parámetros para cada modelo
print("Mejores parámetros para GCN:", study_gcn.best_params)
print("Mejores parámetros para GIN:", study_gin.best_params)
print("Mejores parámetros para SAGE:", study_sage.best_params)
print("Mejores parámetros para GAT:", study_gat.best_params)


# Entrenamiento y guardado de los modelos

# Entrenar y guardar el modelo GCN
best_params = study_gcn.best_params
gcn_model = GCNLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gcn = torch.optim.Adam(gcn_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

criterion = torch.nn.BCEWithLogitsLoss()

gcn_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gcn.zero_grad()
    x = gcn_model(train_data.x, train_data.edge_index)
    out = gcn_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gcn.step()

torch.save(gcn_model.state_dict(), "P23-mejor_modelo_GCN.pth")
print("Modelo GCN guardado.")

# Entrenamiento y guardado del modelo GAT
best_params = study_gat.best_params
if 'heads' in best_params:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=best_params["heads"], dropout=best_params["dropout_rate"])
else:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=1, dropout=best_params["dropout_rate"])

optimizer_gat = torch.optim.Adam(gat_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gat_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gat.zero_grad()
    x = gat_model(train_data.x, train_data.edge_index)
    out = gat_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gat.step()

torch.save(gat_model.state_dict(), "P23-mejor_modelo_GAT.pth")
print("Modelo GAT guardado.")

# Entrenamiento y guardado del modelo GIN
best_params = study_gin.best_params
gin_model = GINLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gin = torch.optim.Adam(gin_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gin_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gin.zero_grad()
    x = gin_model(train_data.x, train_data.edge_index)
    out = gin_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gin.step()

torch.save(gin_model.state_dict(), "P23-mejor_modelo_GIN.pth")
print("Modelo GIN guardado.")

# Entrenamiento y guardado del modelo SAGE
#sage_model = SAGELinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout"])
best_params = study_sage.best_params
sage_model = SAGELinkPredictor(
    in_channels=train_data.x.shape[1],  
    hidden_channels=best_params["hidden_channels"],  
    #num_layers=best_params["num_layers"],  
    #aggr=best_params["aggr_method"],  
    dropout=best_params["dropout_rate"]  
)

optimizer_sage = torch.optim.Adam(sage_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])
criterion = torch.nn.BCEWithLogitsLoss()
sage_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_sage.zero_grad()
    x = sage_model(train_data.x, train_data.edge_index)
    out = sage_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_sage.step()

torch.save(sage_model.state_dict(), "P23-mejor_modelo_SAGE.pth")
print("Modelo SAGE guardado.")


# Cargar todos los modelos para evaluación
gcn_model.load_state_dict(torch.load("P23-mejor_modelo_GCN.pth"))
gcn_model.eval()

gat_model.load_state_dict(torch.load("P23-mejor_modelo_GAT.pth"))
gat_model.eval()

gin_model.load_state_dict(torch.load("P23-mejor_modelo_GIN.pth"))
gin_model.eval()

sage_model.load_state_dict(torch.load("P23-mejor_modelo_SAGE.pth"))
sage_model.eval()

# Evaluar todos los modelos
metrics_gcn = evaluate_model(gcn_model, test_data)
metrics_gat = evaluate_model(gat_model, test_data)
metrics_gin = evaluate_model(gin_model, test_data)
metrics_sage = evaluate_model(sage_model, test_data)

# Crear DataFrame para comparar las métricas
df_metrics = pd.DataFrame([metrics_gcn, metrics_gat, metrics_gin, metrics_sage], index=["GCN", "GAT", "GIN", "SAGE"])

# Mostrar las métricas de los cuatro modelos
print("\n📊 23 Métricas comparativas de GCN, GAT, GIN y SAGE:")
print(df_metrics)
df_metrics.to_csv("23metricas_comparativas.csv", index=True)

#predicciones con test_data
#reverse_mapping = {v: k for k, v in test_data.node_mapping.items()}
reverse_mapping = get_reverse_mapping(test_data)
test_pairs = list(zip(test_data.edge_label_index[0].cpu().numpy(), test_data.edge_label_index[1].cpu().numpy()))

# Mostrar las predicciones de los cuatro modelos
if gcn_model is not None:
    with torch.no_grad():
        x_gcn = gcn_model(test_data.x, test_data.edge_index)
        gcn_predictions = gcn_model.decode(x_gcn, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gcn_predictions = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gcn_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gcn_predictions)]
        gcn_edges = sorted(gcn_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GCN:")
        for author1, author2, score in gcn_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gcn_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P23-predicciones_gcn.csv", index=False)

if gat_model is not None:
    with torch.no_grad():
        x_gat = gat_model(test_data.x, test_data.edge_index)
        gat_predictions = gat_model.decode(x_gat, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gat_predictions = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gat_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gat_predictions)]
        gat_edges = sorted(gat_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GAT:")
        for author1, author2, score in gat_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gat_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P23-predicciones_gat.csv", index=False)


if gin_model is not None:
    with torch.no_grad():
        x_gin = gin_model(test_data.x, test_data.edge_index)
        gin_predictions = gin_model.decode(x_gin, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gin_predictions = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gin_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gin_predictions)]
        gin_edges = sorted(gin_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GIN:")
        for author1, author2, score in gin_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gin_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P23-predicciones_gin.csv", index=False)

#print("Primeras características de nodos:", test_data.x[:5])
#print("Primeras conexiones (edge_index):", test_data.edge_index[:, :5])

if sage_model is not None:
    with torch.no_grad():
        x_sage = sage_model(test_data.x, test_data.edge_index)
        sage_predictions = sage_model.decode(x_sage, test_data.edge_label_index).sigmoid().cpu().numpy()
        #sage_predictions = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        sage_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, sage_predictions)]
        sage_edges = sorted(sage_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones SAGE:")
        for author1, author2, score in sage_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(sage_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P23-predicciones_sage.csv", index=False)

#print("¿GCN y GAT producen las mismas predicciones?", np.allclose(gcn_predictions, gat_predictions))
#print("¿GCN y SAGE producen las mismas predicciones?", np.allclose(gin_predictions, sage_predictions))


print("Configuración GCN:", gcn_model)
print("Configuración GAT:", gat_model)
print("Configuración GIN:", gin_model)
print("Configuración SAGE:", sage_model)


## ensamble de aqui hacia arriba ya no cambiar #
from sklearn.linear_model import LogisticRegression
# Obtener las predicciones de los modelos

true_labels = test_data.edge_label.numpy()
# Verifica la longitud de las etiquetas y las predicciones
#print("Número de etiquetas", len(true_labels))  # Debe ser el mismo tamaño que las predicciones
# Debe ser el mismo tamaño que las etiquetas

predicciones_gcn = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gcn",len(predicciones_gcn))  

predicciones_gat = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gat",len(predicciones_gat))  

predicciones_gin = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gin",len(predicciones_gin))  

predicciones_sage = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de sage",len(predicciones_sage))  

X_stack = np.column_stack((predicciones_gcn, predicciones_gat, predicciones_gin, predicciones_sage))

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, auc, confusion_matrix


# Ensamble de aquí hacia arriba ya no cambiar #
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix

#import xgboost as xgb
# Obtener las predicciones de los modelos

true_labels = test_data.edge_label.numpy()
# Verifica la longitud de las etiquetas y las predicciones
#print("Número de etiquetas", len(true_labels))  # Debe ser el mismo tamaño que las predicciones
# Debe ser el mismo tamaño que las etiquetas

predicciones_gcn = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gcn",len(predicciones_gcn))  

predicciones_gat = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gat",len(predicciones_gat))  

predicciones_gin = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de gin",len(predicciones_gin))  

predicciones_sage = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
#print("Número de predicciones de sage",len(predicciones_sage))  

# Ya entrenado y obtenido las predicciones de los modelos base (GCN, GAT, GIN, SAGE)
X_stack = np.column_stack((predicciones_gcn, predicciones_gat, predicciones_gin, predicciones_sage))

####  optimizar con OPtuna el modelo ensamble meta_model para regresion logistica


# ----- Función genérica de entrenamiento y evaluación -----
def entrenar_y_evaluar(modelo, nombre_modelo):
    modelo.fit(X_stack, true_labels)
    predicciones = modelo.predict(X_stack)
    
    accuracy = accuracy_score(true_labels, predicciones)
    precision = precision_score(true_labels, predicciones)
    recall = recall_score(true_labels, predicciones)
    f1 = f1_score(true_labels, predicciones)
    roc_auc = roc_auc_score(true_labels, predicciones)
    conf_matrix = confusion_matrix(true_labels, predicciones)
    
    print(f"Métricas del modelo meta ({nombre_modelo}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Predicciones con nombres
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = predicciones[i]
        predicciones_autores.append((author1_name, author2_name, score))
    
    return modelo, predicciones_autores
    
# ----- Modelos Meta -----
# 1. Logistic Regression optimizado con Optuna
def objective_logistic(trial):
    meta_model = LogisticRegression(
        #C=trial.suggest_loguniform('C', 1e-5, 1e5),
        C = trial.suggest_float('C', 1e-5, 1e5, log=True),
        max_iter=trial.suggest_int('max_iter', 100, 1000),
        random_state=SEED
    )
    meta_model.fit(X_stack, true_labels)
    predicciones = meta_model.predict(X_stack)
    return roc_auc_score(true_labels, predicciones)

def objective_random_forest(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 20)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_gb(trial):
    model = GradientBoostingClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        max_depth=trial.suggest_int('max_depth', 3, 10)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_svc(trial):
    model = SVC(
        C=trial.suggest_loguniform('C', 1e-5, 1e5),
        probability=True
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_xgb(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        use_label_encoder=False,
        eval_metric='logloss'
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

# Diccionario de objetivos
objectives = {
    "LogisticRegression": objective_logistic,
    "RandomForest": objective_random_forest,
    "GradientBoosting": objective_gb,
    "SVC": objective_svc,
    "XGBoost": objective_xgb
}

# Resultados
results = {}

for name, obj in objectives.items():
    print(f"Optimizando {name}...")
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(obj, n_trials=50)
    best_params = study.best_params

    if name == "LogisticRegression":
        model = LogisticRegression(**best_params)
    elif name == "RandomForest":
        model = RandomForestClassifier(**best_params)
    elif name == "GradientBoosting":
        model = GradientBoostingClassifier(**best_params)
    elif name == "SVC":
        model = SVC(**best_params, probability=True)
    elif name == "XGBoost":
        model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss')

    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)

    # Evaluación
    accuracy = accuracy_score(true_labels, preds)
    precision = precision_score(true_labels, preds)
    recall = recall_score(true_labels, preds)
    f1 = f1_score(true_labels, preds)
    roc_auc = roc_auc_score(true_labels, preds)
    conf_matrix = confusion_matrix(true_labels, preds)

    print(f"\nMétricas del modelo meta ({name}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Guardar predicciones con nombres de autores en diccionario y en archivo
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = float(preds[i])
        predicciones_autores.append((author1_name, author2_name, score))
    results[name] = predicciones_autores
    df = pd.DataFrame(predicciones_autores, columns=["Autor1", "Autor2", "Score"])
        # Formatear la columna Score a 4 decimales
    df["Score"] = df["Score"].map(lambda x: f"{x:.4f}")
    file_name = f"P23-predicciones_meta-{name.replace(' ', '_').lower()}.csv"
    df.to_csv(file_name, index=False)


###  FIN Optimizar con OPtuna modelo ensamble




df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6731
1.0    3386
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234
Best GCN hyperparameters: {'hidden_channels': 44, 'dropout_rate': 0.023047910336305877, 'learning_rate': 0.006800480786994529, 'weight_decay': 0.00044426016159107313, 'epochs': 20}
Best GCN validation loss: 0.6929163336753845
Best GAT hyperparameters: {'hidden_channels': 25, 'heads': 2, 'dropout_rate': 0.1614457649994026, 'learning_rate': 0.0064649801956020835, 'weight_decay': 0.009994722579891792, 'epochs': 35}
Best GAT validation loss: 0.6931162476539612
Mejores parámetros para GCN: {'hidden_channels': 44, 'dropout_rate': 0.023047910336305877, 'learning_rate': 0.006800480786

C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\3138216008.py:818: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\3138216008.py:818: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\3138216008.py:818: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppDa


Métricas del modelo meta (SVC):
Accuracy: 0.7056
Precision: 0.5493
Recall: 0.4334
F1 Score: 0.4845
AUC-ROC: 0.6333
Confusion Matrix:
[[3338  668]
 [1064  814]]

Optimizando XGBoost...

Métricas del modelo meta (XGBoost):
Accuracy: 0.7820
Precision: 0.6806
Recall: 0.5969
F1 Score: 0.6360
AUC-ROC: 0.7328
Confusion Matrix:
[[3480  526]
 [ 757 1121]]



#EL SIGUIENTE PROGRAMA 24 EXPERIMENTA CON LA EFECTIVIDAD DE GLOBAL STRUCTURAL PROXIMITY para grafos separados por nodos unam

# programa 24
df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6730
1.0    3395
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234


Métricas comparativas de GCN, GAT, GIN y SAGE:
      Accuracy  Precision    Recall  F1 Score   AUC-ROC    AUC-PR       MSE
GCN   0.357580   0.273894  0.613419  0.378698  0.374077  0.257878  0.250000
GAT   0.680829   1.000000  0.000000  0.000000  0.500000  0.319171  0.250000
GIN   0.319171   0.319171  1.000000  0.483896  0.701464  0.626719  0.251696
SAGE  0.319171   0.319171  1.000000  0.483896  0.529347  0.330578  0.251407

24 Métricas del modelo meta (LogisticRegression):
Accuracy: 0.6808
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
AUC-ROC: 0.5000
Confusion Matrix:
[[4006    0]
 [1878    0]]

24 Métricas del modelo meta (RandomForest):
Accuracy: 0.6808
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
AUC-ROC: 0.5000
Confusion Matrix:
[[4006    0]
 [1878    0]]

24 Métricas del modelo meta (GradientBoosting):
Accuracy: 0.6808
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
AUC-ROC: 0.5000
Confusion Matrix:
[[4006    0]
 [1878    0]]

24 Métricas del modelo meta (SVC):
Accuracy: 0.6808
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
AUC-ROC: 0.5000
Confusion Matrix:
[[4006    0]
 [1878    0]]

Optimizando XGBoost...

24 Métricas del modelo meta (XGBoost):
Accuracy: 0.6808
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
AUC-ROC: 0.5000
Confusion Matrix:
[[4006    0]
 [1878    0]]

In [9]:
#PROGRAMA 24 ESTADISTICAS ESTRUCTURALES GLOBALES DE LA UNAM Y METAMODELOS
#community_cn community_ra

from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import torch
import torch.nn.init as init
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv
import optuna
import logging
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error
from sklearn.metrics import average_precision_score
from tabulate import tabulate
from torch_geometric.nn import global_mean_pool
import torch.nn.functional as F


#REDUCIENDO ALEATORIEDAD

import random
import numpy as np
import torch

SEED = 41
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)

# Cargar datasets


file_paths = {
    "sample_train_2000_2020_GSP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_train_2000_2020_GSP.csv",
    "sample_val_2021_2022_GSP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_val_2021_2022_GSP.csv",
    "sample_test_2023_2024_GSP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_test_2023_2024_GSP.csv",
}
df_train = pd.read_csv(file_paths["sample_train_2000_2020_GSP"])
df_val   = pd.read_csv(file_paths["sample_val_2021_2022_GSP"])
df_test  = pd.read_csv(file_paths["sample_test_2023_2024_GSP"])
                                              

# Filtrar aristas conectadas

#df_train = df_train[df_train['connected'] == 1].reset_index(drop=True)
#df_val = df_val[df_val['connected'] == 1].reset_index(drop=True)
#df_test = df_test[df_test['connected'] == 1].reset_index(drop=True)

# Normalizar las columnas numéricas, se hace aqui o en el splitRandom, estan las dos opciones, esta da mejores resultados
numeric_columns = [  'community_cn',    
                     'community_ra']

scaler = MinMaxScaler()
df_train[numeric_columns] = scaler.fit_transform(df_train[numeric_columns])
# Transform val/test con el mismo scaler (no fit)
df_test[numeric_columns] = scaler.transform(df_test[numeric_columns])
df_val[numeric_columns] = scaler.transform(df_val[numeric_columns])

df_train[numeric_columns] = df_train[numeric_columns].fillna(0)
df_val[numeric_columns] = df_val[numeric_columns].fillna(0)
df_test[numeric_columns] = df_test[numeric_columns].fillna(0)

def filter_edges_for_existing_nodes(G_train, df_edges, verbose=True):
    """
    Filtra enlaces que tengan nodos no presentes en el grafo de entrenamiento.
    """
    train_nodes = set(G_train.nodes())
    mask = df_edges['source'].isin(train_nodes) & df_edges['target'].isin(train_nodes)
    df_filtered = df_edges[mask].copy()
    ignored_edges = df_edges[~mask].copy()
    
    if verbose:
        print(f"Total enlaces: {len(df_edges)}")
        print(f"Enlaces válidos: {len(df_filtered)}")
        print(f"Enlaces ignorados (nodos faltantes): {len(ignored_edges)}")
    
    return df_filtered, ignored_edges

# Construir grafo de entrenamiento
def build_graph(df):
    G = nx.Graph()
    for _, row in df.iterrows():
        G.add_edge(row['source'], row['target'], weight=row['connected'])
    return G


# Asignar características a los nodos
def add_node_features(G, df):
    node_features = {}
    for _, row in df.iterrows():
        for node in [row['source'], row['target']]:
            if node not in node_features:
                node_features[node] = []
            node_features[node] = [
                row['community_cn'], 
                row['community_ra']
               ]
            
    for node, features in node_features.items():
        if node in G.nodes:
            G.nodes[node]['features'] = features
    return G



G_train = build_graph(df_train[df_train['connected'] == 1])  # solo positivos
G_train = add_node_features(G_train, df_train)

G_val = build_graph(df_val[df_val['connected'] == 1])
G_val = add_node_features(G_val, df_val)

G_test = build_graph(df_test[df_test['connected'] == 1])
G_test = add_node_features(G_test, df_test)

extra_nodos = set(df_train.source) | set(df_train.target) \
              | set(df_val.source) | set(df_val.target) \
              | set(df_test.source) | set(df_test.target)

faltantes_train = [n for n in extra_nodos if n not in G_train.nodes()]
faltantes_val   = [n for n in extra_nodos if n not in G_val.nodes()]
faltantes_test  = [n for n in extra_nodos if n not in G_test.nodes()]


def get_reverse_mapping(data):
    """Devuelve un diccionario índice → nodo original a partir de data.node_mapping_dict"""
    return {idx: node for node, idx in data.node_mapping_dict.items()}


# Convertir a PyTorch Geometric
# ===========================================
# Ejecutar para tus tres datasets segun lo que desees que haga que agregue nodos faltantes o no
# ===========================================

import torch
from torch_geometric.data import Data

def graph_to_pyg(G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=False):
    """
    Convierte tres grafos NetworkX y sus DataFrames de aristas a objetos PyG Data,
    revisando los tres conjuntos para agregar nodos faltantes si se desea.

    Parámetros:
        G_train, G_val, G_test : grafos NetworkX para cada split
        df_train, df_val, df_test : DataFrames de aristas ['source','target','connected']
        add_missing_nodes : bool, si True agrega nodos que aparecen en cualquier DF y no están en los grafos

    Retorna:
        train_data, val_data, test_data : objetos torch_geometric.data.Data
    """

    # =============================
    # RECOLECTAR TODOS LOS NODOS NECESARIOS
    # =============================
    # =============================
    # RECOLECTAR TODOS LOS NODOS NECESARIOS
    # =============================
    if add_missing_nodes:
        extra_nodos = set(df_train.source) | set(df_train.target)
        extra_nodos |= set(df_val.source) | set(df_val.target)
        extra_nodos |= set(df_test.source) | set(df_test.target)
        
        for G in [G_train, G_val, G_test]:
            faltantes = [n for n in extra_nodos if n not in G.nodes()]
            if faltantes:
                feature_dim = len(next(iter(G.nodes(data='features')))[1])
                for n in faltantes:
                    G.add_node(n, features=[0.0]*feature_dim)
            # No imprimimos nodos faltantes para no saturar salida
    if not add_missing_nodes:
        df_train = filter_edges(df_train, G_train)
        df_val   = filter_edges(df_val, G_val)
        df_test  = filter_edges(df_test, G_test)
        
    # =============================
    # FUNCIÓN INTERNA PARA CONVERTIR CADA SPLIT
    # =============================
    def _convert(G, df):
        if df is None:
            return None
        mapping = {node: i for i, node in enumerate(G.nodes())}
        edge_index = torch.tensor(
            [[mapping[u], mapping[v]] for u, v in zip(df.source, df.target)],
            dtype=torch.long
        ).t().contiguous()
        node_features = torch.tensor(
            [G.nodes[node]['features'] for node in G.nodes()],
            dtype=torch.float
        )
        edge_label = torch.tensor(df.connected.values, dtype=torch.float)
        data = Data(
            x=node_features,
            edge_index=edge_index,
            edge_label=edge_label,
            edge_label_index=edge_index.clone(),
            num_nodes=len(G.nodes())
        )
        # Guardar mapping como atributo dict explícito
        data.node_mapping_dict = mapping
        return data

    train_data = _convert(G_train, df_train)
    val_data   = _convert(G_val, df_val)
    test_data  = _convert(G_test, df_test)

    return train_data, val_data, test_data

train_data, val_data, test_data = graph_to_pyg(
    G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=True
)

def sort_edge_index(data):
    sorted_indices = torch.argsort(data.edge_index[1])
    data.edge_index = data.edge_index[:, sorted_indices]
    return data
    
# Ordenar los conjuntos de datos por el modelo SAGE
train_data = sort_edge_index(train_data)
val_data = sort_edge_index(val_data)
test_data = sort_edge_index(test_data)




import numpy as np
print("df_train connected counts:\n", df_train['connected'].value_counts())
print("df_val connected counts:\n", df_val['connected'].value_counts())
print("df_test connected counts:\n", df_test['connected'].value_counts())

true_labels = test_data.edge_label.cpu().numpy()
print("true_labels unique/counts:", np.unique(true_labels, return_counts=True))

# =============================
# CALCULAR DENSIDADES
# =============================


def calc_density(G):
    if G.number_of_nodes() == 0:
        return 0
    return nx.density(G)

def density_for_dataset(data, description):
   # nodes = [reverse_mapping[i] for i in range(data.num_nodes)]
    reverse_mapping = get_reverse_mapping(data) 
    G_tmp = nx.Graph()
    for u, v in data.edge_index.t().numpy():
        G_tmp.add_edge(reverse_mapping[u], reverse_mapping[v])
    density = calc_density(G_tmp)
    print(f"Densidad {description}: {density:.4f}")
    return density

density_train = density_for_dataset(train_data, "Train")
density_val = density_for_dataset(val_data, "Validation")
density_test = density_for_dataset(test_data, "Test")

# Definir modelos

# eliminar mensajes de los trials de optuna
optuna.logging.set_verbosity(optuna.logging.ERROR)


class GCNLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GATLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels=1, heads=1, dropout=0.0):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, dropout=dropout)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GINLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GINConv(torch.nn.Sequential(torch.nn.Linear(in_channels, hidden_channels), torch.nn.ReLU()))
        self.conv2 = GINConv(torch.nn.Sequential(torch.nn.Linear(hidden_channels, 1)))
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)



class SAGELinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

#evaluacion de modelos
def evaluate_model(model, data):
    model.eval()
    with torch.no_grad():
        preds = model(data.x, data.edge_index)
        edge_emb = preds[data.edge_label_index[0]] * preds[data.edge_label_index[1]]
        y_pred = torch.sigmoid(edge_emb.sum(dim=1)).cpu().numpy()

        y_true = data.edge_label.cpu().numpy().flatten()  # ya alineado
        assert len(y_true) == len(y_pred), f"No coinciden: {len(y_true)} vs {len(y_pred)}"

        return {
            "Accuracy": accuracy_score(y_true, y_pred > 0.5),
            "Precision": precision_score(y_true, y_pred > 0.5, zero_division=1),
            "Recall": recall_score(y_true, y_pred > 0.5),
            "F1 Score": f1_score(y_true, y_pred > 0.5),
            "AUC-ROC": roc_auc_score(y_true, y_pred),
            "AUC-PR": average_precision_score(y_true, y_pred),
            "MSE": mean_squared_error(y_true, y_pred)
        }


# Entrenamiento y evaluación

def train_and_evaluate_model(model, train_data, val_data, epochs, criterion, lr, weight_decay):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()

    # Entrenamiento
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        x = model(train_data.x, train_data.edge_index)  # Pasa las features y las aristas
        out = model.decode(x, train_data.edge_label_index).squeeze()  # Decodifica los enlaces
        loss = criterion(out, train_data.edge_label.float())  # Calcula la pérdida
        loss.backward()
        optimizer.step()

    # Evaluación
    model.eval()
    with torch.no_grad():
        x_val = model(val_data.x, val_data.edge_index)
        val_out = model.decode(x_val, val_data.edge_label_index).squeeze()
        val_loss = criterion(val_out, val_data.edge_label.float()).item()  # Calcula la pérdida en validación

    return val_loss


   
# Objetivo para optimización con Optuna
def objective_gcn(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)  
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GCNLinkPredictor(
        in_channels=2,  # Número de características de entrada
        hidden_channels=hidden_channels,
        dropout=dropout_rate,
    )

    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
   # val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)  
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss

def objective_gat(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    heads = trial.suggest_int("heads", 1, 8)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GATLinkPredictor(
        in_channels=2,  # Número de características de entrada
        hidden_channels=hidden_channels,
        heads=heads,
        dropout=dropout_rate
    )
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
    
def objective_gin(trial):
    # Hiperparámetros a optimizar
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GIN
    model = GINLinkPredictor(
        in_channels=2,  # Número de características de entrada
        hidden_channels=hidden_channels,  # Número de canales ocultos
        dropout=dropout_rate  # Tasa de dropout
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss


def objective_sage(trial):
    # Hiperparámetros para optimización
    hidden_channels = trial.suggest_int("hidden_channels", 16, 64)
    num_layers = trial.suggest_int("num_layers", 2, 4)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    aggr_method = trial.suggest_categorical("aggr_method", ["mean", "max", "lstm"])
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GraphSAGE
    
    model = SAGELinkPredictor(
        in_channels=2,  # Ajusta según tu modelo
        hidden_channels=hidden_channels,
        #num_layers=num_layers,
        #aggr=aggr_method,
        dropout=dropout_rate
    )

    # Optimización
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

      # Llama a la función con todos los argumentos
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
           

# Optuna para GCN
#study_gcn = optuna.create_study(direction="minimize")
study_gcn = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gcn.optimize(objective_gcn, n_trials=50)

print("Best GCN hyperparameters:", study_gcn.best_params)
print("Best GCN validation loss:", study_gcn.best_value)

# Optuna para GAT
#study_gat = optuna.create_study(direction="minimize")
study_gat = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gat.optimize(objective_gat, n_trials=50)

print("Best GAT hyperparameters:", study_gat.best_params)
print("Best GAT validation loss:", study_gat.best_value)

# Estudio y optimización para GIN
#study_gin = optuna.create_study(direction="minimize")
study_gin = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gin.optimize(objective_gin, n_trials=10)

# Estudio y optimización para SAGE
#study_sage = optuna.create_study(direction="minimize")
study_sage = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_sage.optimize(objective_sage, n_trials=10)

# Imprimir los mejores parámetros para cada modelo
print("Mejores parámetros para GCN:", study_gcn.best_params)
print("Mejores parámetros para GIN:", study_gin.best_params)
print("Mejores parámetros para SAGE:", study_sage.best_params)
print("Mejores parámetros para GAT:", study_gat.best_params)


# Entrenamiento y guardado de los modelos

# Entrenar y guardar el modelo GCN
best_params = study_gcn.best_params
gcn_model = GCNLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gcn = torch.optim.Adam(gcn_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

criterion = torch.nn.BCEWithLogitsLoss()

gcn_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gcn.zero_grad()
    x = gcn_model(train_data.x, train_data.edge_index)
    out = gcn_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gcn.step()

torch.save(gcn_model.state_dict(), "P24-mejor_modelo_GCN.pth")
print("Modelo GCN guardado.")

# Entrenamiento y guardado del modelo GAT
best_params = study_gat.best_params
if 'heads' in best_params:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=best_params["heads"], dropout=best_params["dropout_rate"])
else:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=1, dropout=best_params["dropout_rate"])

optimizer_gat = torch.optim.Adam(gat_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gat_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gat.zero_grad()
    x = gat_model(train_data.x, train_data.edge_index)
    out = gat_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gat.step()

torch.save(gat_model.state_dict(), "P24-mejor_modelo_GAT.pth")
print("Modelo GAT guardado.")

# Entrenamiento y guardado del modelo GIN
best_params = study_gin.best_params
gin_model = GINLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gin = torch.optim.Adam(gin_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gin_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gin.zero_grad()
    x = gin_model(train_data.x, train_data.edge_index)
    out = gin_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gin.step()

torch.save(gin_model.state_dict(), "P24-mejor_modelo_GIN.pth")
print("Modelo GIN guardado.")

# Entrenamiento y guardado del modelo SAGE
#sage_model = SAGELinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout"])
best_params = study_sage.best_params
sage_model = SAGELinkPredictor(
    in_channels=train_data.x.shape[1],  
    hidden_channels=best_params["hidden_channels"],  
    #num_layers=best_params["num_layers"],  
    #aggr=best_params["aggr_method"],  
    dropout=best_params["dropout_rate"]  
)

optimizer_sage = torch.optim.Adam(sage_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])
criterion = torch.nn.BCEWithLogitsLoss()
sage_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_sage.zero_grad()
    x = sage_model(train_data.x, train_data.edge_index)
    out = sage_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_sage.step()

torch.save(sage_model.state_dict(), "P24-mejor_modelo_SAGE.pth")
print("Modelo SAGE guardado.")


# Cargar todos los modelos para evaluación
gcn_model.load_state_dict(torch.load("P24-mejor_modelo_GCN.pth"))
gcn_model.eval()

gat_model.load_state_dict(torch.load("P24-mejor_modelo_GAT.pth"))
gat_model.eval()

gin_model.load_state_dict(torch.load("P24-mejor_modelo_GIN.pth"))
gin_model.eval()

sage_model.load_state_dict(torch.load("P24-mejor_modelo_SAGE.pth"))
sage_model.eval()

# Evaluar todos los modelos
metrics_gcn = evaluate_model(gcn_model, test_data)
metrics_gat = evaluate_model(gat_model, test_data)
metrics_gin = evaluate_model(gin_model, test_data)
metrics_sage = evaluate_model(sage_model, test_data)

# Crear DataFrame para comparar las métricas
df_metrics = pd.DataFrame([metrics_gcn, metrics_gat, metrics_gin, metrics_sage], index=["GCN", "GAT", "GIN", "SAGE"])

# Mostrar las métricas de los cuatro modelos
print("\n📊 Métricas comparativas de GCN, GAT, GIN y SAGE:")
print(df_metrics)
df_metrics.to_csv("P24-metricas_comparativas.csv", index=True)

#predicciones con test_data
#reverse_mapping = {v: k for k, v in test_data.node_mapping.items()}
reverse_mapping = get_reverse_mapping(test_data)
test_pairs = list(zip(test_data.edge_label_index[0].cpu().numpy(), test_data.edge_label_index[1].cpu().numpy()))

# Mostrar las predicciones de los cuatro modelos
if gcn_model is not None:
    with torch.no_grad():
        x_gcn = gcn_model(test_data.x, test_data.edge_index)
        gcn_predictions = gcn_model.decode(x_gcn, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gcn_predictions = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gcn_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gcn_predictions)]
        gcn_edges = sorted(gcn_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GCN:")
        for author1, author2, score in gcn_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gcn_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P24-predicciones_gcn.csv", index=False)

if gat_model is not None:
    with torch.no_grad():
        x_gat = gat_model(test_data.x, test_data.edge_index)
        gat_predictions = gat_model.decode(x_gat, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gat_predictions = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gat_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gat_predictions)]
        gat_edges = sorted(gat_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GAT:")
        for author1, author2, score in gat_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gat_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P24-predicciones_gat.csv", index=False)


if gin_model is not None:
    with torch.no_grad():
        x_gin = gin_model(test_data.x, test_data.edge_index)
        gin_predictions = gin_model.decode(x_gin, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gin_predictions = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gin_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gin_predictions)]
        gin_edges = sorted(gin_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GIN:")
        for author1, author2, score in gin_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gin_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P24-predicciones_gin.csv", index=False)

#print("Primeras características de nodos:", test_data.x[:5])
#print("Primeras conexiones (edge_index):", test_data.edge_index[:, :5])

if sage_model is not None:
    with torch.no_grad():
        x_sage = sage_model(test_data.x, test_data.edge_index)
        sage_predictions = sage_model.decode(x_sage, test_data.edge_label_index).sigmoid().cpu().numpy()
        #sage_predictions = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        sage_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, sage_predictions)]
        sage_edges = sorted(sage_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones SAGE:")
        for author1, author2, score in sage_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(sage_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P24-predicciones_sage.csv", index=False)

#print("¿GCN y GAT producen las mismas predicciones?", np.allclose(gcn_predictions, gat_predictions))
#print("¿GCN y SAGE producen las mismas predicciones?", np.allclose(gin_predictions, sage_predictions))


print("Configuración GCN:", gcn_model)
print("Configuración GAT:", gat_model)
print("Configuración GIN:", gin_model)
print("Configuración SAGE:", sage_model)


## ensamble de aqui hacia arriba ya no cambiar #
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix

#import xgboost as xgb
# Obtener las predicciones de los modelos

true_labels = test_data.edge_label.cpu().numpy()
print("Valores únicos en true_labels:", np.unique(true_labels))

# Verifica la longitud de las etiquetas y las predicciones
#print("Número de etiquetas", len(true_labels))  # Debe ser el mismo tamaño que las predicciones
# Debe ser el mismo tamaño que las etiquetas

predicciones_gcn = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
print(true_labels.shape, predicciones_gcn.shape)
#print("Número de predicciones de gcn",len(predicciones_gcn))  

predicciones_gat = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
print(true_labels.shape, predicciones_gat.shape)
#print("Número de predicciones de gat",len(predicciones_gat))  

predicciones_gin = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
print(true_labels.shape, predicciones_gin.shape)
#print("Número de predicciones de gin",len(predicciones_gin))  

predicciones_sage = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
print(true_labels.shape, predicciones_sage.shape)
#print("Número de predicciones de sage",len(predicciones_sage))  

# Ya entrenado y obtenido las predicciones de los modelos base (GCN, GAT, GIN, SAGE)
X_stack = np.column_stack((predicciones_gcn, predicciones_gat, predicciones_gin, predicciones_sage))


print("Predicciones GCN:", predicciones_gcn.min(), predicciones_gcn.max())
print("Etiquetas:", np.unique(true_labels))
####  optimizar con OPtuna el modelo ensamble meta_model para regresion logistica


# ----- Función genérica de entrenamiento y evaluación -----
def entrenar_y_evaluar(modelo, nombre_modelo):
    modelo.fit(X_stack, true_labels)
    predicciones = modelo.predict(X_stack)
    
    accuracy = accuracy_score(true_labels, predicciones)
    precision = precision_score(true_labels, predicciones)
    recall = recall_score(true_labels, predicciones)
    f1 = f1_score(true_labels, predicciones)
    roc_auc = roc_auc_score(true_labels, predicciones)
    conf_matrix = confusion_matrix(true_labels, predicciones)
    
    print(f"Métricas del modelo meta ({nombre_modelo}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Predicciones con nombres
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = predicciones[i]
        predicciones_autores.append((author1_name, author2_name, score))
    
    return modelo, predicciones_autores
    
# ----- Modelos Meta -----
# 1. Logistic Regression optimizado con Optuna
def objective_logistic(trial):
    meta_model = LogisticRegression(
        #C=trial.suggest_loguniform('C', 1e-5, 1e5),
        C = trial.suggest_float('C', 1e-5, 1e5, log=True),
        max_iter=trial.suggest_int('max_iter', 100, 1000),
        random_state=SEED
    )
    meta_model.fit(X_stack, true_labels)
    predicciones = meta_model.predict(X_stack)
    return roc_auc_score(true_labels, predicciones)

def objective_random_forest(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 20)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_gb(trial):
    model = GradientBoostingClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        max_depth=trial.suggest_int('max_depth', 3, 10)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_svc(trial):
    model = SVC(
        C=trial.suggest_loguniform('C', 1e-5, 1e5),
        probability=True
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_xgb(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        use_label_encoder=False,
        eval_metric='logloss'
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

# Diccionario de objetivos
objectives = {
    "LogisticRegression": objective_logistic,
    "RandomForest": objective_random_forest,
    "GradientBoosting": objective_gb,
    "SVC": objective_svc,
    "XGBoost": objective_xgb
}

# Resultados
results = {}

for name, obj in objectives.items():
    print(f"Optimizando {name}...")
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(obj, n_trials=50)
    best_params = study.best_params

    if name == "LogisticRegression":
        model = LogisticRegression(**best_params)
    elif name == "RandomForest":
        model = RandomForestClassifier(**best_params)
    elif name == "GradientBoosting":
        model = GradientBoostingClassifier(**best_params)
    elif name == "SVC":
        model = SVC(**best_params, probability=True)
    elif name == "XGBoost":
        model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss')

    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)

    # Evaluación
    accuracy = accuracy_score(true_labels, preds)
    precision = precision_score(true_labels, preds)
    recall = recall_score(true_labels, preds)
    f1 = f1_score(true_labels, preds)
    roc_auc = roc_auc_score(true_labels, preds)
    conf_matrix = confusion_matrix(true_labels, preds)

    print(f"\n24 Métricas del modelo meta ({name}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Guardar predicciones con nombres de autores en diccionario y en archivo
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = float(preds[i])
        predicciones_autores.append((author1_name, author2_name, score))
    results[name] = predicciones_autores
    df = pd.DataFrame(predicciones_autores, columns=["Autor1", "Autor2", "Score"])
        # Formatear la columna Score a 4 decimales
    df["Score"] = df["Score"].map(lambda x: f"{x:.4f}")
    file_name = f"P24-predicciones_meta-{name.replace(' ', '_').lower()}.csv"
    df.to_csv(file_name, index=False)


###  FIN Optimizar con OPtuna modelo ensamble




df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6730
1.0    3395
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234
Best GCN hyperparameters: {'hidden_channels': 31, 'dropout_rate': 0.48856357967189673, 'learning_rate': 0.009682102102935335, 'weight_decay': 3.457954324088416e-05, 'epochs': 55}
Best GCN validation loss: 0.6931466460227966
Best GAT hyperparameters: {'hidden_channels': 44, 'heads': 1, 'dropout_rate': 0.3384081205552792, 'learning_rate': 0.0005303479078830455, 'weight_decay': 0.0011730727946336476, 'epochs': 64}
Best GAT validation loss: 0.6931471824645996
Mejores parámetros para GCN: {'hidden_channels': 31, 'dropout_rate': 0.48856357967189673, 'learning_rate': 0.00968210210293

C:\Users\pilarang\envs\EnvGraphAcademicos\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



24 Métricas del modelo meta (LogisticRegression):
Accuracy: 0.6808
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
AUC-ROC: 0.5000
Confusion Matrix:
[[4006    0]
 [1878    0]]

Optimizando RandomForest...


C:\Users\pilarang\envs\EnvGraphAcademicos\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



24 Métricas del modelo meta (RandomForest):
Accuracy: 0.6808
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
AUC-ROC: 0.5000
Confusion Matrix:
[[4006    0]
 [1878    0]]

Optimizando GradientBoosting...


C:\Users\pilarang\envs\EnvGraphAcademicos\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\3067178414.py:799: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),



24 Métricas del modelo meta (GradientBoosting):
Accuracy: 0.6808
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
AUC-ROC: 0.5000
Confusion Matrix:
[[4006    0]
 [1878    0]]

Optimizando SVC...


C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\3067178414.py:799: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\3067178414.py:799: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\3067178414.py:799: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppDa


24 Métricas del modelo meta (SVC):
Accuracy: 0.6808
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
AUC-ROC: 0.5000
Confusion Matrix:
[[4006    0]
 [1878    0]]

Optimizando XGBoost...

24 Métricas del modelo meta (XGBoost):
Accuracy: 0.6808
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
AUC-ROC: 0.5000
Confusion Matrix:
[[4006    0]
 [1878    0]]



C:\Users\pilarang\envs\EnvGraphAcademicos\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
EL SIGUIENTE PROGRAMA 25 COMPARA FP DE LAS METRICAS ESTRUCTURALES DE UNAM CON ARCHIVOS DE ENTRENAMIENTO, VALIDACION Y PRUEBAS SEPARADOS POR NODOS Y FECHAS

# programa 25  SOLO FEATURE PROXIMITY CON TRAINING 
#	sum_of_areas	sum_of_papers	sum_of_keywords	keywords_match


df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6731
1.0    3386
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234

P25-Métricas comparativas de GCN, GAT, GIN y SAGE:
      Accuracy  Precision    Recall  F1 Score   AUC-ROC    AUC-PR       MSE
GCN   0.365568   0.321257  0.887646  0.471770  0.419798  0.271784  0.250010
GAT   0.347043   0.313733  0.880724  0.462657  0.492571  0.342866  0.250002
GIN   0.319171   0.319171  1.000000  0.483896  0.698884  0.636025  0.269714
SAGE  0.431849   0.324886  0.723642  0.448441  0.531962  0.353581  0.250032

Métricas del modelo meta (LogisticRegression):
Accuracy: 0.6852
Precision: 0.5739
Recall: 0.0538
F1 Score: 0.0983
AUC-ROC: 0.5175
Confusion Matrix:
[[3931   75]
 [1777  101]]

Optimizando RandomForest...

Métricas del modelo meta (RandomForest):
Accuracy: 0.8960
Precision: 0.8540
Recall: 0.8131
F1 Score: 0.8331
AUC-ROC: 0.8740
Confusion Matrix:
[[3745  261]
 [ 351 1527]]
Optimizando GradientBoosting...

Métricas del modelo meta (GradientBoosting):
Accuracy: 0.9018
Precision: 0.8476
Recall: 0.8440
F1 Score: 0.8458
AUC-ROC: 0.8864
Confusion Matrix:
[[3721  285]
 [ 293 1585]]
Métricas del modelo meta (SVC):
Accuracy: 0.7046
Precision: 0.6101
Recall: 0.2066
F1 Score: 0.3087
AUC-ROC: 0.5723
Confusion Matrix:
[[3758  248]
 [1490  388]]

Optimizando XGBoost...

Métricas del modelo meta (XGBoost):
Accuracy: 0.7505
Precision: 0.6306
Recall: 0.5272
F1 Score: 0.5742
AUC-ROC: 0.6912
Confusion Matrix:
[[3426  580]
 [ 888  990]]

In [10]:
#PROGRAMA 25 SOLO FEATURE PROXIMITY CON TRAINING 
#	sum_of_areas	sum_of_papers	sum_of_keywords	keywords_match


from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import torch
import torch.nn.init as init
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv
import optuna
import logging
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error,confusion_matrix
from sklearn.metrics import average_precision_score
from tabulate import tabulate
from torch_geometric.nn import global_mean_pool
import torch.nn.functional as F


import random
import numpy as np
import torch

SEED = 41
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)

# Cargar datasets


file_paths = {
    "sample_train_2000_2020_FP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_train_2000_2020_FP.csv",
    "sample_val_2021_2022_FP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_val_2021_2022_FP.csv",
    "sample_test_2023_2024_FP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_test_2023_2024_FP.csv",
}
df_train = pd.read_csv(file_paths["sample_train_2000_2020_FP"])
df_val   = pd.read_csv(file_paths["sample_val_2021_2022_FP"])
df_test  = pd.read_csv(file_paths["sample_test_2023_2024_FP"])
                                              
#df_train = df_train[df_train['connected'] == 1].reset_index(drop=True)
#df_val = df_val[df_val['connected'] == 1].reset_index(drop=True)
#df_test = df_test[df_test['connected'] == 1].reset_index(drop=True)

#	sum_of_areas	sum_of_papers	sum_of_keywords	keywords_match
# Normalizar columnas numéricas
numeric_columns = [
    'sum_of_areas', 'sum_of_papers',  'sum_of_keywords', 'keywords_match'
]
scaler = MinMaxScaler()
df_train[numeric_columns] = scaler.fit_transform(df_train[numeric_columns])
# Transform val/test con el mismo scaler (no fit)
df_test[numeric_columns] = scaler.transform(df_test[numeric_columns])
df_val[numeric_columns] = scaler.transform(df_val[numeric_columns])

df_train[numeric_columns] = df_train[numeric_columns].fillna(0)
df_val[numeric_columns] = df_val[numeric_columns].fillna(0)
df_test[numeric_columns] = df_test[numeric_columns].fillna(0)

def filter_edges_for_existing_nodes(G_train, df_edges, verbose=True):
    """
    Filtra enlaces que tengan nodos no presentes en el grafo de entrenamiento.
    """
    train_nodes = set(G_train.nodes())
    mask = df_edges['source'].isin(train_nodes) & df_edges['target'].isin(train_nodes)
    df_filtered = df_edges[mask].copy()
    ignored_edges = df_edges[~mask].copy()
    
    if verbose:
        print(f"Total enlaces: {len(df_edges)}")
        print(f"Enlaces válidos: {len(df_filtered)}")
        print(f"Enlaces ignorados (nodos faltantes): {len(ignored_edges)}")
    
    return df_filtered, ignored_edges


# =============================
# CONVERTIR A PyG
# =============================


# Construir grafo 
def build_graph(df):
    G = nx.Graph()
    for _, row in df.iterrows():
        G.add_edge(row['source'], row['target'], weight=row['connected'])
    return G

# Asignar características a los nodos
def add_node_features(G, df):
    node_features = {}
    for _, row in df.iterrows():
        for node in [row['source'], row['target']]:
            if node not in node_features:
                node_features[node] = []
            node_features[node] = [
                 row['sum_of_areas'],
                row['sum_of_papers'],            
                row['sum_of_keywords'], 
                row['keywords_match']
            ]
    for node, features in node_features.items():
        if node in G.nodes:
            G.nodes[node]['features'] = features
    return G

G_train = build_graph(df_train[df_train['connected'] == 1])  # solo positivos
G_train = add_node_features(G_train, df_train)

G_val = build_graph(df_val[df_val['connected'] == 1])
G_val = add_node_features(G_val, df_val)

G_test = build_graph(df_test[df_test['connected'] == 1])
G_test = add_node_features(G_test, df_test)

def filter_edges_for_existing_nodes(G_train, df_edges, verbose=True):
    """
    Filtra enlaces que tengan nodos no presentes en el grafo de entrenamiento.
    """
    train_nodes = set(G_train.nodes())
    mask = df_edges['source'].isin(train_nodes) & df_edges['target'].isin(train_nodes)
    df_filtered = df_edges[mask].copy()
    ignored_edges = df_edges[~mask].copy()
    
    if verbose:
        print(f"Total enlaces: {len(df_edges)}")
        print(f"Enlaces válidos: {len(df_filtered)}")
        print(f"Enlaces ignorados (nodos faltantes): {len(ignored_edges)}")
    
    return df_filtered, ignored_edges

    


extra_nodos = set(df_train.source) | set(df_train.target) \
              | set(df_val.source) | set(df_val.target) \
              | set(df_test.source) | set(df_test.target)

faltantes_train = [n for n in extra_nodos if n not in G_train.nodes()]
faltantes_val   = [n for n in extra_nodos if n not in G_val.nodes()]
faltantes_test  = [n for n in extra_nodos if n not in G_test.nodes()]


def get_reverse_mapping(data):
    """Devuelve un diccionario índice → nodo original a partir de data.node_mapping_dict"""
    return {idx: node for node, idx in data.node_mapping_dict.items()}

# Convertir a PyTorch Geometric
# ===========================================
# Ejecutar para tus tres datasets segun lo que desees que haga que agregue nodos faltantes o no
# ===========================================

import torch
from torch_geometric.data import Data

def graph_to_pyg(G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=False):
    """
    Convierte tres grafos NetworkX y sus DataFrames de aristas a objetos PyG Data,
    revisando los tres conjuntos para agregar nodos faltantes si se desea.

    Parámetros:
        G_train, G_val, G_test : grafos NetworkX para cada split
        df_train, df_val, df_test : DataFrames de aristas ['source','target','connected']
        add_missing_nodes : bool, si True agrega nodos que aparecen en cualquier DF y no están en los grafos

    Retorna:
        train_data, val_data, test_data : objetos torch_geometric.data.Data
    """

    # =============================
    # RECOLECTAR TODOS LOS NODOS NECESARIOS
    # =============================
    if add_missing_nodes:
        extra_nodos = set(df_train.source) | set(df_train.target)
        extra_nodos |= set(df_val.source) | set(df_val.target)
        extra_nodos |= set(df_test.source) | set(df_test.target)
        
        for G in [G_train, G_val, G_test]:
            faltantes = [n for n in extra_nodos if n not in G.nodes()]
            if faltantes:
                feature_dim = len(next(iter(G.nodes(data='features')))[1])
                for n in faltantes:
                    G.add_node(n, features=[0.0]*feature_dim)
            # No imprimimos nodos faltantes para no saturar salida
    if not add_missing_nodes:
        df_train = filter_edges(df_train, G_train)
        df_val   = filter_edges(df_val, G_val)
        df_test  = filter_edges(df_test, G_test)

    # =============================
    # FUNCIÓN INTERNA PARA CONVERTIR CADA SPLIT
    # =============================
    def _convert(G, df):
        if df is None:
            return None
        mapping = {node: i for i, node in enumerate(G.nodes())}
        edge_index = torch.tensor(
            [[mapping[u], mapping[v]] for u, v in zip(df.source, df.target)],
            dtype=torch.long
        ).t().contiguous()
        node_features = torch.tensor(
            [G.nodes[node]['features'] for node in G.nodes()],
            dtype=torch.float
        )
        edge_label = torch.tensor(df.connected.values, dtype=torch.float)
        data = Data(
            x=node_features,
            edge_index=edge_index,
            edge_label=edge_label,
            edge_label_index=edge_index.clone(),
            num_nodes=len(G.nodes())
        )
        # Guardar mapping como atributo dict explícito
        data.node_mapping_dict = mapping
        return data

    train_data = _convert(G_train, df_train)
    val_data   = _convert(G_val, df_val)
    test_data  = _convert(G_test, df_test)

    return train_data, val_data, test_data

train_data, val_data, test_data = graph_to_pyg(
    G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=True
)

def sort_edge_index(data):
    sorted_indices = torch.argsort(data.edge_index[1])
    data.edge_index = data.edge_index[:, sorted_indices]
    return data

train_data = sort_edge_index(train_data)
val_data = sort_edge_index(val_data)
test_data = sort_edge_index(test_data)


import numpy as np
print("df_train connected counts:\n", df_train['connected'].value_counts())
print("df_val connected counts:\n", df_val['connected'].value_counts())
print("df_test connected counts:\n", df_test['connected'].value_counts())

true_labels = test_data.edge_label.cpu().numpy()
print("true_labels unique/counts:", np.unique(true_labels, return_counts=True))

# =============================
# CALCULAR DENSIDADES
# =============================
def calc_density(G):
    if G.number_of_nodes() == 0:
        return 0
    return nx.density(G)

def density_for_dataset(data, description):
   # nodes = [reverse_mapping[i] for i in range(data.num_nodes)]
    reverse_mapping = get_reverse_mapping(data) 
    G_tmp = nx.Graph()
    for u, v in data.edge_index.t().numpy():
        G_tmp.add_edge(reverse_mapping[u], reverse_mapping[v])
    density = calc_density(G_tmp)
    print(f"Densidad {description}: {density:.4f}")
    return density

density_train = density_for_dataset(train_data, "Train")
density_val = density_for_dataset(val_data, "Validation")
density_test = density_for_dataset(test_data, "Test")


# eliminar mensajes de los trials de optuna
optuna.logging.set_verbosity(optuna.logging.ERROR)



# Definir modelos
class GCNLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GATLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels=1, heads=1, dropout=0.0):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, dropout=dropout)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GINLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GINConv(torch.nn.Sequential(torch.nn.Linear(in_channels, hidden_channels), torch.nn.ReLU()))
        self.conv2 = GINConv(torch.nn.Sequential(torch.nn.Linear(hidden_channels, 1)))
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)



class SAGELinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

#evaluacion de modelos
def evaluate_model(model, data):
    model.eval()
    with torch.no_grad():
        preds = model(data.x, data.edge_index)
        edge_emb = preds[data.edge_label_index[0]] * preds[data.edge_label_index[1]]
        y_pred = torch.sigmoid(edge_emb.sum(dim=1)).cpu().numpy()

    y_true = data.edge_label.cpu().numpy().flatten()  # ya alineado
    assert len(y_true) == len(y_pred), f"No coinciden: {len(y_true)} vs {len(y_pred)}"

    return {
        "Accuracy": accuracy_score(y_true, y_pred > 0.5),
        "Precision": precision_score(y_true, y_pred > 0.5, zero_division=1),
        "Recall": recall_score(y_true, y_pred > 0.5),
        "F1 Score": f1_score(y_true, y_pred > 0.5),
        "AUC-ROC": roc_auc_score(y_true, y_pred),
        "AUC-PR": average_precision_score(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred)
    }

# Entrenamiento y evaluación

def train_and_evaluate_model(model, train_data, val_data, epochs, criterion, lr, weight_decay):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()

    # Entrenamiento
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        x = model(train_data.x, train_data.edge_index)  # Pasa las features y las aristas
        out = model.decode(x, train_data.edge_label_index).squeeze()  # Decodifica los enlaces
        loss = criterion(out, train_data.edge_label.float())  # Calcula la pérdida
        loss.backward()
        optimizer.step()

    # Evaluación
    model.eval()
    with torch.no_grad():
        x_val = model(val_data.x, val_data.edge_index)
        val_out = model.decode(x_val, val_data.edge_label_index).squeeze()
        val_loss = criterion(val_out, val_data.edge_label.float()).item()  # Calcula la pérdida en validación

    return val_loss


   
# Objetivo para optimización con Optuna
def objective_gcn(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)  
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GCNLinkPredictor(
        in_channels=4,  # Número de características de entrada
        hidden_channels=hidden_channels,
        dropout=dropout_rate,
    )

    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
   # val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)  
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss

def objective_gat(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    heads = trial.suggest_int("heads", 1, 8)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GATLinkPredictor(
        in_channels=4,  # Número de características de entrada
        hidden_channels=hidden_channels,
        heads=heads,
        dropout=dropout_rate
    )
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
    
def objective_gin(trial):
    # Hiperparámetros a optimizar
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GIN
    model = GINLinkPredictor(
        in_channels=4,  # Número de características de entrada
        hidden_channels=hidden_channels,  # Número de canales ocultos
        dropout=dropout_rate  # Tasa de dropout
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss


def objective_sage(trial):
    # Hiperparámetros para optimización
    hidden_channels = trial.suggest_int("hidden_channels", 16, 64)
    num_layers = trial.suggest_int("num_layers", 2, 4)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    aggr_method = trial.suggest_categorical("aggr_method", ["mean", "max", "lstm"])
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GraphSAGE
    
    model = SAGELinkPredictor(
        in_channels=4,  # Ajusta según tu modelo
        hidden_channels=hidden_channels,
        #num_layers=num_layers,
        #aggr=aggr_method,
        dropout=dropout_rate
    )

    # Optimización
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

      # Llama a la función con todos los argumentos
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
           

# Optuna para GCN
#study_gcn = optuna.create_study(direction="minimize")
study_gcn = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gcn.optimize(objective_gcn, n_trials=50)

print("Best GCN hyperparameters:", study_gcn.best_params)
print("Best GCN validation loss:", study_gcn.best_value)

# Optuna para GAT
#study_gat = optuna.create_study(direction="minimize
study_gat = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gat.optimize(objective_gat, n_trials=50)

print("Best GAT hyperparameters:", study_gat.best_params)
print("Best GAT validation loss:", study_gat.best_value)

# Estudio y optimización para GIN
#study_gin = optuna.create_study(direction="minimize")
study_gin = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gin.optimize(objective_gin, n_trials=10)

# Estudio y optimización para SAGE
#study_sage = optuna.create_study(direction="minimize")
study_sage = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_sage.optimize(objective_sage, n_trials=10)

# Imprimir los mejores parámetros para cada modelo
print("Mejores parámetros para GCN:", study_gcn.best_params)
print("Mejores parámetros para GIN:", study_gin.best_params)
print("Mejores parámetros para SAGE:", study_sage.best_params)
print("Mejores parámetros para GAT:", study_gat.best_params)


# Entrenamiento y guardado de los modelos

# Entrenar y guardar el modelo GCN
best_params = study_gcn.best_params
gcn_model = GCNLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gcn = torch.optim.Adam(gcn_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

criterion = torch.nn.BCEWithLogitsLoss()

gcn_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gcn.zero_grad()
    x = gcn_model(train_data.x, train_data.edge_index)
    out = gcn_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gcn.step()

torch.save(gcn_model.state_dict(), "P25-mejor_modelo_GCN.pth")
print("P25-Modelo GCN guardado.")

# Entrenamiento y guardado del modelo GAT
best_params = study_gat.best_params
if 'heads' in best_params:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=best_params["heads"], dropout=best_params["dropout_rate"])
else:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=1, dropout=best_params["dropout_rate"])

optimizer_gat = torch.optim.Adam(gat_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gat_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gat.zero_grad()
    x = gat_model(train_data.x, train_data.edge_index)
    out = gat_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gat.step()

torch.save(gat_model.state_dict(), "P25-mejor_modelo_GAT.pth")
print("P25-Modelo GAT guardado.")

# Entrenamiento y guardado del modelo GIN
best_params = study_gin.best_params
gin_model = GINLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gin = torch.optim.Adam(gin_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gin_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gin.zero_grad()
    x = gin_model(train_data.x, train_data.edge_index)
    out = gin_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gin.step()

torch.save(gin_model.state_dict(), "P25-mejor_modelo_GIN.pth")
print("P25-Modelo GIN guardado.")

# Entrenamiento y guardado del modelo SAGE
#sage_model = SAGELinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout"])
best_params = study_sage.best_params
sage_model = SAGELinkPredictor(
    in_channels=train_data.x.shape[1],  
    hidden_channels=best_params["hidden_channels"],  
    #num_layers=best_params["num_layers"],  
    #aggr=best_params["aggr_method"],  
    dropout=best_params["dropout_rate"]  
)

optimizer_sage = torch.optim.Adam(sage_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])
criterion = torch.nn.BCEWithLogitsLoss()
sage_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_sage.zero_grad()
    x = sage_model(train_data.x, train_data.edge_index)
    out = sage_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_sage.step()

torch.save(sage_model.state_dict(), "P25-mejor_modelo_SAGE.pth")
print("P25-Modelo SAGE guardado.")


# Cargar todos los modelos para evaluación
gcn_model.load_state_dict(torch.load("P25-mejor_modelo_GCN.pth"))
gcn_model.eval()

gat_model.load_state_dict(torch.load("P25-mejor_modelo_GAT.pth"))
gat_model.eval()

gin_model.load_state_dict(torch.load("P25-mejor_modelo_GIN.pth"))
gin_model.eval()

sage_model.load_state_dict(torch.load("P25-mejor_modelo_SAGE.pth"))
sage_model.eval()

# Evaluar todos los modelos
metrics_gcn = evaluate_model(gcn_model, test_data)
metrics_gat = evaluate_model(gat_model, test_data)
metrics_gin = evaluate_model(gin_model, test_data)
metrics_sage = evaluate_model(sage_model, test_data)

# Crear DataFrame para comparar las métricas
df_metrics = pd.DataFrame([metrics_gcn, metrics_gat, metrics_gin, metrics_sage], index=["GCN", "GAT", "GIN", "SAGE"])

# Mostrar las métricas de los cuatro modelos
print("\n📊 P25-Métricas comparativas de GCN, GAT, GIN y SAGE:")
print(df_metrics)
df_metrics.to_csv("P25-metricas_comparativas.csv", index=True)

#predicciones con test_data
#reverse_mapping = {v: k for k, v in test_data.node_mapping.items()}
reverse_mapping = get_reverse_mapping(test_data)
#dejar test_pairs…
test_pairs = list(zip(test_data.edge_label_index[0].cpu().numpy(), test_data.edge_label_index[1].cpu().numpy()))

# Mostrar las predicciones de los cuatro modelos
if gcn_model is not None:
    with torch.no_grad():
        x_gcn = gcn_model(test_data.x, test_data.edge_index)
        gcn_predictions = gcn_model.decode(x_gcn, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gcn_predictions = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gcn_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gcn_predictions)]
        gcn_edges = sorted(gcn_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GCN:")
        for author1, author2, score in gcn_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gcn_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P25-predicciones_gcn.csv", index=False)

if gat_model is not None:
    with torch.no_grad():
        x_gat = gat_model(test_data.x, test_data.edge_index)
        gat_predictions = gat_model.decode(x_gat, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gat_predictions = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gat_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gat_predictions)]
        gat_edges = sorted(gat_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GAT:")
        for author1, author2, score in gat_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gat_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P25-predicciones_gat.csv", index=False)


if gin_model is not None:
    with torch.no_grad():
        x_gin = gin_model(test_data.x, test_data.edge_index)
        gin_predictions = gin_model.decode(x_gin, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gin_predictions = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gin_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gin_predictions)]
        gin_edges = sorted(gin_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GIN:")
        for author1, author2, score in gin_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gin_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P25-predicciones_gin.csv", index=False)

#print("Primeras características de nodos:", test_data.x[:5])
#print("Primeras conexiones (edge_index):", test_data.edge_index[:, :5])

if sage_model is not None:
    with torch.no_grad():
        x_sage = sage_model(test_data.x, test_data.edge_index)
        sage_predictions = sage_model.decode(x_sage, test_data.edge_label_index).sigmoid().cpu().numpy()
        #sage_predictions = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        sage_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, sage_predictions)]
        sage_edges = sorted(sage_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones SAGE:")
        for author1, author2, score in sage_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(sage_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P25-predicciones_sage.csv", index=False)

#print("¿GCN y GAT producen las mismas predicciones?", np.allclose(gcn_predictions, gat_predictions))
#print("¿GCN y SAGE producen las mismas predicciones?", np.allclose(gin_predictions, sage_predictions))


print("Configuración GCN:", gcn_model)
print("Configuración GAT:", gat_model)
print("Configuración GIN:", gin_model)
print("Configuración SAGE:", sage_model)


## ensamble de aqui hacia arriba ya no cambiar #
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix

#import xgboost as xgb
# Obtener las predicciones de los modelos

true_labels = test_data.edge_label.cpu().numpy()
print("Valores únicos en true_labels:", np.unique(true_labels))
# Verifica la longitud de las etiquetas y las predicciones
#print("Número de etiquetas", len(true_labels))  # Debe ser el mismo tamaño que las predicciones
# Debe ser el mismo tamaño que las etiquetas

predicciones_gcn = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
print(true_labels.shape, predicciones_gcn.shape)
#print("Número de predicciones de gcn",len(predicciones_gcn))  

predicciones_gat = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
print(true_labels.shape, predicciones_gat.shape)
#print("Número de predicciones de gat",len(predicciones_gat))  

predicciones_gin = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
print(true_labels.shape, predicciones_gin.shape)
#print("Número de predicciones de gin",len(predicciones_gin))  

predicciones_sage = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
print(true_labels.shape, predicciones_sage.shape)
#print("Número de predicciones de sage",len(predicciones_sage))  

# Ya entrenado y obtenido las predicciones de los modelos base (GCN, GAT, GIN, SAGE)
X_stack = np.column_stack((predicciones_gcn, predicciones_gat, predicciones_gin, predicciones_sage))

####  optimizar con OPtuna el modelo ensamble meta_model para regresion logistica


# ----- Función genérica de entrenamiento y evaluación -----
def entrenar_y_evaluar(modelo, nombre_modelo):
    modelo.fit(X_stack, true_labels)
    predicciones = modelo.predict(X_stack)
    
    accuracy = accuracy_score(true_labels, predicciones)
    precision = precision_score(true_labels, predicciones)
    recall = recall_score(true_labels, predicciones)
    f1 = f1_score(true_labels, predicciones)
    roc_auc = roc_auc_score(true_labels, predicciones)
    conf_matrix = confusion_matrix(true_labels, predicciones)
    
    print(f"Métricas del modelo meta ({nombre_modelo}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Predicciones con nombres
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = predicciones[i]
        predicciones_autores.append((author1_name, author2_name, score))
    
    return modelo, predicciones_autores
    
# ----- Modelos Meta -----
# 1. Logistic Regression optimizado con Optuna
def objective_logistic(trial):
    meta_model = LogisticRegression(
        #C=trial.suggest_loguniform('C', 1e-5, 1e5),
        C = trial.suggest_float('C', 1e-5, 1e5, log=True),
        max_iter=trial.suggest_int('max_iter', 100, 1000),
        random_state=SEED
    )
    meta_model.fit(X_stack, true_labels)
    predicciones = meta_model.predict(X_stack)
    return roc_auc_score(true_labels, predicciones)

def objective_random_forest(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 20)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_gb(trial):
    model = GradientBoostingClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        max_depth=trial.suggest_int('max_depth', 3, 10)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_svc(trial):
    model = SVC(
        C=trial.suggest_loguniform('C', 1e-5, 1e5),
        probability=True
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_xgb(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        use_label_encoder=False,
        eval_metric='logloss'
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

# Diccionario de objetivos
objectives = {
    "LogisticRegression": objective_logistic,
    "RandomForest": objective_random_forest,
    "GradientBoosting": objective_gb,
    "SVC": objective_svc,
    "XGBoost": objective_xgb
}

# Resultados
results = {}

for name, obj in objectives.items():
    print(f"Optimizando {name}...")
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(obj, n_trials=50)
    best_params = study.best_params

    if name == "LogisticRegression":
        model = LogisticRegression(**best_params)
    elif name == "RandomForest":
        model = RandomForestClassifier(**best_params)
    elif name == "GradientBoosting":
        model = GradientBoostingClassifier(**best_params)
    elif name == "SVC":
        model = SVC(**best_params, probability=True)
    elif name == "XGBoost":
        model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss')

    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)

    # Evaluación
    accuracy = accuracy_score(true_labels, preds)
    precision = precision_score(true_labels, preds)
    recall = recall_score(true_labels, preds)
    f1 = f1_score(true_labels, preds)
    roc_auc = roc_auc_score(true_labels, preds)
    conf_matrix = confusion_matrix(true_labels, preds)

    print(f"\nMétricas del modelo meta ({name}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Guardar predicciones con nombres de autores en diccionario y en archivo
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = float(preds[i])
        predicciones_autores.append((author1_name, author2_name, score))
    results[name] = predicciones_autores
    df = pd.DataFrame(predicciones_autores, columns=["Autor1", "Autor2", "Score"])
        # Formatear la columna Score a 4 decimales
    df["Score"] = df["Score"].map(lambda x: f"{x:.4f}")
    file_name = f"P25-predicciones_meta-{name.replace(' ', '_').lower()}.csv"
    df.to_csv(file_name, index=False)


###  FIN Optimizar con OPtuna modelo ensamble


df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6731
1.0    3386
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234
Best GCN hyperparameters: {'hidden_channels': 55, 'dropout_rate': 0.3607503208282641, 'learning_rate': 0.0036037919058457416, 'weight_decay': 0.0046122362032693645, 'epochs': 32}
Best GCN validation loss: 0.6931470632553101
Best GAT hyperparameters: {'hidden_channels': 72, 'heads': 5, 'dropout_rate': 0.27302163385939343, 'learning_rate': 0.00510933179051516, 'weight_decay': 0.004494633221207727, 'epochs': 43}
Best GAT validation loss: 0.6931472420692444
Mejores parámetros para GCN: {'hidden_channels': 55, 'dropout_rate': 0.3607503208282641, 'learning_rate': 0.00360379190584574

C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\1810112747.py:807: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\1810112747.py:807: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\1810112747.py:807: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppDa

KeyboardInterrupt: 

# PROGRAMA 26 LSP+FP GRAFOS DE ENTRENAMIENTO POR NODOS Y FECHAS UNAM
df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6731
1.0    3386
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234
P26-Métricas comparativas de GCN, GAT, GIN y SAGE:
      Accuracy  Precision    Recall  F1 Score   AUC-ROC    AUC-PR       MSE
GCN   0.425051   0.354646  0.977636  0.520482  0.756251  0.675480  0.249943
GAT   0.319171   0.319171  1.000000  0.483896  0.338922  0.261635  0.250079
GIN   0.320700   0.319660  1.000000  0.484458  0.732685  0.600432  0.556132
SAGE  0.559823   0.418311  0.970714  0.584670  0.740176  0.489418  0.249969



programa 27 con cambios 

27-Métricas comparativas de GCN, GAT, GIN y SAGE:
      Accuracy  Precision    Recall  F1 Score   AUC-ROC    AUC-PR       MSE
GCN   0.321380   0.319632  0.997870  0.484175  0.640049  0.433128  0.250004
GAT   0.319171   0.319171  1.000000  0.483896  0.305525  0.232578  0.250142
GIN   0.325799   0.320563  0.993610  0.484738  0.752496  0.683732  0.308963
SAGE  0.320700   0.319229  0.996273  0.483525  0.243334  0.211093  0.250102


Métricas del modelo meta (LogisticRegression):
Accuracy: 0.6990
Precision: 0.5425
Recall: 0.3637
F1 Score: 0.4354
AUC-ROC: 0.6100
Confusion Matrix:
[[3430  576]
 [1195  683]]

Optimizando RandomForest...

Métricas del modelo meta (RandomForest):
Accuracy: 0.9905
Precision: 0.9830
Recall: 0.9872
F1 Score: 0.9851
AUC-ROC: 0.9896
Confusion Matrix:
[[3974   32]
 [  24 1854]]

Optimizando GradientBoosting...

Métricas del modelo meta (GradientBoosting):
Accuracy: 0.9956
Precision: 0.9889
Recall: 0.9973
F1 Score: 0.9931
AUC-ROC: 0.9960
Confusion Matrix:
[[3985   21]
 [   5 1873]]

 Métricas del modelo meta (SVC):
Accuracy: 0.7039
Precision: 0.5428
Recall: 0.4590
F1 Score: 0.4974
AUC-ROC: 0.6389
Confusion Matrix:
[[3280  726]
 [1016  862]]

Optimizando XGBoost...

Métricas del modelo meta (XGBoost):
Accuracy: 0.7736
Precision: 0.6641
Recall: 0.5884
F1 Score: 0.6239
AUC-ROC: 0.7244
Confusion Matrix:
[[3447  559]
 [ 773 1105]]

In [11]:
#PROGRAMA 26  LSP+FP CON DATOS ORDENADOS POR NODOS Y ARCHIVOS DE ENTRENAMIENTO, VAL Y TEST GENERADOS POR FECHAS UNAM
#lenght_short_path	jaccard_coefficient	adamic_adar_index	resource_allocation	preferential_attachment	sum_of_neighbors	log_secundary_neighbors	
#clustering_index_sum	sum_of_areas	sum_of_papers	sum_of_keywords	keywords_match (12)


from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import torch
import torch.nn.init as init
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv
import optuna
import logging
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error,confusion_matrix
from sklearn.metrics import average_precision_score
from tabulate import tabulate
from torch_geometric.nn import global_mean_pool
import torch.nn.functional as F


import random
import numpy as np
import torch

SEED = 41
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)

# Cargar datasets
file_paths = {
    "sample_train_2000_2020_LSP_FP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_train_2000_2020_LSP_FP.csv",
    "sample_val_2021_2022_LSP_FP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_val_2021_2022_LSP_FP.csv",
    "sample_test_2023_2024_LSP_FP": "C:/Users/pilarang/0-ProyAcademicosGrafo-Integrado/2-ServSoc_Implementacion por fechas/sample_test_2023_2024_LSP_FP.csv",
}
df_train = pd.read_csv(file_paths["sample_train_2000_2020_LSP_FP"])
df_val   = pd.read_csv(file_paths["sample_val_2021_2022_LSP_FP"])
df_test  = pd.read_csv(file_paths["sample_test_2023_2024_LSP_FP"])
                                              

# Filtrar aristas conectadas

#df_train = df_train[df_train['connected'] == 1].reset_index(drop=True)
#df_val = df_val[df_val['connected'] == 1].reset_index(drop=True)
#df_test = df_test[df_test['connected'] == 1].reset_index(drop=True)

# Reemplazar NaN en 'lenght_short_path' por mediana
df_train['lenght_short_path'] = df_train['lenght_short_path'].fillna(df_train['lenght_short_path'].median())
df_val['lenght_short_path'] = df_val['lenght_short_path'].fillna(df_val['lenght_short_path'].median())
df_test['lenght_short_path'] = df_test['lenght_short_path'].fillna(df_test['lenght_short_path'].median())


# Normalizar las columnas numéricas, se hace aqui o en el splitRandom, estan las dos opciones, esta da mejores resultados
numeric_columns = [
    'lenght_short_path',	'jaccard_coefficient',	'adamic_adar_index',	
    'resource_allocation',	'preferential_attachment',	'sum_of_neighbors',	
    'log_secundary_neighbors',	'clustering_index_sum',	'sum_of_areas',	
    'sum_of_papers',	'sum_of_keywords',	'keywords_match'
]

scaler = MinMaxScaler()
df_train[numeric_columns] = scaler.fit_transform(df_train[numeric_columns])
# Transform val/test con el mismo scaler (no fit)
df_test[numeric_columns] = scaler.transform(df_test[numeric_columns])
df_val[numeric_columns] = scaler.transform(df_val[numeric_columns])

df_train[numeric_columns] = df_train[numeric_columns].fillna(0)
df_val[numeric_columns] = df_val[numeric_columns].fillna(0)
df_test[numeric_columns] = df_test[numeric_columns].fillna(0)


def filter_edges_for_existing_nodes(G_train, df_edges, verbose=True):
    """
    Filtra enlaces que tengan nodos no presentes en el grafo de entrenamiento.
    """
    train_nodes = set(G_train.nodes())
    mask = df_edges['source'].isin(train_nodes) & df_edges['target'].isin(train_nodes)
    df_filtered = df_edges[mask].copy()
    ignored_edges = df_edges[~mask].copy()
    
    if verbose:
        print(f"Total enlaces: {len(df_edges)}")
        print(f"Enlaces válidos: {len(df_filtered)}")
        print(f"Enlaces ignorados (nodos faltantes): {len(ignored_edges)}")
    
    return df_filtered, ignored_edges
# Construir grafo de entrenamiento
def build_graph(df):
    G = nx.Graph()
    for _, row in df.iterrows():
        G.add_edge(row['source'], row['target'], weight=row['connected'])
    return G



# Asignar características a los nodos

#lenght_short_path	jaccard_coefficient	adamic_adar_index	resource_allocation	preferential_attachment	sum_of_neighbors	
#log_secundary_neighbors	clustering_index_sum	sum_of_areas	sum_of_papers	sum_of_keywords	keywords_match (12)

def add_node_features(G, df):
    node_features = {}
    for _, row in df.iterrows():
        for node in [row['source'], row['target']]:
            if node not in node_features:
                node_features[node] = []
            node_features[node] = [
                row['lenght_short_path'],
                row['jaccard_coefficient'],
                row['adamic_adar_index'],
                row['resource_allocation'],
                row['preferential_attachment'],
                row['sum_of_neighbors'],
                row['log_secundary_neighbors'],
                row['clustering_index_sum'],
                row['sum_of_areas'],
                row['sum_of_papers'],
                row['sum_of_keywords'],
                row['keywords_match']
            ]
    for node, features in node_features.items():
        if node in G.nodes:
            G.nodes[node]['features'] = features
    return G

G_train = build_graph(df_train[df_train['connected'] == 1])  # solo positivos
G_train = add_node_features(G_train, df_train)

G_val = build_graph(df_val[df_val['connected'] == 1])
G_val = add_node_features(G_val, df_val)

G_test = build_graph(df_test[df_test['connected'] == 1])
G_test = add_node_features(G_test, df_test)

extra_nodos = set(df_train.source) | set(df_train.target) \
              | set(df_val.source) | set(df_val.target) \
              | set(df_test.source) | set(df_test.target)

faltantes_train = [n for n in extra_nodos if n not in G_train.nodes()]
faltantes_val   = [n for n in extra_nodos if n not in G_val.nodes()]
faltantes_test  = [n for n in extra_nodos if n not in G_test.nodes()]

def get_reverse_mapping(data):
    """Devuelve un diccionario índice → nodo original a partir de data.node_mapping_dict"""
    return {idx: node for node, idx in data.node_mapping_dict.items()}

# ===========================================
# Ejecutar para tus tres datasets segun lo que desees que haga que agregue nodos faltantes o no
# ===========================================

import torch
from torch_geometric.data import Data

def graph_to_pyg(G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=False):
    """
    Convierte tres grafos NetworkX y sus DataFrames de aristas a objetos PyG Data,
    revisando los tres conjuntos para agregar nodos faltantes si se desea.

    Parámetros:
        G_train, G_val, G_test : grafos NetworkX para cada split
        df_train, df_val, df_test : DataFrames de aristas ['source','target','connected']
        add_missing_nodes : bool, si True agrega nodos que aparecen en cualquier DF y no están en los grafos

    Retorna:
        train_data, val_data, test_data : objetos torch_geometric.data.Data
    """

    # =============================
    # RECOLECTAR TODOS LOS NODOS NECESARIOS
    # =============================
    if add_missing_nodes:
        extra_nodos = set(df_train.source) | set(df_train.target)
        extra_nodos |= set(df_val.source) | set(df_val.target)
        extra_nodos |= set(df_test.source) | set(df_test.target)
        
        for G in [G_train, G_val, G_test]:
            faltantes = [n for n in extra_nodos if n not in G.nodes()]
            if faltantes:
                feature_dim = len(next(iter(G.nodes(data='features')))[1])
                for n in faltantes:
                    G.add_node(n, features=[0.0]*feature_dim)
            # No imprimimos nodos faltantes para no saturar salida
    if not add_missing_nodes:
        df_train = filter_edges(df_train, G_train)
        df_val   = filter_edges(df_val, G_val)
        df_test  = filter_edges(df_test, G_test)

    # =============================
    # FUNCIÓN INTERNA PARA CONVERTIR CADA SPLIT
    # =============================
    def _convert(G, df):
        if df is None:
            return None
        mapping = {node: i for i, node in enumerate(G.nodes())}
        edge_index = torch.tensor(
            [[mapping[u], mapping[v]] for u, v in zip(df.source, df.target)],
            dtype=torch.long
        ).t().contiguous()
        node_features = torch.tensor(
            [G.nodes[node]['features'] for node in G.nodes()],
            dtype=torch.float
        )
        edge_label = torch.tensor(df.connected.values, dtype=torch.float)
        data = Data(
            x=node_features,
            edge_index=edge_index,
            edge_label=edge_label,
            edge_label_index=edge_index.clone(),
            num_nodes=len(G.nodes())
        )
        # Guardar mapping como atributo dict explícito
        data.node_mapping_dict = mapping
        return data

    train_data = _convert(G_train, df_train)
    val_data   = _convert(G_val, df_val)
    test_data  = _convert(G_test, df_test)

    return train_data, val_data, test_data


train_data, val_data, test_data = graph_to_pyg(
    G_train, G_val, G_test, df_train, df_val, df_test, add_missing_nodes=True
)
    


def sort_edge_index(data):
    sorted_indices = torch.argsort(data.edge_index[1])
    data.edge_index = data.edge_index[:, sorted_indices]
    return data

train_data = sort_edge_index(train_data)
val_data = sort_edge_index(val_data)
test_data = sort_edge_index(test_data)

import numpy as np
print("df_train connected counts:\n", df_train['connected'].value_counts())
print("df_val connected counts:\n", df_val['connected'].value_counts())
print("df_test connected counts:\n", df_test['connected'].value_counts())

true_labels = test_data.edge_label.cpu().numpy()
print("true_labels unique/counts:", np.unique(true_labels, return_counts=True))

# =============================
# CALCULAR DENSIDADES
# =============================


def calc_density(G):
    if G.number_of_nodes() == 0:
        return 0
    return nx.density(G)

def density_for_dataset(data, description):
   # nodes = [reverse_mapping[i] for i in range(data.num_nodes)]
    reverse_mapping = get_reverse_mapping(data) 
    G_tmp = nx.Graph()
    for u, v in data.edge_index.t().numpy():
        G_tmp.add_edge(reverse_mapping[u], reverse_mapping[v])
    density = calc_density(G_tmp)
    print(f"Densidad {description}: {density:.4f}")
    return density

density_train = density_for_dataset(train_data, "Train")
density_val = density_for_dataset(val_data, "Validation")
density_test = density_for_dataset(test_data, "Test")

# eliminar mensajes de los trials de optuna
optuna.logging.set_verbosity(optuna.logging.ERROR)



# Definir modelos
class GCNLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GATLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels=1, heads=1, dropout=0.0):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, dropout=dropout)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

class GINLinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = GINConv(torch.nn.Sequential(torch.nn.Linear(in_channels, hidden_channels), torch.nn.ReLU()))
        self.conv2 = GINConv(torch.nn.Sequential(torch.nn.Linear(hidden_channels, 1)))
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)



class SAGELinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.0):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, x, edge_label_index):
        return (x[edge_label_index[0]] * x[edge_label_index[1]]).sum(dim=-1)

#evaluacion de modelos
def evaluate_model(model, data):
    model.eval()
    with torch.no_grad():
        preds = model(data.x, data.edge_index)
        edge_emb = preds[data.edge_label_index[0]] * preds[data.edge_label_index[1]]
        y_pred = torch.sigmoid(edge_emb.sum(dim=1)).cpu().numpy()

    y_true = data.edge_label.cpu().numpy().flatten()  # ya alineado
    assert len(y_true) == len(y_pred), f"No coinciden: {len(y_true)} vs {len(y_pred)}"

    return {
        "Accuracy": accuracy_score(y_true, y_pred > 0.5),
        "Precision": precision_score(y_true, y_pred > 0.5, zero_division=1),
        "Recall": recall_score(y_true, y_pred > 0.5),
        "F1 Score": f1_score(y_true, y_pred > 0.5),
        "AUC-ROC": roc_auc_score(y_true, y_pred),
        "AUC-PR": average_precision_score(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred)
    }
# Entrenamiento y evaluación

def train_and_evaluate_model(model, train_data, val_data, epochs, criterion, lr, weight_decay):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()

    # Entrenamiento
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        x = model(train_data.x, train_data.edge_index)  # Pasa las features y las aristas
        out = model.decode(x, train_data.edge_label_index).squeeze()  # Decodifica los enlaces
        loss = criterion(out, train_data.edge_label.float())  # Calcula la pérdida
        loss.backward()
        optimizer.step()

    # Evaluación
    model.eval()
    with torch.no_grad():
        x_val = model(val_data.x, val_data.edge_index)
        val_out = model.decode(x_val, val_data.edge_label_index).squeeze()
        val_loss = criterion(val_out, val_data.edge_label.float()).item()  # Calcula la pérdida en validación

    return val_loss


   
# Objetivo para optimización con Optuna
def objective_gcn(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)  
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GCNLinkPredictor(
        in_channels=12,  # Número de características de entrada
        hidden_channels=hidden_channels,
        dropout=dropout_rate,
    )

    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
   # val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)  
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss

def objective_gat(trial):
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    heads = trial.suggest_int("heads", 1, 8)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2) 
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    model = GATLinkPredictor(
        in_channels=12,  # Número de características de entrada
        hidden_channels=hidden_channels,
        heads=heads,
        dropout=dropout_rate
    )
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs=epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
    
def objective_gin(trial):
    # Hiperparámetros a optimizar
    hidden_channels = trial.suggest_int("hidden_channels", 16, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GIN
    model = GINLinkPredictor(
        in_channels=12,  # Número de características de entrada
        hidden_channels=hidden_channels,  # Número de canales ocultos
        dropout=dropout_rate  # Tasa de dropout
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss


def objective_sage(trial):
    # Hiperparámetros para optimización
    hidden_channels = trial.suggest_int("hidden_channels", 16, 64)
    num_layers = trial.suggest_int("num_layers", 2, 4)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2)
    aggr_method = trial.suggest_categorical("aggr_method", ["mean", "max", "lstm"])
    epochs = trial.suggest_int("epochs", 10, 100)

    # Definir el modelo GraphSAGE
    
    model = SAGELinkPredictor(
        in_channels=12,  # Ajusta según tu modelo
        hidden_channels=hidden_channels,
        #num_layers=num_layers,
        #aggr=aggr_method,
        dropout=dropout_rate
    )

    # Optimización
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

      # Llama a la función con todos los argumentos
    criterion = torch.nn.BCEWithLogitsLoss()
    #val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion=criterion)
    val_loss = train_and_evaluate_model(model, train_data, val_data, epochs, criterion, learning_rate, weight_decay)
    return val_loss
           

# Optuna para GCN
#study_gcn = optuna.create_study(direction="minimize")
study_gcn = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gcn.optimize(objective_gcn, n_trials=50)

print("Best GCN hyperparameters:", study_gcn.best_params)
print("Best GCN validation loss:", study_gcn.best_value)

# Optuna para GAT
#study_gat = optuna.create_study(direction="minimize
study_gat = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gat.optimize(objective_gat, n_trials=50)

print("Best GAT hyperparameters:", study_gat.best_params)
print("Best GAT validation loss:", study_gat.best_value)

# Estudio y optimización para GIN
#study_gin = optuna.create_study(direction="minimize")
study_gin = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_gin.optimize(objective_gin, n_trials=10)

# Estudio y optimización para SAGE
#study_sage = optuna.create_study(direction="minimize")
study_sage = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_sage.optimize(objective_sage, n_trials=10)

# Imprimir los mejores parámetros para cada modelo
print("Mejores parámetros para GCN:", study_gcn.best_params)
print("Mejores parámetros para GIN:", study_gin.best_params)
print("Mejores parámetros para SAGE:", study_sage.best_params)
print("Mejores parámetros para GAT:", study_gat.best_params)


# Entrenamiento y guardado de los modelos

# Entrenar y guardar el modelo GCN
best_params = study_gcn.best_params
gcn_model = GCNLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gcn = torch.optim.Adam(gcn_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

criterion = torch.nn.BCEWithLogitsLoss()

gcn_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gcn.zero_grad()
    x = gcn_model(train_data.x, train_data.edge_index)
    out = gcn_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gcn.step()

torch.save(gcn_model.state_dict(), "P26-mejor_modelo_GCN.pth")
print("P26-Modelo GCN guardado.")

# Entrenamiento y guardado del modelo GAT
best_params = study_gat.best_params
if 'heads' in best_params:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=best_params["heads"], dropout=best_params["dropout_rate"])
else:
    gat_model = GATLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], out_channels=1, heads=1, dropout=best_params["dropout_rate"])

optimizer_gat = torch.optim.Adam(gat_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gat_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gat.zero_grad()
    x = gat_model(train_data.x, train_data.edge_index)
    out = gat_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gat.step()

torch.save(gat_model.state_dict(), "P26-mejor_modelo_GAT.pth")
print("P26-Modelo GAT guardado.")

# Entrenamiento y guardado del modelo GIN
best_params = study_gin.best_params
gin_model = GINLinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout_rate"])
optimizer_gin = torch.optim.Adam(gin_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])

gin_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_gin.zero_grad()
    x = gin_model(train_data.x, train_data.edge_index)
    out = gin_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_gin.step()

torch.save(gin_model.state_dict(), "P26-mejor_modelo_GIN.pth")
print("P26-Modelo GIN guardado.")

# Entrenamiento y guardado del modelo SAGE
#sage_model = SAGELinkPredictor(train_data.x.shape[1], best_params["hidden_channels"], dropout=best_params["dropout"])
best_params = study_sage.best_params
sage_model = SAGELinkPredictor(
    in_channels=train_data.x.shape[1],  
    hidden_channels=best_params["hidden_channels"],  
    #num_layers=best_params["num_layers"],  
    #aggr=best_params["aggr_method"],  
    dropout=best_params["dropout_rate"]  
)

optimizer_sage = torch.optim.Adam(sage_model.parameters(), lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"])
criterion = torch.nn.BCEWithLogitsLoss()
sage_model.train()
for epoch in range(best_params["epochs"]):
    optimizer_sage.zero_grad()
    x = sage_model(train_data.x, train_data.edge_index)
    out = sage_model.decode(x, train_data.edge_label_index).squeeze()
    loss = criterion(out, train_data.edge_label.float())
    loss.backward()
    optimizer_sage.step()

torch.save(sage_model.state_dict(), "P26-mejor_modelo_SAGE.pth")
print("P26-Modelo SAGE guardado.")


# Cargar todos los modelos para evaluación
gcn_model.load_state_dict(torch.load("P26-mejor_modelo_GCN.pth"))
gcn_model.eval()

gat_model.load_state_dict(torch.load("P26-mejor_modelo_GAT.pth"))
gat_model.eval()

gin_model.load_state_dict(torch.load("P26-mejor_modelo_GIN.pth"))
gin_model.eval()

sage_model.load_state_dict(torch.load("P26-mejor_modelo_SAGE.pth"))
sage_model.eval()

# Evaluar todos los modelos
metrics_gcn = evaluate_model(gcn_model, test_data)
metrics_gat = evaluate_model(gat_model, test_data)
metrics_gin = evaluate_model(gin_model, test_data)
metrics_sage = evaluate_model(sage_model, test_data)

# Crear DataFrame para comparar las métricas
df_metrics = pd.DataFrame([metrics_gcn, metrics_gat, metrics_gin, metrics_sage], index=["GCN", "GAT", "GIN", "SAGE"])

# Mostrar las métricas de los cuatro modelos
print("\n📊 P26-Métricas comparativas de GCN, GAT, GIN y SAGE:")
print(df_metrics)
df_metrics.to_csv("P26-metricas_comparativas.csv", index=True)

#predicciones con test_data
#reverse_mapping = {v: k for k, v in test_data.node_mapping.items()}
reverse_mapping = get_reverse_mapping(test_data)
test_pairs = list(zip(test_data.edge_label_index[0].cpu().numpy(), test_data.edge_label_index[1].cpu().numpy()))

# Mostrar las predicciones de los cuatro modelos
if gcn_model is not None:
    with torch.no_grad():
        x_gcn = gcn_model(test_data.x, test_data.edge_index)
        gcn_predictions = gcn_model.decode(x_gcn, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gcn_predictions = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gcn_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gcn_predictions)]
        gcn_edges = sorted(gcn_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GCN:")
        for author1, author2, score in gcn_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gcn_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P26-predicciones_gcn.csv", index=False)

if gat_model is not None:
    with torch.no_grad():
        x_gat = gat_model(test_data.x, test_data.edge_index)
        gat_predictions = gat_model.decode(x_gat, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gat_predictions = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gat_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gat_predictions)]
        gat_edges = sorted(gat_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GAT:")
        for author1, author2, score in gat_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gat_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P26-predicciones_gat.csv", index=False)


if gin_model is not None:
    with torch.no_grad():
        x_gin = gin_model(test_data.x, test_data.edge_index)
        gin_predictions = gin_model.decode(x_gin, test_data.edge_label_index).sigmoid().cpu().numpy()
        #gin_predictions = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        gin_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, gin_predictions)]
        gin_edges = sorted(gin_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones GIN:")
        for author1, author2, score in gin_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(gin_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P26-predicciones_gin.csv", index=False)

#print("Primeras características de nodos:", test_data.x[:5])
#print("Primeras conexiones (edge_index):", test_data.edge_index[:, :5])

if sage_model is not None:
    with torch.no_grad():
        x_sage = sage_model(test_data.x, test_data.edge_index)
        sage_predictions = sage_model.decode(x_sage, test_data.edge_label_index).sigmoid().cpu().numpy()
        #sage_predictions = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
        sage_edges = [(reverse_mapping[u], reverse_mapping[v], score) for (u, v), score in zip(test_pairs, sage_predictions)]
        sage_edges = sorted(sage_edges, key=lambda x: x[2], reverse=True)[:20]
        print("\nPrimeras 20 predicciones SAGE:")
        for author1, author2, score in sage_edges:
            print(f"{author1} - {author2}: {score:.4f}")
        pd.DataFrame(sage_edges, columns=["Autor1", "Autor2", "Score"]).to_csv("P26-predicciones_sage.csv", index=False)

#print("¿GCN y GAT producen las mismas predicciones?", np.allclose(gcn_predictions, gat_predictions))
#print("¿GCN y SAGE producen las mismas predicciones?", np.allclose(gin_predictions, sage_predictions))


print("Configuración GCN:", gcn_model)
print("Configuración GAT:", gat_model)
print("Configuración GIN:", gin_model)
print("Configuración SAGE:", sage_model)


## ensamble de aqui hacia arriba ya no cambiar #
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix

#import xgboost as xgb
# Obtener las predicciones de los modelos

true_labels = test_data.edge_label.cpu().numpy()
print("Valores únicos en true_labels:", np.unique(true_labels))
# Verifica la longitud de las etiquetas y las predicciones
#print("Número de etiquetas", len(true_labels))  # Debe ser el mismo tamaño que las predicciones
# Debe ser el mismo tamaño que las etiquetas

predicciones_gcn = gcn_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
print(true_labels.shape, predicciones_gcn.shape)
#print("Número de predicciones de gcn",len(predicciones_gcn))  

predicciones_gat = gat_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
print(true_labels.shape, predicciones_gat.shape)
#print("Número de predicciones de gat",len(predicciones_gat))  

predicciones_gin = gin_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
print(true_labels.shape, predicciones_gin.shape)
#print("Número de predicciones de gin",len(predicciones_gin))  

predicciones_sage = sage_model.decode(test_data.x, test_data.edge_label_index).sigmoid().cpu().numpy()
print(true_labels.shape, predicciones_sage.shape)
#print("Número de predicciones de sage",len(predicciones_sage))  

# Ya entrenado y obtenido las predicciones de los modelos base (GCN, GAT, GIN, SAGE)
X_stack = np.column_stack((predicciones_gcn, predicciones_gat, predicciones_gin, predicciones_sage))

####  optimizar con OPtuna el modelo ensamble meta_model para regresion logistica


# ----- Función genérica de entrenamiento y evaluación -----
def entrenar_y_evaluar(modelo, nombre_modelo):
    modelo.fit(X_stack, true_labels)
    predicciones = modelo.predict(X_stack)
    
    accuracy = accuracy_score(true_labels, predicciones)
    precision = precision_score(true_labels, predicciones)
    recall = recall_score(true_labels, predicciones)
    f1 = f1_score(true_labels, predicciones)
    roc_auc = roc_auc_score(true_labels, predicciones)
    conf_matrix = confusion_matrix(true_labels, predicciones)
    
    print(f"Métricas del modelo meta ({nombre_modelo}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Predicciones con nombres
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = predicciones[i]
        predicciones_autores.append((author1_name, author2_name, score))
    
    return modelo, predicciones_autores
    
# ----- Modelos Meta -----
# 1. Logistic Regression optimizado con Optuna
def objective_logistic(trial):
    meta_model = LogisticRegression(
        #C=trial.suggest_loguniform('C', 1e-5, 1e5),
        C = trial.suggest_float('C', 1e-5, 1e5, log=True),
        max_iter=trial.suggest_int('max_iter', 100, 1000),
        random_state=SEED
    )
    meta_model.fit(X_stack, true_labels)
    predicciones = meta_model.predict(X_stack)
    return roc_auc_score(true_labels, predicciones)

def objective_random_forest(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 20)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_gb(trial):
    model = GradientBoostingClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        max_depth=trial.suggest_int('max_depth', 3, 10)
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_svc(trial):
    model = SVC(
        C=trial.suggest_loguniform('C', 1e-5, 1e5),
        probability=True
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

def objective_xgb(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 300),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        use_label_encoder=False,
        eval_metric='logloss'
    )
    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)
    return roc_auc_score(true_labels, preds)

# Diccionario de objetivos
objectives = {
    "LogisticRegression": objective_logistic,
    "RandomForest": objective_random_forest,
    "GradientBoosting": objective_gb,
    "SVC": objective_svc,
    "XGBoost": objective_xgb
}

# Resultados
results = {}

for name, obj in objectives.items():
    print(f"Optimizando {name}...")
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(obj, n_trials=50)
    best_params = study.best_params

    if name == "LogisticRegression":
        model = LogisticRegression(**best_params)
    elif name == "RandomForest":
        model = RandomForestClassifier(**best_params)
    elif name == "GradientBoosting":
        model = GradientBoostingClassifier(**best_params)
    elif name == "SVC":
        model = SVC(**best_params, probability=True)
    elif name == "XGBoost":
        model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss')

    model.fit(X_stack, true_labels)
    preds = model.predict(X_stack)

    # Evaluación
    accuracy = accuracy_score(true_labels, preds)
    precision = precision_score(true_labels, preds)
    recall = recall_score(true_labels, preds)
    f1 = f1_score(true_labels, preds)
    roc_auc = roc_auc_score(true_labels, preds)
    conf_matrix = confusion_matrix(true_labels, preds)

    print(f"\nMétricas del modelo meta ({name}):")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{conf_matrix}\n")

    # Guardar predicciones con nombres de autores en diccionario y en archivo
    predicciones_autores = []
    for i, (author1_idx, author2_idx) in enumerate(test_pairs):
        author1_name = reverse_mapping[author1_idx]
        author2_name = reverse_mapping[author2_idx]
        score = float(preds[i])
        predicciones_autores.append((author1_name, author2_name, score))
    results[name] = predicciones_autores
    df = pd.DataFrame(predicciones_autores, columns=["Autor1", "Autor2", "Score"])
        # Formatear la columna Score a 4 decimales
    df["Score"] = df["Score"].map(lambda x: f"{x:.4f}")
    file_name = f"P26-predicciones_meta-{name.replace(' ', '_').lower()}.csv"
    df.to_csv(file_name, index=False)


###  FIN Optimizar con OPtuna modelo ensamble


df_train connected counts:
 connected
0.0    13691
1.0     1806
Name: count, dtype: int64
df_val connected counts:
 connected
0.0    6731
1.0    3386
Name: count, dtype: int64
df_test connected counts:
 connected
0.0    4006
1.0    1878
Name: count, dtype: int64
true_labels unique/counts: (array([0., 1.], dtype=float32), array([4006, 1878]))
Densidad Train: 0.1027
Densidad Validation: 0.0158
Densidad Test: 0.0234
Best GCN hyperparameters: {'hidden_channels': 53, 'dropout_rate': 0.14151681758586332, 'learning_rate': 0.0019441944472424217, 'weight_decay': 0.0031779336001614405, 'epochs': 53}
Best GCN validation loss: 0.6930052638053894
Best GAT hyperparameters: {'hidden_channels': 37, 'heads': 3, 'dropout_rate': 0.24058433440788762, 'learning_rate': 0.0007882526313348943, 'weight_decay': 0.007052775890067773, 'epochs': 38}
Best GAT validation loss: 0.6930972933769226
Mejores parámetros para GCN: {'hidden_channels': 53, 'dropout_rate': 0.14151681758586332, 'learning_rate': 0.0019441944472

C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\3864228828.py:806: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\3864228828.py:806: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppData\Local\Temp\ipykernel_20704\3864228828.py:806: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C=trial.suggest_loguniform('C', 1e-5, 1e5),
C:\Users\Dra. Pilar\AppDa


Métricas del modelo meta (SVC):
Accuracy: 0.7039
Precision: 0.5428
Recall: 0.4590
F1 Score: 0.4974
AUC-ROC: 0.6389
Confusion Matrix:
[[3280  726]
 [1016  862]]

Optimizando XGBoost...

Métricas del modelo meta (XGBoost):
Accuracy: 0.7736
Precision: 0.6641
Recall: 0.5884
F1 Score: 0.6239
AUC-ROC: 0.7244
Confusion Matrix:
[[3447  559]
 [ 773 1105]]



CONCLUSION EN GRAFICO SOBRE FP,LSP Y GSP checar ahora con nodos por fechas
caso es predicción de enlaces con grafos dinámicos (porque los conjuntos de train, valid, test se arman por fechas y nodos, no por un muestreo aleatorio de aristas). Eso implica que la evaluación debe medir qué tan bien generaliza el modelo para predecir enlaces futuros entre nodos ya existentes o nuevos.

De las métricas que aparecen en tu programa:

✅ AUC-ROC (Area Under ROC Curve)

Muy usada en predicción de enlaces porque mide la capacidad de distinguir entre enlaces reales y no-enlaces.

Buena métrica global de discriminación, aunque puede ser optimista si la clase positiva es rara.

✅ AUC-PR (Average Precision, área bajo la curva Precision-Recall)

Es crítica en predicción de enlaces, porque normalmente tienes un grafo muy disperso (muchos más no-enlaces que enlaces).

Captura mejor el desempeño en escenarios con clases muy desbalanceadas (lo típico en grafos grandes).

✅ F1 Score

Buen balance entre precisión y exhaustividad.

Representa cuántos de los enlaces predichos son correctos y cuántos de los reales fueron recuperados.

➖ Accuracy

Poco informativa en grafos, porque hay muchísimos más no-enlaces que enlaces: un modelo trivial que siempre diga “no enlace” tendría accuracy muy alto.

➖ Precision y Recall individuales

Útiles para análisis detallado, pero aisladas no cuentan toda la historia. Por eso suele usarse F1 (que las combina).

⚠️ MSE (Mean Squared Error)

No es típica en predicción de enlaces, porque aquí no es un problema de regresión sino de clasificación binaria.

Puede usarse como proxy de qué tan cercanas están las probabilidades predichas a 0 o 1, pero no es la métrica más representativa.

📌 Recall (exhaustividad): mide la proporción de enlaces verdaderos que tu modelo sí logra detectar.

En tu escenario (grafos por fechas), un recall alto significa que el modelo es capaz de descubrir la mayoría de los enlaces que realmente se formaron en el futuro.

Problema: si se busca recall muy alto, el modelo puede tender a predecir demasiados enlaces (incluso falsos), bajando la precisión.

🔹 Por eso, recall aislado no basta: se necesita combinarlo con precisión → ahí entra el F1 Score.

📌 En resumen, para tu caso de predicción temporal de enlaces:

Recall → relevante porque indica qué tanto del grafo futuro (enlaces reales) logra recuperar el modelo.

Precision → indica qué tan confiables son las predicciones positivas que hace.

F1 Score → balance entre ambas, más interpretativo en escenarios desbalanceados.

👉 Yo te lo pondría así:

AUC-PR → principal (escenario desbalanceado).

AUC-ROC → visión global de discriminación.

F1 Score → balance práctico entre precisión y recall.

Recall (apoyo) → útil para evaluar qué tan bien descubre el modelo los enlaces verdaderos, especialmente si te interesa minimizar falsos negativos (no perder conexiones reales).

Precision (apoyo) → útil para evaluar qué tan confiables son los enlaces predichos.

Accuracy y MSE → secundarias.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Modelos y programas
models = ['GCN', 'GAT', 'GIN', 'SAGE']
programs = ['P20 FP+LSP+GSP','P21 GSP+FP','P22 GDP+LSP','P23 LSP','P24 GSP','P25 FP','P26 LSP+FP']

# Métricas
F1 = {
    'P20 FP+LSP+GSP': [0.483896, 0.483896, 0.483378, 0.409354],
    'P21 GSP+FP':      [0.483249, 0.483896, 0.540119, 0.391831],
    'P22 GDP+LSP':     [0.483896, 0.484817, 0.484817, 0.551922],
    'P23 LSP':         [0.497924, 0.484317, 0.483896, 0.418188],
    'P24 GSP':         [0.408176, 0.0, 0.520670, 0.0],
    'P25 FP':          [0.483079, 0.483896, 0.483896, 0.485869],
    'P26 LSP+FP':      [0.483896, 0.483675, 0.484317, 0.491705]
}

AUC = {
    'P20 FP+LSP+GSP': [0.753557, 0.458864, 0.690488, 0.495287],
    'P21 GSP+FP':      [0.594888, 0.502470, 0.765187, 0.479038],
    'P22 GDP+LSP':     [0.710354, 0.502195, 0.734417, 0.736630],
    'P23 LSP':         [0.746819, 0.711345, 0.627724, 0.443824],
    'P24 GSP':         [0.593493, 0.500000, 0.743572, 0.501123],
    'P25 FP':          [0.562626, 0.448870, 0.632316, 0.580204],
    'P26 LSP+FP':      [0.580300, 0.660910, 0.745537, 0.598031]
}

Recall = {
    'P20 FP+LSP+GSP': [1.000000, 1.000000, 0.960064, 0.563898],
    'P21 GSP+FP':      [0.994675, 1.000000, 0.847710, 0.551651],
    'P22 GDP+LSP':     [1.000000, 0.998935, 0.998935, 0.817891],
    'P23 LSP':         [0.957934, 0.998935, 1.000000, 0.646432],
    'P24 GSP':         [0.340256, 0.0, 0.902023, 0.0],
    'P25 FP':          [0.995740, 1.000000, 1.000000, 0.883387],
    'P26 LSP+FP':      [1.000000, 0.997870, 0.998935, 0.741747]
}

x = np.arange(len(models))
width = 0.1

fig, axes = plt.subplots(1, 3, figsize=(18,6))
colors = ['skyblue','dodgerblue','lightgreen','green','lightcoral','red','orange']
patterns = ['','//','..','xx','\\\\','--','++']

# F1 Score
for i, prog in enumerate(programs):
    axes[0].bar(x + (i-3)*width, F1[prog], width, label=prog, color=colors[i], hatch=patterns[i])
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].set_ylim(0,1)
axes[0].set_title('F1 Score por Modelo y Programa')
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# AUC-ROC
for i, prog in enumerate(programs):
    axes[1].bar(x + (i-3)*width, AUC[prog], width, label=prog, color=colors[i], hatch=patterns[i])
axes[1].set_xticks(x)
axes[1].set_xticklabels(models)
axes[1].set_ylim(0,1)
axes[1].set_title('AUC-ROC por Modelo y Programa')
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

# Recall
for i, prog in enumerate(programs):
    axes[2].bar(x + (i-3)*width, Recall[prog], width, label=prog, color=colors[i], hatch=patterns[i])
axes[2].set_xticks(x)
axes[2].set_xticklabels(models)
axes[2].set_ylim(0,1)
axes[2].set_title('Recall por Modelo y Programa')
axes[2].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Programas')

# Guardar la figura
output_base = r"C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\Comparativa_Programas_Features"
plt.savefig(output_base + ".pdf", format="pdf")
plt.savefig(output_base + ".eps", format="eps")

plt.show()


English (Academic Interpretation)

The figure illustrates the comparative performance of four graph neural network models (GCN, GAT, GIN, SAGE) across seven feature configurations (Programs 20–26) for link prediction tasks on temporally split node-based graphs. Performance metrics considered are F1 Score, AUC-ROC, and Recall.

Observations and Analysis:

Feature Effectiveness:
Programs incorporating local and global structural features combined with node proximity features (e.g., Program 20: FP+LSP+GSP) consistently achieve higher F1 and Recall scores across most models. Conversely, programs with a limited number of features, such as solely GSP (Program 24), show poor predictive performance, likely due to insufficient structural information for sparse graphs (densities: Train 0.1027, Validation 0.0158, Test 0.0234).

Model Performance:
Among the GNNs, GCN and GIN exhibit more stable and relatively superior performance across all feature sets, particularly in F1 Score and AUC-ROC. SAGE shows greater variability and lower performance in sparse feature configurations, indicating sensitivity to feature type and graph density.

Trade-offs and Recommendations:
High recall is observed when structural features (LSP and GSP) are combined with proximity features, suggesting that dense local neighborhoods enhance link prediction. Models that rely solely on global features (GSP) or limited proximity features (FP) underperform, emphasizing the importance of feature richness for temporal node-based graphs.

Conclusion:
Integrating multiple feature types (local, global, and proximity) is recommended for robust link prediction on sparse, temporally split node-based graphs. GCN and GIN models offer consistent and reliable performance, while SAGE may require feature augmentation or hyperparameter tuning to reach comparable levels.

Español (Traducción)

La figura ilustra el desempeño comparativo de cuatro modelos de redes neuronales gráficas (GCN, GAT, GIN, SAGE) en siete configuraciones de características (Programas 20–26) para tareas de predicción de enlaces en grafos basados en nodos y separados temporalmente. Las métricas de desempeño consideradas son F1 Score, AUC-ROC y Recall.

Observaciones y Análisis:

Efectividad de las características:
Los programas que incorporan características estructurales locales y globales combinadas con características de proximidad de nodos (por ejemplo, Programa 20: FP+LSP+GSP) alcanzan de manera consistente mayores puntuaciones en F1 y Recall en la mayoría de los modelos. Por el contrario, los programas con un número limitado de características, como únicamente GSP (Programa 24), muestran un bajo desempeño predictivo, probablemente debido a información estructural insuficiente para grafos dispersos (densidades: Train 0.1027, Validation 0.0158, Test 0.0234).

Desempeño de los modelos:
Entre las GNN, GCN y GIN presentan un desempeño más estable y relativamente superior en todos los conjuntos de características, particularmente en F1 Score y AUC-ROC. SAGE muestra mayor variabilidad y desempeño inferior en configuraciones de características escasas, indicando sensibilidad al tipo de característica y densidad del grafo.

Compromisos y recomendaciones:
Se observa un Recall elevado cuando las características estructurales (LSP y GSP) se combinan con características de proximidad, lo que sugiere que vecindarios locales densos mejoran la predicción de enlaces. Los modelos que dependen únicamente de características globales (GSP) o de características de proximidad limitadas (FP) tienen bajo desempeño, enfatizando la importancia de la riqueza de características para grafos basados en nodos separados temporalmente.

Conclusión:
Se recomienda integrar múltiples tipos de características (locales, globales y de proximidad) para lograr una predicción de enlaces robusta en grafos dispersos y separados por tiempo. Los modelos GCN y GIN ofrecen un desempeño consistente y confiable, mientras que SAGE podría requerir aumento de características o ajuste de hiperparámetros para alcanzar niveles comparables.

\begin{table}[h!]
\centering
\caption{Características utilizadas por cada programa en términos de features}
\label{tab:features_programas}
\begin{tabular}{|c|l|}
\hline
\textbf{Programa} & \textbf{Features utilizadas} \\ \hline
20 & FP + LSP + GSP: sum\_of\_papers, sum\_of\_neighbors, log\_secundary\_neighbors, sum\_of\_keywords, keywords\_match, sum\_of\_areas, lenght\_short\_path, clustering\_index\_sum, jaccard\_coefficient, resource\_allocation, adamic\_adar\_index, preferential\_attachment, community\_cn, community\_ra \\ \hline
21 & GSP + FP: sum\_of\_areas, sum\_of\_papers, sum\_of\_keywords, keywords\_match, community\_cn, community\_ra \\ \hline
22 & GSP + LSP: lenght\_short\_path, jaccard\_coefficient, adamic\_adar\_index, resource\_allocation, preferential\_attachment, sum\_of\_neighbors, log\_secundary\_neighbors, clustering\_index\_sum \\ \hline
23 & Solo LSP + GSP: lenght\_short\_path, jaccard\_coefficient, adamic\_adar\_index, resource\_allocation, preferential\_attachment, sum\_of\_neighbors, log\_secundary\_neighbors, clustering\_index\_sum, community\_cn, community\_ra \\ \hline
24 & Solo GSP: community\_cn, community\_ra \\ \hline
25 & Solo FP: sum\_of\_areas, sum\_of\_papers, sum\_of\_keywords, keywords\_match \\ \hline
26 & LSP + FP: lenght\_short\_path, jaccard\_coefficient, adamic\_adar\_index, resource\_allocation, preferential\_attachment, sum\_of\_neighbors, log\_secundary\_neighbors, clustering\_index\_sum, sum\_of\_areas, sum\_of\_papers, sum\_of\_keywords, keywords\_match \\ \hline
\end{tabular}
\end{table}
